# Task 23 — Training & Deploying TensorFlow Models at Scale

**Track:** ML (Tasks 16–23) · **Notebook type:** end-to-end, phase-by-phase · **Framework:** TensorFlow 2.x / Keras 3

This notebook takes a trained model out of the notebook and into production, and then makes training itself faster.
It is organised into **8 phases + exercises + a capstone**, and every phase follows the same pattern:

1. **Concept** — what the technique is and why it exists (written in plain language)
2. **Code** — a runnable implementation
3. **Result / interpretation** — what the output means
4. **Phase summary** — the deliverable for that phase

## 0. Overview

### The story in one paragraph
A model that only lives in a notebook is useless to the rest of a company. The first half of this notebook
turns a trained classifier into a **service**: export it in a portable format (SavedModel), serve it with
**TF Serving**, call it over **REST and gRPC**, roll versions forward and back without downtime, batch requests,
plan for scale and the cloud, and shrink it for **phones and browsers** (TFLite, quantization, TensorFlow.js).
The second half is about **training speed**: using GPUs properly (memory, placement, parallel execution),
then spreading one training job across several devices and machines with the **Distribution Strategies API**,
cluster configuration, cloud jobs and hyperparameter tuning. A capstone ties everything together.

### Roadmap

| Phase | Topic | Main deliverable |
|---|---|---|
| 1 | Exporting & inspecting SavedModels | Versioned SavedModel, CLI inspection, preprocessing inside the model, `export_model()` with self-test |
| 2 | TF Serving with Docker: REST vs gRPC | Working REST + gRPC clients, payload & latency benchmarks, failure drills |
| 3 | Versioning, rollback, batching, scale-out | v2 rollout + rollback evidence, A/B (canary) test, batching charts, runbook |
| 4 | Managed prediction service in the cloud | Design document (Vertex AI), cost model, security checklist, client code |
| 5 | TFLite, quantization, TensorFlow.js | Size/accuracy/latency table, NumPy quantizer, QAT, deployment-target table |
| 6 | GPUs: memory, placement, parallel execution | Device listings, memory options, placement tests, scheduler walkthrough, thread pools |
| 7 | Model vs data parallelism, Distribution Strategies | Strategy timing table, custom distributed loop, saturation model, LR/batch experiment |
| 8 | Clusters, cloud jobs, tuning, capstone | Two-worker run, tuning analog, Flowers capstone, plans & checklists |
| — | Exercises 1–15 | Written answers + worked calculations |

### How to run this notebook
* **Recommended:** a machine with Docker (for the real TF Serving server) **or** Google Colab with a GPU runtime.
* Every part that needs something optional (Docker, a GPU, a cloud account, `tensorflowjs`) **detects it automatically**.
  When it is missing, the notebook uses a clearly-labelled **local fallback** (for example an in-process
  version-watching server and a Python micro-batcher) so the notebook still runs **top to bottom without errors**.
* `QUICK = True` (next section) keeps every training run short. Set it to `False` for full-quality numbers.
* No credentials are stored anywhere in this notebook. Cloud cells are disabled by default (`USE_CLOUD = False`).

### Version notes (what changed since the original 2019 material)
| Then | Now (used here) |
|---|---|
| `model.save()` → SavedModel | Keras 3: `model.save()` → `.keras` file; serving export via `model.export()` / `keras.export.ExportArchive` |
| `keras.models.load_model(savedmodel_dir)` | Keras 3: `keras.layers.TFSMLayer` (or load the `.keras` file) |
| Google Cloud AI Platform, Stackdriver | **Vertex AI**, Cloud Logging / Cloud Monitoring |
| `tf.test.is_gpu_available()` | `tf.config.list_physical_devices("GPU")` |
| `tf.config.experimental.set_virtual_device_configuration` | `tf.config.set_logical_device_configuration` |
| `tf.distribute.experimental.MultiWorkerMirroredStrategy` | `tf.distribute.MultiWorkerMirroredStrategy` |
| `tf.lite.Optimize.OPTIMIZE_FOR_SIZE` | `tf.lite.Optimize.DEFAULT` (TFLite runtime is now branded **LiteRT**) |

## 0.1 Setup

### Step A — dependencies
`tensorflow-serving-api` normally pins a specific TensorFlow version, so installing it the usual way can silently
**downgrade** TensorFlow. We install it with `--no-deps` (it only needs the gRPC stubs and protobuf, which are
already present with TensorFlow).

In [ ]:
import importlib, subprocess, sys

def ensure(pip_name, import_name=None, no_deps=False):
    """Install a package only if it cannot be imported."""
    try:
        importlib.import_module(import_name or pip_name)
        return True
    except ImportError:
        cmd = [sys.executable, "-m", "pip", "install", "-q", pip_name]
        if no_deps:
            cmd.append("--no-deps")
        print("Installing", pip_name, "...")
        r = subprocess.run(cmd, capture_output=True, text=True)
        if r.returncode != 0:
            print("  could not install", pip_name, "->", r.stderr[-300:])
            return False
        return True

for pip_name, import_name, no_deps in [
    ("requests", None, False),
    ("pandas", None, False),
    ("matplotlib", None, False),
    ("grpcio", "grpc", False),
    ("tensorboard", None, False),          # recent TF wheels no longer pull it in; needed for tf.summary (Phase 8)
    ("tensorflow-serving-api", "tensorflow_serving", True),
]:
    ensure(pip_name, import_name, no_deps)

### Step B — device configuration (must run before any TensorFlow computation)

Logical-device settings can only be changed **before** TensorFlow initialises its runtime. Phases 6–8 need
at least two devices to demonstrate data parallelism, so we decide the layout once, here:

* **1 physical GPU** → split it into `N_VIRTUAL` *virtual GPUs*, each with a fixed memory limit
* **≥ 2 physical GPUs** → use them as they are
* **no GPU** → split the CPU into `N_VIRTUAL` *logical CPUs* (lets `MirroredStrategy` code run anywhere)

`DEVICES` is the list every later phase uses for distribution experiments.

In [ ]:
import os
os.environ.setdefault("TF_CPP_MIN_LOG_LEVEL", "2")

import tensorflow as tf
from tensorflow import keras

N_VIRTUAL   = 2       # virtual devices to create when only one physical device exists
VGPU_MEM_MB = 3072    # memory limit per virtual GPU (fits easily on a 15 GB Colab T4)

physical_gpus = tf.config.list_physical_devices("GPU")
try:
    if len(physical_gpus) == 1:
        tf.config.set_logical_device_configuration(
            physical_gpus[0],
            [tf.config.LogicalDeviceConfiguration(memory_limit=VGPU_MEM_MB) for _ in range(N_VIRTUAL)])
    elif len(physical_gpus) == 0:
        cpu = tf.config.list_physical_devices("CPU")[0]
        tf.config.set_logical_device_configuration(
            cpu, [tf.config.LogicalDeviceConfiguration() for _ in range(N_VIRTUAL)])
except RuntimeError as e:
    print("Runtime already initialised (restart the kernel to change the device layout):", e)

LOGICAL_GPUS = tf.config.list_logical_devices("GPU")
HAS_GPU = len(physical_gpus) > 0
DEVICES = [d.name for d in LOGICAL_GPUS] if LOGICAL_GPUS else [d.name for d in tf.config.list_logical_devices("CPU")]

def make_mirrored(devices=None, **kwargs):
    """MirroredStrategy over our devices. On CPU-only machines use ReductionToOneDevice: the default
    collective (RING) implementation can mix up collective keys between several strategy objects
    in one process, which crashes the second model with a 'shape mismatch in the collective' error."""
    if not HAS_GPU and "cross_device_ops" not in kwargs:
        kwargs["cross_device_ops"] = tf.distribute.ReductionToOneDevice()
    return tf.distribute.MirroredStrategy(devices or DEVICES, **kwargs)

print("TensorFlow :", tf.__version__)
print("Keras      :", keras.__version__)
print("Physical GPUs:", physical_gpus)
print("Devices used for distribution experiments:", DEVICES)

### Step C — global configuration, paths and shared helpers

In [ ]:
import json, time, shutil, math, random, pathlib, socket, threading, queue, base64, gzip, io, tarfile
import concurrent.futures as cf
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

try:
    import requests
    REQUESTS_OK = True
except ImportError:
    REQUESTS_OK = False

# ---------------- configuration ----------------
SEED       = 42
QUICK      = True          # short runs; set False for full-quality numbers
USE_CLOUD  = False         # Phase 4 / Phase 8 cloud cells stay off unless you set this (and your project) yourself
EPOCHS     = 2 if QUICK else 10

WORK         = pathlib.Path("task23_work").resolve()
SERVING_ROOT = WORK / "serving"            # everything TF Serving will see is under here
MODEL_NAME   = "my_mnist_model"
MODEL_ROOT   = SERVING_ROOT / MODEL_NAME   # versions live in MODEL_ROOT/0001, MODEL_ROOT/0002, ...
for p in (WORK, SERVING_ROOT):
    p.mkdir(parents=True, exist_ok=True)

random.seed(SEED); np.random.seed(SEED); tf.random.set_seed(SEED)
pd.set_option("display.width", 140)
RESULTS = {}               # every phase stores its key tables here (used by the capstone summary)

# ---------------- shared helpers ----------------
def human(nbytes):
    for unit in ["B", "KB", "MB", "GB"]:
        if nbytes < 1024:
            return f"{nbytes:.1f} {unit}"
        nbytes /= 1024
    return f"{nbytes:.1f} TB"

def sh(cmd, timeout=900, quiet=False):
    """Run a shell command, print (the tail of) its output, return the CompletedProcess."""
    if not quiet:
        print("$", cmd)
    r = subprocess.run(cmd, shell=True, capture_output=True, text=True, timeout=timeout)
    if not quiet:
        out = (r.stdout or "") + (r.stderr or "")
        print(out[-4000:].rstrip())
    return r

def bench(fn, warmup=2, repeats=5, inner=1):
    """Task 18 rules: discard warm-up runs, repeat, report mean ± std in milliseconds."""
    for _ in range(warmup):
        fn()
    times = []
    for _ in range(repeats):
        t0 = time.perf_counter()
        for _ in range(inner):
            fn()
        times.append((time.perf_counter() - t0) / inner)
    a = np.array(times) * 1000
    return float(a.mean()), float(a.std())

def dir_tree(path, prefix=""):
    path = pathlib.Path(path)
    lines = []
    entries = sorted(path.iterdir(), key=lambda p: (p.is_file(), p.name))
    for i, p in enumerate(entries):
        last = i == len(entries) - 1
        conn = "└── " if last else "├── "
        lines.append(prefix + conn + p.name + ("/" if p.is_dir() else f"  ({human(p.stat().st_size)})"))
        if p.is_dir():
            lines += dir_tree(p, prefix + ("    " if last else "│   "))
    return lines

def free_port():
    with socket.socket() as s:
        s.bind(("localhost", 0))
        return s.getsockname()[1]

print("Working directory:", WORK)

### Step D — data (MNIST)
MNIST is the main serving example (small, quick, easy to check by eye). We keep two copies:
`X_*_raw` (pixel values 0–255, what a real client would send) and `X_*` (scaled to 0–1, what the first model expects).

In [ ]:
(X_train_full_raw, y_train_full), (X_test_raw, y_test) = keras.datasets.mnist.load_data()
X_train_raw, X_valid_raw = X_train_full_raw[:-5000], X_train_full_raw[-5000:]
y_train,     y_valid     = y_train_full[:-5000],     y_train_full[-5000:]

X_train = (X_train_raw / 255.0).astype("float32")
X_valid = (X_valid_raw / 255.0).astype("float32")
X_test  = (X_test_raw  / 255.0).astype("float32")

X_new = X_test[:5]          # the 5 reference images used for every "same predictions?" check
print("train", X_train.shape, "valid", X_valid.shape, "test", X_test.shape, "| X_new", X_new.shape)

---
# Phase 1 — Exporting & Inspecting SavedModels

**Goal of this phase:** produce a portable, versioned model package that any server (TF Serving, a cloud service,
the TFLite/TF.js converters) can load without our Python code, look inside it, and wrap the whole export in a
reusable helper that checks its own output.

## 1.1 Why put a model behind a dedicated prediction service?

Imagine three teams — web, mobile and analytics — all needing the digit classifier. If each team copies the model
file into its own codebase, every retrain becomes three deployments, and sooner or later one team runs an older
model than the others. A **prediction service** fixes this by making the model a separate component with a
small, stable interface ("send pixels, receive probabilities"):

| Benefit | What it means in practice |
|---|---|
| **Decoupling** | The rest of the system only knows an HTTP/gRPC endpoint; the model can change underneath it |
| **Independent scaling** | Add serving replicas when traffic grows without scaling the web app |
| **Version switching** | Deploy v2, roll back to v1, in one place |
| **Experiments** | Route a slice of traffic to a new version (A/B test, canary) |
| **Consistency** | Every consumer gets predictions from the *same* model version |
| **Testability** | The service can be load-tested and contract-tested on its own |

**Why not just write a Flask app?** A 20-line Flask wrapper works for a demo, but production needs things that are
hard to get right: loading new versions with zero downtime, unloading old ones safely, batching concurrent
requests for the GPU, efficient binary protocols, multi-model hosting, metrics and health endpoints, and a
multithreaded C++ runtime without Python's GIL. TF Serving already does all of this and is battle-tested at
Google scale, so writing our own would mean re-implementing (and debugging) a model server.

## 1.2 Train the version-1 classifier
A small dense network is enough here — the focus of this task is everything *around* the model.

In [ ]:
def build_mnist_model(hidden=100, extra_layer=False, lr=1e-3):
    layers = [keras.Input(shape=(28, 28), name="images"),
              keras.layers.Flatten(),
              keras.layers.Dense(hidden, activation="relu")]
    if extra_layer:
        layers.append(keras.layers.Dense(50, activation="relu"))
    layers.append(keras.layers.Dense(10, activation="softmax"))
    model = keras.Sequential(layers)
    model.compile(loss="sparse_categorical_crossentropy",
                  optimizer=keras.optimizers.Nadam(learning_rate=lr), metrics=["accuracy"])
    return model

tf.random.set_seed(SEED)
model_v1 = build_mnist_model()
hist = model_v1.fit(X_train, y_train, epochs=EPOCHS, validation_data=(X_valid, y_valid), verbose=2)
v1_test_acc = model_v1.evaluate(X_test, y_test, verbose=0)[1]
print(f"v1 test accuracy: {v1_test_acc:.4f}")

## 1.3 Export version `0001`

### Why is the version a *directory name*?
TF Serving (and Vertex AI, and our own helpers) watch a **model base directory** and treat every numeric
sub-folder as a version: `my_mnist_model/0001`, `my_mnist_model/0002`, ... This gives us:
* **deploy** = copy a new numbered folder in; **rollback** = delete it — no config change, no restart;
* old and new versions can sit side by side, so the server can switch over gracefully;
* the version number is visible to anyone listing the bucket/folder, which makes audits trivial.

### Export method
In Keras 3, `model.save()` writes a single `.keras` file, *not* a SavedModel. To create a serving SavedModel we use
`keras.export.ExportArchive` (the machinery behind `model.export()`), which lets us **name the endpoint and its
input** explicitly. On older TF/Keras 2 we fall back to plain `tf.saved_model.save` with a `tf.Module`.
Both produce a standard SavedModel with a `serving_default` signature.

In [ ]:
MNIST_SPEC = tf.TensorSpec(shape=[None, 28, 28], dtype=tf.float32, name="images")

def write_savedmodel(model, path, input_spec):
    """Write `model` as a SavedModel with one `serving_default` signature returning {'probs': ...}."""
    path = pathlib.Path(path)
    def serve(images):
        return {"probs": model(images, training=False)}
    if hasattr(keras, "export") and hasattr(keras.export, "ExportArchive"):
        archive = keras.export.ExportArchive()
        archive.track(model)
        archive.add_endpoint(name="serving_default", fn=serve, input_signature=[input_spec])
        archive.write_out(str(path))
    else:  # Keras 2 fallback
        module = tf.Module()
        module.model = model
        module.serve = tf.function(serve, input_signature=[input_spec])
        tf.saved_model.save(module, str(path), signatures={"serving_default": module.serve})
    return path

v1_path = MODEL_ROOT / "0001"
if v1_path.exists():
    shutil.rmtree(v1_path)
write_savedmodel(model_v1, v1_path, MNIST_SPEC)
print("Exported to", v1_path)

## 1.4 What is inside a SavedModel? (annotated directory tree)

In [ ]:
print(MODEL_ROOT.name + "/")
print("\n".join(dir_tree(MODEL_ROOT)))

**Annotation**

| Item | What it holds |
|---|---|
| `0001/` | One **version** of the model. The server picks versions by these numeric folder names. |
| `saved_model.pb` | The **computation graph(s)** and signatures, serialized as a protocol buffer (Task 19). It contains *no* Python code — only TensorFlow ops. |
| `variables/variables.data-00000-of-00001` | The trained **weights**. Large models are split into several shards (`-00000-of-00004`, ...). |
| `variables/variables.index` | Index telling TensorFlow which weight lives in which shard and at which offset. |
| `assets/` | Extra files the graph needs at run time, e.g. a **vocabulary file** for a lookup table, class names, or example instances. Empty for this model; the text model in 1.8 uses it. |
| `fingerprint.pb` *(newer TF)* | Hashes that identify the exact model contents — useful for tracking which model is serving. |
| `assets.extra/` *(optional, Phase 3)* | Not used by the graph; TF Serving reads **warm-up requests** from here. |

## 1.5 Load it back two ways and compare

* `tf.saved_model.load()` → a **generic trackable object**, not a Keras model. You call its signature like a function,
  passing tensors of exactly the right dtype by keyword. No `.fit()`, no `.predict()`, no `.summary()`.
* Keras way → in Keras 2 `keras.models.load_model(savedmodel_dir)` returned a Keras model. **Keras 3 no longer loads
  SavedModels** with `load_model`; it offers `keras.layers.TFSMLayer` (wraps a SavedModel as an inference-only layer)
  or you save/load the native `.keras` file. We try all of these and compare predictions on the same 5 images.

In [ ]:
# Way 1: generic SavedModel object
loaded = tf.saved_model.load(str(v1_path))
serving_fn = loaded.signatures["serving_default"]
print("Type returned by tf.saved_model.load:", type(loaded).__name__)
print("Signatures:", list(loaded.signatures.keys()))
pred_tf = serving_fn(images=tf.constant(X_new, dtype=tf.float32))["probs"].numpy()

# Way 2: Keras
pred_keras = None
try:
    reloaded = keras.models.load_model(str(v1_path))          # works on Keras 2 only
    pred_keras = reloaded.predict(X_new, verbose=0)
    print("keras.models.load_model(SavedModel) -> Keras model (Keras 2 behaviour)")
except Exception as e:
    print("keras.models.load_model on a SavedModel failed (expected on Keras 3):", type(e).__name__)
    if hasattr(keras.layers, "TFSMLayer"):
        tfsm = keras.layers.TFSMLayer(str(v1_path), call_endpoint="serving_default")
        pred_keras = tfsm(X_new)["probs"]
        pred_keras = pred_keras.numpy() if hasattr(pred_keras, "numpy") else np.asarray(pred_keras)
        print("Used keras.layers.TFSMLayer instead (inference-only Keras layer around the SavedModel)")

# Native Keras file, for completeness
keras_file = WORK / "mnist_v1.keras"
model_v1.save(keras_file)
pred_native = keras.models.load_model(keras_file).predict(X_new, verbose=0)

ref = model_v1.predict(X_new, verbose=0)
print("\nmax |SavedModel(tf.load) - original| :", np.abs(pred_tf - ref).max())
print("max |SavedModel(Keras)   - original| :", np.abs(pred_keras - ref).max())
print("max |.keras file         - original| :", np.abs(pred_native - ref).max())
print("Predicted classes:", pred_tf.argmax(1), "| true:", y_test[:5])

## 1.6 Metagraphs, tags and signature definitions

* A **metagraph** = one computation graph + the information needed to run it (variables, assets, signatures).
  A SavedModel can store several, e.g. one for CPU serving and one for GPU serving.
* A **tag** is a label used to select a metagraph. The standard tag for inference is **`serve`**
  (others: `train`, `gpu`, `tpu`). Tools always ask "which tag set?" — `--tag_set serve`.
* A **signature definition** names an entry point inside a metagraph: its input tensors (name, dtype, shape) and
  output tensors. **`serving_default`** is the one servers call when you do not specify a signature.
  Exports also contain an internal **`__saved_model_init_op`** entry that initialises tables/variables when the
  model is loaded — you never call it yourself.

Below we read `saved_model.pb` directly as a protocol buffer to see this structure.

In [ ]:
from tensorflow.core.protobuf import saved_model_pb2
from tensorflow.core.framework import types_pb2

def describe_savedmodel(path):
    sm = saved_model_pb2.SavedModel()
    sm.ParseFromString((pathlib.Path(path) / "saved_model.pb").read_bytes())
    rows = []
    for mg in sm.meta_graphs:
        tags = list(mg.meta_info_def.tags)
        for sig_name, sd in mg.signature_def.items():
            for direction, tensors in (("input", sd.inputs), ("output", sd.outputs)):
                for name, ti in tensors.items():
                    shape = [d.size for d in ti.tensor_shape.dim] if not ti.tensor_shape.unknown_rank else "?"
                    rows.append({"metagraph tags": tags, "signature": sig_name, "direction": direction,
                                 "tensor name": name, "dtype": types_pb2.DataType.Name(ti.dtype),
                                 "shape": shape})
            if not sd.inputs and not sd.outputs:
                rows.append({"metagraph tags": tags, "signature": sig_name, "direction": "-",
                             "tensor name": "(no tensors: init op)", "dtype": "-", "shape": "-"})
    print(f"{len(sm.meta_graphs)} metagraph(s) found")
    return pd.DataFrame(rows)

def signature_io(path, sig="serving_default"):
    """Return (input_name, output_name, input_dtype) of a signature."""
    fn = tf.saved_model.load(str(path)).signatures[sig]
    in_specs = fn.structured_input_signature[1]
    in_name = list(in_specs.keys())[0]
    out_name = list(fn.structured_outputs.keys())[0]
    return in_name, out_name, in_specs[in_name].dtype

display_df = describe_savedmodel(v1_path)
print(display_df.to_string(index=False))
IN_NAME, OUT_NAME, _ = signature_io(v1_path)
print("\nInput name:", IN_NAME, "| output name:", OUT_NAME)

## 1.7 Inspect and test-run with `saved_model_cli`
`saved_model_cli` ships with TensorFlow. If the executable is not on the `PATH` (common in virtual environments),
the same tool runs as a Python module.

In [ ]:
def saved_model_cli(args):
    exe = shutil.which("saved_model_cli")
    base = exe if exe else f"{sys.executable} -m tensorflow.python.tools.saved_model_cli"
    return sh(f"{base} {args}")

cli_show = saved_model_cli(f'show --dir "{v1_path}" --all')

**What to record from the output above**
* tag-set `serve`, signature `serving_default`
* input: name `images`, `DT_FLOAT`, shape `(-1, 28, 28)` — `-1` means "any batch size"
* output: name `probs`, `DT_FLOAT`, shape `(-1, 10)` — one probability per digit class

(With the 2019 export the names were auto-generated, e.g. `flatten_input` / `dense_1`; naming them ourselves makes
client code readable and stable across retrains.)

Now a **test run** on three images saved to a `.npy` file. This is a quick sanity check of the *file on disk*
without writing any Python client — useful in CI, but not a serving solution (it starts TensorFlow for every call).

In [ ]:
X_cli = X_test[:3]
npy_path = WORK / "my_mnist_tests.npy"
np.save(npy_path, X_cli)
cli_run = saved_model_cli(f'run --dir "{v1_path}" --tag_set serve --signature_def serving_default '
                          f'--inputs {IN_NAME}="{npy_path}"')
print("\nExpected (in-process) probabilities, rounded:\n", np.round(model_v1.predict(X_cli, verbose=0), 2))

The CLI prints a `(3, 10)` array: **10 probabilities per image** that match the in-process predictions.

## 1.8 Bundle preprocessing *inside* the exported model

Version 1 expects pixels already divided by 255. Every client must remember to do that — and the first client that
forgets gets garbage predictions with no error (classic **training/serving skew**, Task 19). The fix is to put the
preprocessing **inside** the model, so clients send raw data:

* **no duplicated preprocessing code** in the web app, the mobile app and the batch job;
* **one artefact to update** — change the preprocessing and the model together;
* **no mismatch risk** between the preprocessing used in training and in serving.

### (a) Image model: raw pixels 0–255 in, probabilities out

In [ ]:
tf.random.set_seed(SEED)
raw_model = keras.Sequential([
    keras.Input(shape=(28, 28), name="images"),
    keras.layers.Rescaling(1 / 255.0),              # <- preprocessing now lives in the graph
    keras.layers.Flatten(),
    keras.layers.Dense(100, activation="relu"),
    keras.layers.Dense(10, activation="softmax"),
])
raw_model.compile(loss="sparse_categorical_crossentropy", optimizer="nadam", metrics=["accuracy"])
raw_model.fit(X_train_raw.astype("float32"), y_train, epochs=EPOCHS,
              validation_data=(X_valid_raw.astype("float32"), y_valid), verbose=2)

raw_path = WORK / "mnist_raw_pixels" / "0001"
if raw_path.exists():
    shutil.rmtree(raw_path)
write_savedmodel(raw_model, raw_path, MNIST_SPEC)

client_payload = X_test_raw[:5].astype("float32")      # what a client sends: untouched 0..255 pixels
fn = tf.saved_model.load(str(raw_path)).signatures["serving_default"]
probs = fn(images=tf.constant(client_payload))["probs"].numpy()
print("Client sent pixel range:", client_payload.min(), "to", client_payload.max())
print("Predicted:", probs.argmax(1), "| true:", y_test[:5])

### (b) Text model: raw strings in (TextVectorization + vocabulary bundled)
A second, tiny sentiment model shows the same idea for text. The `TextVectorization` layer's vocabulary becomes part
of the SavedModel (look for it in `assets/` or embedded as a constant), so clients just send sentences.

In [ ]:
texts = ["great movie loved it", "what a wonderful film", "brilliant acting and story", "really enjoyed this",
         "fantastic and moving", "best film this year", "superb direction", "i loved every minute",
         "terrible movie hated it", "what a boring film", "awful acting and story", "really disliked this",
         "dull and slow", "worst film this year", "poor direction", "i hated every minute"]
labels = np.array([1] * 8 + [0] * 8, dtype="float32")
TEXT_SPEC = tf.TensorSpec(shape=[None, 1], dtype=tf.string, name="images")   # name kept generic for the helper

text_model, text_path = None, WORK / "sentiment_model" / "0001"
try:
    vectorizer = keras.layers.TextVectorization(max_tokens=100, output_sequence_length=8)
    vectorizer.adapt(tf.constant(texts))
    tf.random.set_seed(SEED)
    text_model = keras.Sequential([
        keras.Input(shape=(1,), dtype="string"),
        vectorizer,
        keras.layers.Embedding(101, 8),
        keras.layers.GlobalAveragePooling1D(),
        keras.layers.Dense(1, activation="sigmoid"),
    ])
    text_model.compile(loss="binary_crossentropy", optimizer=keras.optimizers.Adam(0.05), metrics=["accuracy"])
    ds = tf.data.Dataset.from_tensor_slices((tf.constant(texts)[:, None], labels)).batch(8)
    text_model.fit(ds, epochs=40, verbose=0)
    if text_path.exists():
        shutil.rmtree(text_path)
    write_savedmodel(text_model, text_path, TEXT_SPEC)
    tfn = tf.saved_model.load(str(text_path)).signatures["serving_default"]
    queries = tf.constant([["loved this wonderful film"], ["boring and awful"]])
    print("Raw-string predictions (1 = positive):", tfn(images=queries)["probs"].numpy().ravel().round(3))
    print("\n".join(dir_tree(text_path)))
except Exception as e:
    print("Text-model export not supported by this TF/Keras build:", type(e).__name__, str(e)[:300])

## 1.9 The limits of SavedModels — a deliberate failure

A SavedModel stores a **graph of TensorFlow ops**. Anything that is *not* a TensorFlow op cannot be stored:
* `tf.py_function` / `tf.numpy_function` wrap **arbitrary Python**; the graph only keeps a pointer
  ("call Python callback #N"), and that callback only exists inside the process that created it.
* Dynamic Keras models (`dynamic=True`, Task 17) run eagerly in Python and therefore have no graph to export.

To prove it, we export a function containing a `tf.py_function`, then load and call it **in a fresh Python process**
(within the same process the callback still exists, which hides the problem — a classic trap).

In [ ]:
def python_scaling(x):                 # arbitrary NumPy/Python code
    return (x.numpy() / 255.0).astype("float32")

class PyFuncModel(tf.Module):
    @tf.function(input_signature=[tf.TensorSpec([None, 28, 28], tf.float32)])
    def serve(self, images):
        scaled = tf.py_function(python_scaling, [images], tf.float32)
        return {"mean_pixel": tf.reduce_mean(scaled, axis=[1, 2])}

pyf_path = WORK / "pyfunc_model" / "0001"
if pyf_path.exists():
    shutil.rmtree(pyf_path)
try:
    pm = PyFuncModel()
    tf.saved_model.save(pm, str(pyf_path), signatures={"serving_default": pm.serve})
    print("Export step: succeeded (the graph only records a *reference* to the Python callback)")
except Exception as e:
    print("Export step failed:", type(e).__name__, str(e)[:300])

probe = f"""
import os; os.environ['TF_CPP_MIN_LOG_LEVEL']='3'
import numpy as np, tensorflow as tf
f = tf.saved_model.load(r'{pyf_path}').signatures['serving_default']
print(f(images=tf.zeros([1,28,28]))['mean_pixel'].numpy())
"""
r = subprocess.run([sys.executable, "-c", probe], capture_output=True, text=True, timeout=300)
err_lines = [l for l in (r.stderr or "").splitlines() if "Error" in l or "error" in l or "callback" in l]
print("Fresh-process call return code:", r.returncode)
print("Key error lines:\n", "\n".join(err_lines[-4:]) if err_lines else (r.stdout or r.stderr)[-500:])

**What failed and why:** the export "works", but calling the model in a new process raises an error saying the
py_func **callback cannot be found** (a `NotFoundError`/`UnknownError` mentioning `pyfunc_…`). The Python function
was never saved — TF Serving (C++, no Python at all) would fail in exactly the same way.

**Alternatives:** rewrite the logic with TensorFlow ops (here: `Rescaling` or `x / 255.0`); or, if the model truly
needs Python, serve it with a custom web service (FastAPI/Flask + Gunicorn) or a Python-native server
(e.g. KServe / Ray Serve / TorchServe-style custom handlers) and accept the extra operational work.

## 1.10 `export_model()` — the reusable export helper with a post-export self-test

Requirements: create the versioned directory, export, **reload from disk**, run the same 5 inputs, and assert the
predictions match the in-memory model to `1e-5`. Two production details are added:
* it writes to a hidden temporary folder first and **renames** it into place at the end, so a watching server never
  sees a half-written version (TF Serving ignores non-numeric folder names);
* it refuses to overwrite an existing version unless `overwrite=True`.

In [ ]:
def export_model(model, name, version, input_spec=MNIST_SPEC, test_inputs=None,
                 root=SERVING_ROOT, atol=1e-5, overwrite=True):
    """Export `model` to root/name/<version:04d>, then reload it and verify predictions."""
    base = pathlib.Path(root) / name
    final = base / f"{int(version):04d}"
    tmp = base / f".tmp_{int(version):04d}"
    base.mkdir(parents=True, exist_ok=True)
    if final.exists():
        if not overwrite:
            raise FileExistsError(f"{final} already exists")
        shutil.rmtree(final)
    if tmp.exists():
        shutil.rmtree(tmp)
    write_savedmodel(model, tmp, input_spec)

    # ---- post-export self-test (on the files in the temp folder, before they become visible) ----
    if test_inputs is not None:
        fn = tf.saved_model.load(str(tmp)).signatures["serving_default"]
        in_name = list(fn.structured_input_signature[1].keys())[0]
        x = tf.constant(test_inputs, dtype=input_spec.dtype)
        from_disk = fn(**{in_name: x})["probs"].numpy()
        in_memory = np.asarray(model(x, training=False))
        max_diff = float(np.abs(from_disk - in_memory).max())
        assert max_diff <= atol, f"Self-test FAILED: max diff {max_diff:.2e} > {atol}"
        print(f"[export_model] self-test passed: max |disk - memory| = {max_diff:.2e} (tol {atol})")
    os.replace(tmp, final)                     # atomic publish
    print(f"[export_model] published {final}")
    return final

v1_path = export_model(model_v1, MODEL_NAME, 1, test_inputs=X_new)

### Phase 1 — summary / deliverable
* ✅ Versioned SavedModel `serving/my_mnist_model/0001` with an annotated directory tree
* ✅ Loaded two ways (generic object vs Keras) with identical predictions
* ✅ Metagraph / tag / signature structure shown, `saved_model_cli show` and `run` output recorded
* ✅ Preprocessing inside the model: raw-pixel image model and raw-string text model
* ✅ `py_function` limitation reproduced in a fresh process, with alternatives
* ✅ `export_model()` helper with atomic publish + 1e-5 self-test (reused in every later phase)

---
# Phase 2 — TF Serving with Docker: REST vs. gRPC

**Goal of this phase:** run the real model server, call it with both protocols, measure the cost of each, and make
the client robust to the errors you will meet in practice.

## 2.1 What TF Serving is
TF Serving is Google's production model server, written in C++. It:
* sustains **high load** with low overhead (no Python in the request path);
* serves **several models, and several versions of each**, from one process;
* **watches** the model base directory and loads new versions automatically (and unloads old ones);
* exposes two APIs: **REST** (JSON over HTTP, port 8501) and **gRPC** (protocol buffers over HTTP/2, port 8500);
* offers request **batching**, model **warm-up**, monitoring endpoints and config-file-driven multi-model hosting.

## 2.2 Why Docker
The TensorFlow team recommends the official `tensorflow/serving` image because it is **simple** (one command),
**does not pollute** your system with native dependencies, and is **built for performance** (and a `-gpu` variant
exists). Install Docker from <https://docs.docker.com/get-docker/>, then `docker pull tensorflow/serving`.

> **Fallback used in this notebook:** when Docker is not available (e.g. on Colab), a small pure-Python
> `LocalVersionedServer` imitates the parts of TF Serving we need — it watches the same version folders and serves the
> latest (or a pinned) version. Every result table records which backend produced it.

In [ ]:
DOCKER = shutil.which("docker") is not None and sh("docker info", quiet=True).returncode == 0
TFS_CONTAINER = "tfs_mnist"
REST_PORT, GRPC_PORT = 8501, 8500
print("Docker available:", DOCKER)
if DOCKER:
    sh("docker pull tensorflow/serving", timeout=1800)

## 2.3 Run the server — and what every flag means

The interactive form you would type in a terminal:
```bash
docker run -it --rm -p 8500:8500 -p 8501:8501 \
    -v "$PWD/task23_work/serving/my_mnist_model:/models/my_mnist_model" \
    -e MODEL_NAME=my_mnist_model \
    tensorflow/serving
```
| Flag | Meaning |
|---|---|
| `-it` | interactive + terminal: logs stream to your console and Ctrl-C stops the server |
| `--rm` | delete the container when it stops (no leftover containers) |
| `-p 8500:8500` | publish the container's **gRPC** port on the host |
| `-p 8501:8501` | publish the container's **REST** port on the host |
| `-v host:/models/my_mnist_model` | mount the host's model base directory into the container where the server expects it |
| `-e MODEL_NAME=my_mnist_model` | which model to serve; by default the server loads the **highest version number** it finds |
| `tensorflow/serving` | the image name; anything after it is passed to `tensorflow_model_server` as extra flags |

Inside a notebook we use `-d` (detached, runs in the background) plus `--name` instead of `-it --rm`, and read the
logs with `docker logs`.

In [ ]:
def docker_run_serving(container, mounts, rest_port=8501, grpc_port=8500, env=None, args=""):
    """Start TF Serving detached. mounts = [(host_path, container_path), ...]"""
    sh(f"docker rm -f {container}", quiet=True)
    mount_str = " ".join(f'-v "{pathlib.Path(h).resolve()}:{c}"' for h, c in mounts)
    env_str = " ".join(f"-e {k}={v}" for k, v in (env or {}).items())
    return sh(f"docker run -d --name {container} -p {grpc_port}:8500 -p {rest_port}:8501 "
              f"{mount_str} {env_str} tensorflow/serving {args}")

def model_status(model_name, rest_port=REST_PORT):
    r = requests.get(f"http://localhost:{rest_port}/v1/models/{model_name}", timeout=2)
    r.raise_for_status()
    return r.json()["model_version_status"]

def served_versions(model_name, rest_port=REST_PORT):
    return sorted(int(s["version"]) for s in model_status(model_name, rest_port) if s["state"] == "AVAILABLE")

def wait_ready(model_name, rest_port=REST_PORT, timeout=120, want_version=None):
    t0 = time.time()
    while time.time() - t0 < timeout:
        try:
            vs = served_versions(model_name, rest_port)
            if vs and (want_version is None or want_version in vs):
                return True
        except Exception:
            pass
        time.sleep(0.5)
    return False

SERVING_UP = False
if DOCKER and REQUESTS_OK:
    docker_run_serving(TFS_CONTAINER, [(MODEL_ROOT, f"/models/{MODEL_NAME}")],
                       env={"MODEL_NAME": MODEL_NAME})
    SERVING_UP = wait_ready(MODEL_NAME)
    print("Server ready:", SERVING_UP)
    sh(f"docker logs --tail 25 {TFS_CONTAINER}")
else:
    print("TF Serving not started (no Docker / requests) -> the local fallback server will be used.")

**What to look for in the log excerpt:** `Successfully loaded servable version {name: my_mnist_model version: 1}`,
then `Running gRPC ModelServer at 0.0.0.0:8500` and `Exporting HTTP/REST API at:localhost:8501`.

## 2.4 The local fallback server (only used without Docker)
It mirrors TF Serving's behaviour that matters here: it rescans the base directory on every call, serves the highest
complete version by default, can be asked for a specific version, and forgets versions that were deleted.

In [ ]:
class LocalVersionedServer:
    """Pure-Python stand-in for TF Serving's file-system watcher (no Docker needed)."""
    def __init__(self, base_dir, signature="serving_default"):
        self.base_dir = pathlib.Path(base_dir)
        self.signature = signature
        self._cache = {}
        self._lock = threading.Lock()

    def versions(self):
        if not self.base_dir.exists():
            return []
        return sorted(int(p.name) for p in self.base_dir.iterdir()
                      if p.is_dir() and p.name.isdigit() and (p / "saved_model.pb").exists())

    def _get(self, v):
        with self._lock:
            for old in [k for k in self._cache if k not in self.versions()]:
                del self._cache[old]                                  # "unload" deleted versions
            if v not in self._cache:
                fn = tf.saved_model.load(str(self.base_dir / f"{v:04d}")).signatures[self.signature]
                in_name = list(fn.structured_input_signature[1].keys())[0]
                out_name = list(fn.structured_outputs.keys())[0]
                dtype = fn.structured_input_signature[1][in_name].dtype
                self._cache[v] = (fn, in_name, out_name, dtype)
            return self._cache[v]

    def predict(self, X, version=None):
        vs = self.versions()
        if not vs:
            raise RuntimeError("no servable versions found")
        v = int(version) if version is not None else vs[-1]
        if v not in vs:
            raise KeyError(f"Servable not found for version {v}")
        fn, in_name, out_name, dtype = self._get(v)
        return fn(**{in_name: tf.constant(X, dtype=dtype)})[out_name].numpy(), v

## 2.5 Query with REST

Request body for TF Serving's *predict* API, in **row format**:
```json
{"signature_name": "serving_default", "instances": [[[0.0, 0.0, ...], ...], ...]}
```
POST to `http://localhost:8501/v1/models/my_mnist_model:predict`, call `raise_for_status()`, read `"predictions"`.

In [ ]:
def rest_predict(X, model_name=MODEL_NAME, version=None, signature="serving_default",
                 rest_port=REST_PORT, timeout=30):
    url = f"http://localhost:{rest_port}/v1/models/{model_name}"
    if version is not None:
        url += f"/versions/{int(version)}"
    payload = {"signature_name": signature, "instances": np.asarray(X).tolist()}
    response = requests.post(url + ":predict", json=payload, timeout=timeout)
    response.raise_for_status()
    return np.array(response.json()["predictions"])

local_mnist = LocalVersionedServer(MODEL_ROOT)
if SERVING_UP:
    y_rest = rest_predict(X_new)
    backend = "TF Serving (REST)"
else:
    y_rest, _ = local_mnist.predict(X_new)
    backend = "local fallback"
print("Backend:", backend)
print(np.round(y_rest, 3))
print("Matches Phase 1 predictions:", np.allclose(y_rest, model_v1.predict(X_new, verbose=0), atol=1e-5))

## 2.6 How expensive is JSON? (payload size)

A float32 needs **4 bytes** in binary. In JSON the same number is text such as `0.011764706112444401` — often
**15–20 characters**, i.e. 120–160 bits instead of 32. Below we measure it on 100 real images (lots of exact zeros in
MNIST make it *look* better than for natural images, so we also measure random floats).

In [ ]:
def json_ratio(X):
    js = json.dumps({"signature_name": "serving_default", "instances": X.tolist()}).encode()
    raw = X.astype("float32").nbytes
    return len(js), raw, len(js) / raw

rows = []
for label, X in [("100 MNIST images (many zeros)", X_test[:100]),
                 ("100 x 28x28 random floats", np.random.rand(100, 28, 28).astype("float32"))]:
    js, raw, ratio = json_ratio(X)
    rows.append({"payload": label, "JSON bytes": js, "raw float32 bytes": raw, "JSON / raw": round(ratio, 2)})
nonzero = X_test[:100][X_test[:100] > 0]
chars = np.mean([len(json.dumps(float(v))) for v in nonzero[:2000]])
payload_df = pd.DataFrame(rows)
print(payload_df.to_string(index=False))
print(f"\nAverage characters per non-zero float in JSON: {chars:.1f}  (≈ {chars*8:.0f} bits vs 32 bits binary)")
RESULTS["json_payload"] = payload_df

## 2.7 Query with gRPC

gRPC sends a binary `PredictRequest` protocol buffer. Steps:
1. build `PredictRequest`; set `model_spec.name`, `model_spec.signature_name` (and optionally `model_spec.version`);
2. put the input under the signature's **input name** using `tf.make_tensor_proto`;
3. open a channel to port 8500 and call `PredictionServiceStub(channel).Predict(request, timeout)` (blocking);
4. convert `response.outputs[output_name]` back with `tf.make_ndarray`.

In [ ]:
try:
    import grpc
    from tensorflow_serving.apis import predict_pb2, prediction_service_pb2_grpc
    GRPC_OK = True
except Exception as e:
    GRPC_OK = False
    print("gRPC client libraries unavailable:", e)

_channels = {}
def grpc_predict(X, model_name=MODEL_NAME, version=None, signature="serving_default",
                 grpc_port=GRPC_PORT, input_name=None, output_name=None, timeout=30.0):
    input_name = input_name or IN_NAME
    output_name = output_name or OUT_NAME
    request = predict_pb2.PredictRequest()
    request.model_spec.name = model_name
    request.model_spec.signature_name = signature
    if version is not None:
        request.model_spec.version.value = int(version)
    X = np.asarray(X)
    if X.dtype.kind == "f":
        X = X.astype("float32")
    request.inputs[input_name].CopyFrom(tf.make_tensor_proto(X))
    if grpc_port not in _channels:                      # reuse channels: creating one per call is slow
        _channels[grpc_port] = grpc.insecure_channel(f"localhost:{grpc_port}",
                                                     options=[("grpc.max_receive_message_length", 256 << 20),
                                                              ("grpc.max_send_message_length", 256 << 20)])
    stub = prediction_service_pb2_grpc.PredictionServiceStub(_channels[grpc_port])
    response = stub.Predict(request, timeout=timeout)
    return tf.make_ndarray(response.outputs[output_name])

if SERVING_UP and GRPC_OK:
    y_grpc = grpc_predict(X_new)
    print("gRPC == REST:", np.allclose(y_grpc, y_rest, atol=1e-6))
    req = predict_pb2.PredictRequest(); req.inputs[IN_NAME].CopyFrom(tf.make_tensor_proto(X_new))
    print("Serialized gRPC request for 5 images:", len(req.SerializeToString()), "bytes",
          "| JSON:", len(json.dumps({"instances": X_new.tolist()})), "bytes")
else:
    print("gRPC demo skipped (needs the Docker server + tensorflow-serving-api). "
          "The function above is the complete client.")

## 2.8 Security caveat
`grpc.insecure_channel` and plain `http://` send data **unencrypted and unauthenticated** — fine on `localhost`,
never acceptable on a network. In production you would add:
* **TLS**: `grpc.secure_channel(host, grpc.ssl_channel_credentials(root_certs))`; TF Serving accepts an SSL config
  (`--ssl_config_file`) for gRPC, and REST is usually put behind an HTTPS reverse proxy / ingress;
* **authentication/authorization** at a gateway (API keys, OAuth/JWT, mTLS between services);
* **network isolation** (private subnet, firewall: only the API gateway may reach ports 8500/8501);
* **rate limiting and request-size limits**, and no direct exposure of the model server to the internet.

## 2.9 Benchmark REST vs gRPC (Task 18 rules: warm-up excluded, ≥5 repeats, mean ± std)

In [ ]:
def protocol_benchmark(X_pool, batch_sizes, model_name, input_name, output_name,
                       rest_port=REST_PORT, grpc_port=GRPC_PORT, local_server=None, repeats=5):
    rows = []
    for bs in batch_sizes:
        Xb = X_pool[:bs]
        json_bytes = len(json.dumps({"signature_name": "serving_default", "instances": Xb.tolist()}).encode())
        row = {"batch": bs, "JSON payload": human(json_bytes), "binary float32": human(Xb.astype("float32").nbytes)}
        if SERVING_UP:
            m, s = bench(lambda: rest_predict(Xb, model_name, rest_port=rest_port), repeats=repeats)
            row["REST ms"] = f"{m:.1f} ± {s:.1f}"
            if GRPC_OK:
                m, s = bench(lambda: grpc_predict(Xb, model_name, grpc_port=grpc_port,
                                                  input_name=input_name, output_name=output_name), repeats=repeats)
                row["gRPC ms"] = f"{m:.1f} ± {s:.1f}"
        else:
            m, s = bench(lambda: local_server.predict(Xb), repeats=repeats)
            row["in-process ms (fallback)"] = f"{m:.1f} ± {s:.1f}"
            m, s = bench(lambda: json.loads(json.dumps({"instances": Xb.tolist()})), repeats=repeats)
            row["JSON encode+decode ms"] = f"{m:.1f} ± {s:.1f}"
        rows.append(row)
    return pd.DataFrame(rows)

mnist_bench = protocol_benchmark(X_test, [1, 32, 256], MODEL_NAME, IN_NAME, OUT_NAME, local_server=local_mnist)
print("MNIST model (backend:", "TF Serving" if SERVING_UP else "local fallback", ")")
print(mnist_bench.to_string(index=False))
RESULTS["mnist_protocol_bench"] = mnist_bench

### Bigger inputs: 224×224×3 images
Each image is 150,528 floats, so the JSON overhead grows dramatically. We export a light image model with
preprocessing inside (it only needs to *accept* that input — we are measuring transport, not accuracy) and serve it
from a second container on ports 8510/8511. The capstone repeats this with the real Flowers model.

In [ ]:
BIG_NAME = "big_input_model"
BIG_SPEC = tf.TensorSpec([None, 224, 224, 3], tf.float32, name="images")
tf.random.set_seed(SEED)
big_model = keras.Sequential([
    keras.Input(shape=(224, 224, 3)),
    keras.layers.Rescaling(1 / 255.0),
    keras.layers.Conv2D(8, 3, strides=4, activation="relu"),
    keras.layers.GlobalAveragePooling2D(),
    keras.layers.Dense(5, activation="softmax"),
])
X_big = np.random.randint(0, 256, size=(16, 224, 224, 3)).astype("float32")
export_model(big_model, BIG_NAME, 1, input_spec=BIG_SPEC, test_inputs=X_big[:2])

BIG_REST, BIG_GRPC = 8511, 8510
if SERVING_UP:
    docker_run_serving("tfs_big", [(SERVING_ROOT / BIG_NAME, f"/models/{BIG_NAME}")],
                       rest_port=BIG_REST, grpc_port=BIG_GRPC, env={"MODEL_NAME": BIG_NAME},
                       args="--rest_api_num_threads=8")
    wait_ready(BIG_NAME, BIG_REST)
big_bench = protocol_benchmark(X_big, [1, 4, 16], BIG_NAME, "images", "probs",
                               rest_port=BIG_REST, grpc_port=BIG_GRPC,
                               local_server=LocalVersionedServer(SERVING_ROOT / BIG_NAME), repeats=5)
print(big_bench.to_string(index=False))
RESULTS["big_protocol_bench"] = big_bench
if SERVING_UP:
    sh("docker rm -f tfs_big", quiet=True)

**Practical rule**
* **REST** — small or occasional payloads, browser/any-language clients, easy debugging with `curl`.
* **gRPC** — large tensors (images, audio, embeddings), high request rates, service-to-service traffic where both
  ends are under your control. The gap grows with payload size because JSON encoding/parsing time and bytes on the
  wire grow ~5–8× faster than the binary tensor.

## 2.10 Two REST mitigations: Base64 and gzip
1. **Base64 of binary data** — TF Serving's REST API accepts `{"b64": "..."}` for **string (DT_STRING) inputs**, e.g.
   JPEG bytes decoded *inside* the model. Base64 costs only +33 % over the raw bytes, far less than JSON text floats.
   (A float tensor itself cannot be sent as b64 to a float input — you design the signature to take bytes.)
2. **gzip the request body** — compresses the repeated characters in JSON very well.

In [ ]:
Xb = X_test[:100]
js = json.dumps({"instances": Xb.tolist()}).encode()
raw = Xb.astype("float32").tobytes()
b64 = base64.b64encode(raw)
gz = gzip.compress(js)
png_like = base64.b64encode(gzip.compress(X_test_raw[:100].tobytes()))   # what a compressed-uint8 client could send
mitig_df = pd.DataFrame([
    {"encoding": "JSON floats", "bytes": len(js)},
    {"encoding": "raw float32 (gRPC-like)", "bytes": len(raw)},
    {"encoding": "Base64(raw float32)", "bytes": len(b64)},
    {"encoding": "gzip(JSON)", "bytes": len(gz)},
    {"encoding": "Base64(gzip(uint8 pixels))", "bytes": len(png_like)},
])
mitig_df["vs raw"] = (mitig_df["bytes"] / len(raw)).round(2)
print(mitig_df.to_string(index=False))

if SERVING_UP:   # does this server accept a gzip-encoded body?
    try:
        r = requests.post(f"http://localhost:{REST_PORT}/v1/models/{MODEL_NAME}:predict",
                          data=gzip.compress(json.dumps({"instances": X_new.tolist()}).encode()),
                          headers={"Content-Encoding": "gzip", "Content-Type": "application/json"}, timeout=10)
        print("\ngzip request -> HTTP", r.status_code, "| ok" if r.ok else r.text[:200])
    except Exception as e:
        print("gzip request failed:", e)

## 2.11 Failure drills and a client wrapper that explains them

Four mistakes every team makes at least once. A good client turns the server's raw error into an exception that says
**what is wrong and how to fix it**.

In [ ]:
class ServingError(Exception):
    """Base class for clear, actionable serving errors."""
class ModelNotFound(ServingError): pass
class BadSignature(ServingError): pass
class BadInputShape(ServingError): pass
class BadInputType(ServingError): pass

def classify_error(message, status=None):
    m = message.lower()
    if "servable not found" in m or "could not find any versions" in m or "not found for version" in m:
        return ModelNotFound
    if "signature" in m and ("not found" in m or "serving signature name" in m):
        return BadSignature
    if any(k in m for k in ("is not of expected type", "could not convert", "cannot convert",
                            "unable to convert", "could not parse", "string")):
        return BadInputType
    if any(k in m for k in ("shape", "reshape", "incompatible", "dimension", "rank")):
        return BadInputShape
    if "dtype" in m:
        return BadInputType
    return ServingError

class SafeClient:
    def __init__(self, model_name=MODEL_NAME, local=None):
        self.model_name, self.local = model_name, local
    def predict(self, X, signature="serving_default", model_name=None, version=None):
        name = model_name or self.model_name
        try:
            if SERVING_UP:
                url = f"http://localhost:{REST_PORT}/v1/models/{name}:predict"
                r = requests.post(url, json={"signature_name": signature, "instances": np.asarray(X).tolist()},
                                  timeout=10)
                if not r.ok:
                    raise RuntimeError(f"HTTP {r.status_code}: {r.json().get('error', r.text)}")
                return np.array(r.json()["predictions"])
            # ---- local fallback emulating the same failure modes ----
            if name != self.model_name:
                raise RuntimeError(f"Servable not found for request: Latest({name})")
            fn = tf.saved_model.load(str(self.local.base_dir / f"{self.local.versions()[-1]:04d}"))
            if signature not in fn.signatures:
                raise RuntimeError(f"Serving signature name: \"{signature}\" not found in signature def")
            out, _ = self.local.predict(X, version)
            return out
        except Exception as e:
            msg = str(e)
            cls = classify_error(msg)
            hint = {ModelNotFound: "check the model name/version and that the server loaded it",
                    BadSignature: "use 'serving_default' or a signature listed by saved_model_cli",
                    BadInputShape: f"expected input shape {MNIST_SPEC.shape}",
                    BadInputType: f"expected dtype {MNIST_SPEC.dtype.name}",
                    ServingError: "see server logs"}[cls]
            lines = [l.strip() for l in msg.splitlines() if l.strip()]
            keyw = ("shape", "dtype", "convert", "not found", "signature", "expected", "incompatible", "type")
            useful = next((l for l in lines[1:] if any(k in l.lower() for k in keyw)), lines[0] if lines else msg)
            raise cls(f"{cls.__name__}: {useful[:160]} -> hint: {hint}") from None

client = SafeClient(local=local_mnist)
drills = {
    "(a) wrong shape (27x27)":      dict(X=X_test[:2, :27, :27]),
    "(b) wrong dtype (strings)":    dict(X=np.array([[["a"] * 28] * 28])),
    "(c) wrong signature name":     dict(X=X_new, signature="predict_digits"),
    "(d) model that doesn't exist": dict(X=X_new, model_name="no_such_model"),
}
drill_rows = []
for label, kwargs in drills.items():
    try:
        client.predict(**kwargs)
        drill_rows.append({"drill": label, "exception": "(no error!)", "message": ""})
    except ServingError as e:
        drill_rows.append({"drill": label, "exception": type(e).__name__, "message": str(e)[:150]})
drill_df = pd.DataFrame(drill_rows)
print("Backend:", "TF Serving" if SERVING_UP else "local fallback")
print(drill_df.to_string(index=False))
RESULTS["failure_drills"] = drill_df

### Phase 2 — summary / deliverable
* ✅ TF Serving container started (or local fallback), log excerpt shows version 1 loaded and both endpoints up
* ✅ REST and gRPC clients returning identical probabilities
* ✅ Payload-size table (JSON ≈ 4–5× raw floats for real data) and latency tables for MNIST and 224×224×3 inputs
* ✅ Base64/gzip mitigations measured
* ✅ Failure-drill table + `SafeClient` wrapper raising `ModelNotFound / BadSignature / BadInputShape / BadInputType`

---
# Phase 3 — Versioning, Rollback, Batching & Scaling Out

**Goal of this phase:** practise the day-2 operations of a model service: ship a new version without downtime,
prove it from the client side, roll back, run a canary/A-B split, batch requests for throughput, warm the model up,
and plan for horizontal scale. Ends with a runbook.

## 3.1 Train version 2 and watch the server pick it up

Version 2 is a deliberately *different* model (an extra hidden layer, a few more epochs) so its outputs differ from v1.

### What TF Serving does when `0002` appears (default "availability-preserving" policy)
1. the file-system poller notices a new numeric folder (default poll ≈ every second);
2. it **reserves resources**, reads the SavedModel, and **loads version 2**;
3. once v2 is ready, new requests are routed to it; v1 is **quiesced** — it finishes the requests already in flight;
4. when v1 has no pending requests it is **unloaded** and its memory is freed.

For a short time **both versions are in memory**. On a CPU host that's usually fine; on a GPU it can be the
difference between fitting and an out-of-memory crash. The alternative ("resource-preserving") policy **unloads the
old version first and then loads the new one**: no double memory, but a brief window in which the model is
unavailable. In TF Serving this is the server's *aspired-version policy*
(`AvailabilityPreservingPolicy` is the stock default, `ResourcePreservingPolicy` is the unload-first variant chosen
when embedding/customising the server); managed platforms expose the same trade-off as rollout settings.

In [ ]:
tf.random.set_seed(SEED + 1)
model_v2 = build_mnist_model(extra_layer=True)
model_v2.fit(X_train, y_train, epochs=EPOCHS + 1, validation_data=(X_valid, y_valid), verbose=2)
v2_test_acc = model_v2.evaluate(X_test, y_test, verbose=0)[1]
print(f"v1 test acc {v1_test_acc:.4f} | v2 test acc {v2_test_acc:.4f}")

## 3.2 Prove the switch from the client side
A polling client asks "which version answered?" every 100 ms while, in a background thread, we publish `0002` with
our `export_model()` helper. With Docker we read the server's model-status endpoint; with the fallback we read the
local server. The plot is the evidence of a zero-downtime switch.

In [ ]:
def current_version():
    if SERVING_UP:
        return max(served_versions(MODEL_NAME))
    return local_mnist.versions()[-1]

def poll_versions(duration_s, action=None, action_at_s=1.5, period=0.1):
    """Poll every `period` seconds; run `action` in a thread after `action_at_s` seconds."""
    log, errors = [], 0
    if action:
        threading.Timer(action_at_s, action).start()
    t0 = time.time()
    while time.time() - t0 < duration_s:
        t = time.time() - t0
        try:
            v = current_version()
            (rest_predict(X_new[:1]) if SERVING_UP else local_mnist.predict(X_new[:1]))  # real traffic too
            log.append((t, v))
        except Exception:
            errors += 1
            log.append((t, 0))
        time.sleep(period)
    return log, errors

deploy_log, deploy_errors = poll_versions(
    duration_s=10 if SERVING_UP else 5,
    action=lambda: export_model(model_v2, MODEL_NAME, 2, test_inputs=X_new))
if SERVING_UP:
    sh(f"docker logs --tail 15 {TFS_CONTAINER}")

In [ ]:
def plot_versions(log, title, marker_at=1.5):
    t, v = zip(*log)
    plt.figure(figsize=(8, 2.6))
    plt.step(t, v, where="post", lw=2)
    plt.axvline(marker_at, ls="--", c="grey", label="new folder published")
    plt.yticks([0, 1, 2], ["error", "v1", "v2"])
    plt.xlabel("seconds"); plt.title(title); plt.legend(loc="lower right"); plt.grid(alpha=0.3)
    plt.show()

plot_versions(deploy_log, f"Client view while deploying v2 (failed requests: {deploy_errors})")

**Interpretation:** the line steps from v1 to v2 shortly after the folder appears (the delay is the poll interval plus
load time), and there are **no failed requests** — the old version kept answering until the new one was ready.

## 3.3 Roll back
Rolling back is just deleting the folder of the bad version. The server sees that `0002` disappeared, re-loads the
newest remaining version (`0001`) and routes traffic back.

In [ ]:
def delete_v2():
    shutil.rmtree(MODEL_ROOT / "0002", ignore_errors=True)

rollback_log, rollback_errors = poll_versions(duration_s=10 if SERVING_UP else 5, action=delete_v2)
plot_versions(rollback_log, f"Client view during rollback (failed requests: {rollback_errors})")
print("Now serving version:", current_version())

**Why rollback is this simple:** a version *is* a directory and the server's job is to converge to "serve the newest
directory" — so any change to the directory is a deployment, and deleting is a rollback. In a real system you would
not do this by hand:
* keep every version in an artefact store (bucket) so rollback = re-publish, never "retrain";
* **automated health checks** (a canned request with known output) right after load;
* **monitoring thresholds** (p99 latency, error rate, prediction-distribution shift) that trigger an automatic rollback;
* an audit trail of who deployed what, when.

## 3.4 Pin versions and run a 90/10 canary (A/B) test
To serve two versions at once TF Serving needs a **model config file** with a `specific` version policy.
Clients then pick a version:
* REST: `POST /v1/models/my_mnist_model/versions/2:predict`
* gRPC: `request.model_spec.version.value = 2`

Our client randomly sends **10 %** of requests to v2 and 90 % to v1. Task 13's A/B rules apply: the
**randomisation unit** should normally be the *user* (so one user always sees one version — here we use a hashed
synthetic user id), and we watch **guardrail metrics** (latency, errors) as well as the quality metric (accuracy on
labelled traffic).

In [ ]:
export_model(model_v2, MODEL_NAME, 2, test_inputs=X_new)           # bring v2 back for the experiment
models_config = SERVING_ROOT / "models.config"
models_config.write_text(f"""
model_config_list {{
  config {{
    name: "{MODEL_NAME}"
    base_path: "/models/{MODEL_NAME}"
    model_platform: "tensorflow"
    model_version_policy {{ specific {{ versions: 1 versions: 2 }} }}
  }}
}}
""".strip())
print(models_config.read_text())

if SERVING_UP:
    docker_run_serving(TFS_CONTAINER, [(SERVING_ROOT, "/models")],
                       args="--model_config_file=/models/models.config --model_config_file_poll_wait_seconds=5")
    print("v1 & v2 ready:", wait_ready(MODEL_NAME, want_version=2) and wait_ready(MODEL_NAME, want_version=1))
    print("Available versions:", served_versions(MODEL_NAME))

In [ ]:
def predict_version(X, version):
    if SERVING_UP:
        return rest_predict(X, version=version)
    return local_mnist.predict(X, version=version)[0]

def assign_version(user_id, canary_share=0.10):
    # deterministic per user: same user -> same version on every request
    import zlib   # stable hash (Python's built-in hash() changes between processes)
    return 2 if (zlib.crc32(f"exp-mnist-v2:{user_id}".encode()) % 1000) / 1000 < canary_share else 1

rng = np.random.default_rng(SEED)
N_REQ = 600 if QUICK else 3000
ab_rows = []
for i in range(N_REQ):
    user_id = int(rng.integers(0, 5000))
    idx = int(rng.integers(0, len(X_test)))
    v = assign_version(user_id)
    t0 = time.perf_counter()
    probs = predict_version(X_test[idx:idx + 1], v)
    ab_rows.append({"version": v, "latency_ms": (time.perf_counter() - t0) * 1000,
                    "correct": int(np.argmax(probs) == y_test[idx])})
ab = pd.DataFrame(ab_rows)
ab_summary = ab.groupby("version").agg(requests=("correct", "size"), traffic_share=("correct", "size"),
                                      accuracy=("correct", "mean"),
                                      p50_ms=("latency_ms", "median"),
                                      p95_ms=("latency_ms", lambda s: s.quantile(0.95)))
ab_summary["traffic_share"] = (ab_summary["traffic_share"] / len(ab)).round(3)
print("Backend:", "TF Serving" if SERVING_UP else "local fallback")
print(ab_summary.round(4).to_string())
RESULTS["ab_test"] = ab_summary

**Reading the result as a canary:** with only ~60 requests on v2 the accuracy difference has a wide confidence
interval — a canary is mainly a **safety** check (no errors, no latency regression), not a precise quality
comparison. The decision rule would be: *promote* v2 to 50 % then 100 % if guardrails stay green for the agreed period
and the quality metric is not significantly worse; otherwise *roll back* (delete `0002` or remove it from the config).

## 3.5 Automatic batching

A GPU does nearly the same amount of work for 1 image as for 64, so serving requests one by one wastes it.
With `--enable_batching`, TF Serving holds incoming requests for up to `batch_timeout_micros`, merges them into one
batch (up to `max_batch_size`), runs the model **once**, and splits the result back to each caller.
**Trade-off:** a longer wait window forms bigger batches (↑ throughput) but every request may wait that long (↑ latency).

The batching parameters file:
```
max_batch_size { value: 128 }        # largest merged batch
batch_timeout_micros { value: 2000 } # how long to wait for more requests
max_enqueued_batches { value: 1000 } # queue size before rejecting
num_batch_threads { value: 4 }       # parallel batches being processed
```
We load-test with **50 concurrent clients** in three configurations: batching off, on with 1 ms, on with 5 ms.
Without Docker, a Python `MicroBatcher` that implements the same algorithm is used.

In [ ]:
class MicroBatcher:
    """Minimal dynamic batcher: same idea as TF Serving's --enable_batching."""
    def __init__(self, fn, max_batch=128, timeout_s=0.002):
        self.fn, self.max_batch, self.timeout = fn, max_batch, timeout_s
        self.q, self._stop = queue.Queue(), False
        self.batch_sizes = []
        self.thread = threading.Thread(target=self._loop, daemon=True)
        self.thread.start()
    def __call__(self, x):
        fut = cf.Future()
        self.q.put((x, fut))
        return fut.result()
    def _loop(self):
        while not self._stop:
            try:
                items = [self.q.get(timeout=0.1)]
            except queue.Empty:
                continue
            deadline = time.perf_counter() + self.timeout
            while len(items) < self.max_batch:
                remaining = deadline - time.perf_counter()
                if remaining <= 0:
                    break
                try:
                    items.append(self.q.get(timeout=remaining))
                except queue.Empty:
                    break
            X = np.concatenate([x for x, _ in items])
            self.batch_sizes.append(len(items))
            try:
                out = self.fn(X)
                start = 0
                for x, fut in items:
                    fut.set_result(out[start:start + len(x)]); start += len(x)
            except Exception as e:
                for _, fut in items:
                    fut.set_exception(e)
    def close(self):
        self._stop = True

def load_test(predict_one, n_clients=50, requests_per_client=20):
    lat = []
    lock = threading.Lock()
    def client(cid):
        rng_ = np.random.default_rng(cid)
        for _ in range(requests_per_client):
            i = int(rng_.integers(0, len(X_test)))
            t0 = time.perf_counter()
            predict_one(X_test[i:i + 1])
            with lock:
                lat.append((time.perf_counter() - t0) * 1000)
    t0 = time.perf_counter()
    with cf.ThreadPoolExecutor(n_clients) as ex:
        list(ex.map(client, range(n_clients)))
    wall = time.perf_counter() - t0
    lat = np.array(lat)
    return {"throughput_rps": len(lat) / wall, "p50_ms": np.percentile(lat, 50),
            "p95_ms": np.percentile(lat, 95), "p99_ms": np.percentile(lat, 99)}

batching_rows = []
RPC = 20 if QUICK else 60
if SERVING_UP:
    for label, timeout_us in [("off", None), ("on, 1 ms", 1000), ("on, 5 ms", 5000)]:
        args = "--model_config_file=/models/models.config"
        if timeout_us:
            (SERVING_ROOT / "batching.config").write_text(
                f"max_batch_size {{ value: 128 }}\nbatch_timeout_micros {{ value: {timeout_us} }}\n"
                f"max_enqueued_batches {{ value: 1000 }}\nnum_batch_threads {{ value: 4 }}\n")
            args += " --enable_batching=true --batching_parameters_file=/models/batching.config"
        docker_run_serving(TFS_CONTAINER, [(SERVING_ROOT, "/models")], args=args)
        wait_ready(MODEL_NAME, want_version=1)
        res = load_test(lambda x: rest_predict(x, version=1), requests_per_client=RPC)
        batching_rows.append({"batching": label, **res})
else:
    fn1, _, _, _ = local_mnist._get(1)
    raw_call = lambda X: fn1(images=tf.constant(X))["probs"].numpy()
    for label, timeout_s in [("off", None), ("on, 1 ms", 0.001), ("on, 5 ms", 0.005)]:
        if timeout_s is None:
            res = load_test(raw_call, requests_per_client=RPC)
            avg_batch = 1.0
        else:
            mb = MicroBatcher(raw_call, timeout_s=timeout_s)
            res = load_test(mb, requests_per_client=RPC)
            avg_batch = float(np.mean(mb.batch_sizes)); mb.close()
        batching_rows.append({"batching": label, **res, "avg batch size": avg_batch})
batching_df = pd.DataFrame(batching_rows).round(2)
print("Backend:", "TF Serving" if SERVING_UP else "local MicroBatcher fallback")
print(batching_df.to_string(index=False))
RESULTS["batching"] = batching_df

fig, axes = plt.subplots(1, 2, figsize=(11, 3.5))
axes[0].bar(batching_df["batching"], batching_df["throughput_rps"], color="#4C72B0")
axes[0].set_title("Throughput (requests/s)"); axes[0].grid(axis="y", alpha=0.3)
x = np.arange(len(batching_df)); w = 0.27
for k, (col, c) in enumerate([("p50_ms", "#55A868"), ("p95_ms", "#DD8452"), ("p99_ms", "#C44E52")]):
    axes[1].bar(x + (k - 1) * w, batching_df[col], w, label=col, color=c)
axes[1].set_xticks(x); axes[1].set_xticklabels(batching_df["batching"])
axes[1].set_title("Latency percentiles (ms)"); axes[1].legend(); axes[1].grid(axis="y", alpha=0.3)
plt.tight_layout(); plt.show()

**Interpretation:** for this tiny model on a CPU the per-call overhead dominates, so batching mainly *reduces the number
of model invocations* (see the average batch size). The general rule holds: a longer window gives larger batches
and higher throughput but adds up to that window to every request's latency. On a GPU with a heavy model the
throughput gain is much larger. Choose the window from your **latency budget** (Task 15): if p99 must stay under
50 ms and the model takes 10 ms, a 2–5 ms window is safe; 30 ms is not.

## 3.6 Model warm-up
The first request after a load is slow (graph optimisation, memory allocation, kernel autotuning). If the SavedModel
contains `assets.extra/tf_serving_warmup_requests` — a TFRecord file of `PredictionLog` records — TF Serving replays
them **before** marking the version available, so real users never hit the cold path.

In [ ]:
def write_warmup(version_dir, X, model_name=MODEL_NAME, input_name=None, n=5):
    from tensorflow_serving.apis import prediction_log_pb2
    extra = pathlib.Path(version_dir) / "assets.extra"
    extra.mkdir(exist_ok=True)
    with tf.io.TFRecordWriter(str(extra / "tf_serving_warmup_requests")) as w:
        for _ in range(n):
            req = predict_pb2.PredictRequest()
            req.model_spec.name = model_name
            req.model_spec.signature_name = "serving_default"
            req.inputs[input_name or IN_NAME].CopyFrom(tf.make_tensor_proto(np.asarray(X, dtype="float32")))
            log = prediction_log_pb2.PredictionLog(predict_log=prediction_log_pb2.PredictLog(request=req))
            w.write(log.SerializeToString())
    return extra / "tf_serving_warmup_requests"

def first_request_latency():
    docker_run_serving(TFS_CONTAINER, [(MODEL_ROOT, f"/models/{MODEL_NAME}")], env={"MODEL_NAME": MODEL_NAME})
    wait_ready(MODEL_NAME)
    t0 = time.perf_counter(); rest_predict(X_new[:1]); first = (time.perf_counter() - t0) * 1000
    t0 = time.perf_counter(); rest_predict(X_new[:1]); second = (time.perf_counter() - t0) * 1000
    return first, second

if SERVING_UP and GRPC_OK:
    shutil.rmtree(MODEL_ROOT / "0002", ignore_errors=True)      # serve only v1 for a clean comparison
    shutil.rmtree(MODEL_ROOT / "0001" / "assets.extra", ignore_errors=True)
    no_warm = first_request_latency()
    write_warmup(MODEL_ROOT / "0001", X_new[:1])
    warm = first_request_latency()
    warm_df = pd.DataFrame([{"setup": "no warm-up", "1st request ms": no_warm[0], "2nd request ms": no_warm[1]},
                            {"setup": "with warm-up", "1st request ms": warm[0], "2nd request ms": warm[1]}])
    sh(f"docker logs {TFS_CONTAINER} 2>&1 | grep -i warmup | tail -3")
else:
    # Fallback: a freshly loaded SavedModel function vs the same function after one call
    rows = []
    for setup, do_warm in [("no warm-up", False), ("with warm-up", True)]:
        fn = tf.saved_model.load(str(MODEL_ROOT / "0001")).signatures["serving_default"]
        if do_warm:
            for _ in range(5):
                fn(images=tf.constant(X_new[:1]))
        t0 = time.perf_counter(); fn(images=tf.constant(X_new[:1])); first = (time.perf_counter() - t0) * 1000
        t0 = time.perf_counter(); fn(images=tf.constant(X_new[:1])); second = (time.perf_counter() - t0) * 1000
        rows.append({"setup": setup, "1st request ms": first, "2nd request ms": second})
    warm_df = pd.DataFrame(rows)
    if GRPC_OK:
        print("Warm-up file written:", write_warmup(MODEL_ROOT / "0001", X_new[:1]))
print(warm_df.round(2).to_string(index=False))
RESULTS["warmup"] = warm_df

## 3.7 Scaling out: many servers behind a load balancer

```
                         ┌──────────────────────┐
  clients ──HTTPS/gRPC──▶│  Load balancer /     │   health checks: GET /v1/models/<name>
                         │  API gateway (TLS,   │   (only route to replicas whose version is AVAILABLE)
                         │  auth, rate limits)  │
                         └───┬────────┬────────┬┘
                             ▼        ▼        ▼
                       ┌────────┐┌────────┐┌────────┐
                       │TFS pod1││TFS pod2││TFS pod3│   ← Kubernetes Deployment, HPA scales 2..N pods
                       └───┬────┘└───┬────┘└───┬────┘     on CPU/GPU utilisation or QPS
                           └─────────┼─────────┘
                                     ▼
                          model bucket gs://…/my_mnist_model/0001, 0002   (read-only, shared)
```
What you need to run this yourself:
* **container orchestration** (Kubernetes): Deployment + Service, a Horizontal Pod Autoscaler;
* **health/readiness probes** so traffic only reaches pods whose model is loaded;
* **rolling updates** (or the version-folder mechanism above) for zero-downtime model and server upgrades;
* **monitoring & alerting**: TF Serving's Prometheus endpoint (`--monitoring_config_file`), dashboards, logs;
* shared model storage, TLS certificates, secrets management, capacity planning, on-call.

A minimal (illustrative) Kubernetes manifest:
```yaml
apiVersion: apps/v1
kind: Deployment
metadata: {name: mnist-serving}
spec:
  replicas: 3
  selector: {matchLabels: {app: mnist-serving}}
  template:
    metadata: {labels: {app: mnist-serving}}
    spec:
      containers:
      - name: tfs
        image: tensorflow/serving
        args: ["--model_name=my_mnist_model", "--model_base_path=gs://YOUR_BUCKET/my_mnist_model"]
        ports: [{containerPort: 8500}, {containerPort: 8501}]
        readinessProbe: {httpGet: {path: /v1/models/my_mnist_model, port: 8501}}
        resources: {requests: {cpu: "1", memory: 1Gi}, limits: {cpu: "2", memory: 2Gi}}
---
apiVersion: v1
kind: Service
metadata: {name: mnist-serving}
spec:
  selector: {app: mnist-serving}
  ports: [{name: grpc, port: 8500}, {name: rest, port: 8501}]
```
**Why many teams pick a managed platform instead:** running VMs, the cluster, TF Serving configuration, autoscaling
tuning, TLS, monitoring and upgrades is easily a full-time job. A managed prediction service (Phase 4) does most of it
for a per-hour price.

## 3.8 Serving runbook

| Step | Action | Check / pass criterion |
|---|---|---|
| **1. Build** | Train; export with `export_model(..., version=N)` (self-test must pass); run offline evaluation vs current production | test metrics ≥ production − tolerance; self-test diff ≤ 1e-5 |
| **2. Stage** | Upload `N/` to the model bucket next to the current version (never overwrite a version) | folder complete (`saved_model.pb` + `variables/`), checksum recorded |
| **3. Verify** | Load on a staging server; send canned requests with known answers; `saved_model_cli show` | correct signature/shape; predictions match the self-test |
| **4. Canary** | Serve N and N−1 (specific-version policy); route 5–10 % of users to N | error rate, p95/p99 latency, prediction distribution vs N−1 for an agreed window |
| **5. Promote** | Raise to 50 % → 100 %; switch policy back to "latest" | guardrails stay green |
| **6. Roll back** | Remove N from the config / delete folder `N/`; server returns to N−1 within seconds | traffic back on N−1, error rate normal |
| **7. Monitor** | Dashboards & alerts | latency p50/p95/p99, error rate (4xx/5xx), throughput (QPS), model load events, input-feature drift (PSI/KS), prediction drift, delayed-label accuracy |
| **8. Clean up** | Keep last K versions in the bucket, delete older ones from the serving path | storage cost, audit log |

In [ ]:
# leave the server in a clean state for the next phases: v1 only, default "latest" policy
shutil.rmtree(MODEL_ROOT / "0002", ignore_errors=True)
if SERVING_UP:
    docker_run_serving(TFS_CONTAINER, [(MODEL_ROOT, f"/models/{MODEL_NAME}")], env={"MODEL_NAME": MODEL_NAME})
    wait_ready(MODEL_NAME)
print("Serving versions:", local_mnist.versions())

### Phase 3 — summary / deliverable
* ✅ v2 trained and published with the helper; server log excerpt and **client-side transition plot** (no failed requests)
* ✅ Rollback by deleting `0002`, verified from the client
* ✅ Version pinning (REST URL / gRPC `model_spec.version`) and a **90/10 canary** with per-version accuracy and latency
* ✅ Batching off vs 1 ms vs 5 ms: throughput and p50/p95/p99 charts, trade-off explained
* ✅ Warm-up file written to `assets.extra` and first-request latency compared
* ✅ Scale-out architecture, Kubernetes sketch, and the serving runbook

---
# Phase 4 — A Managed Prediction Service in the Cloud

**Decision for this notebook: design document + ready-to-run (but disabled) code.**
Hands-on cloud deployment needs a billing account, so the default path documents every step, estimates the cost and
gives a security checklist. If you do have a project, set `USE_CLOUD = True` and fill in `GCP_PROJECT` /
`GCP_BUCKET` below — the same cells then perform the real deployment and teardown.

> **Before touching any cloud service:** create a **budget alert**, set a calendar reminder to turn things off,
> never commit key files, and finish with the teardown cell at the end of this phase.

## 4.1 What a managed prediction service gives you
Vertex AI Prediction (successor of the "AI Platform Prediction" used in the 2019 material) runs a TF-Serving-based
container for you and adds: **HTTPS/TLS** and **token-based (IAM) authentication**, **autoscaling** of serving nodes
with traffic, **traffic splitting** between deployed model versions, request/response **logging**, and metrics in
**Cloud Monitoring** (formerly Stackdriver). You hand it a SavedModel folder; it hands you an endpoint.

## 4.2 Setup steps (current Vertex AI equivalents of the original console steps)

| # | Original (AI Platform, 2019) | Current (Vertex AI) — verify against the live docs before running |
|---|---|---|
| 1 | Create a Google account, accept terms, start free trial | Same; activate a **billing account** and set a budget alert |
| 2 | Create/select a **project** — the *project ID* is chosen once and permanent, the *project number* is generated | Same. All buckets, models, endpoints and service accounts belong to a project |
| 3 | Enable the AI Platform API | Enable **Vertex AI API** (`aiplatform.googleapis.com`) and Cloud Storage |
| 4 | Create a **Cloud Storage bucket** | Same. Bucket names share **one global namespace** and are visible to others, so use a unique prefix, don't put secrets in the name, and keep the bucket **private** (uniform access, no `allUsers`). Pick the same region as the endpoint |
| 5 | Upload the `my_mnist_model/0001` folder | Same (`gsutil cp -r` / `gcloud storage cp -r`). Size limits and quotas apply and can be raised |
| 6 | Create a **model**, then a **version** (framework TF, runtime version, machine type, path to `…/0002/`, automatic scaling, min nodes) | **Upload a Model** to the Model Registry with a *prebuilt TensorFlow prediction container* and `artifact_uri = gs://…/my_mnist_model/0001`; each upload of a new folder becomes a **model version**. Then **create an Endpoint** and **deploy** the model to it with machine type, min/max replica count and a traffic split |
| 7 | Query via the generic API client (`googleapiclient`) | Query with the **Vertex AI SDK** (`google-cloud-aiplatform`) or the REST endpoint `…/endpoints/ENDPOINT_ID:predict` |

`gcloud` equivalent (fill in names; check the current prebuilt container URI list in the Vertex docs):
```bash
gcloud services enable aiplatform.googleapis.com storage.googleapis.com
gcloud storage buckets create gs://$BUCKET --location=$REGION --uniform-bucket-level-access
gcloud storage cp -r task23_work/serving/my_mnist_model gs://$BUCKET/

gcloud ai models upload --region=$REGION --display-name=my_mnist_model \
    --artifact-uri=gs://$BUCKET/my_mnist_model/0001 \
    --container-image-uri=$PREBUILT_TF_CPU_IMAGE          # e.g. a "tf2-cpu.2-x" prediction image
gcloud ai endpoints create --region=$REGION --display-name=mnist-endpoint
gcloud ai endpoints deploy-model $ENDPOINT_ID --region=$REGION --model=$MODEL_ID \
    --display-name=mnist-v1 --machine-type=n1-standard-2 \
    --min-replica-count=1 --max-replica-count=3 --traffic-split=0=100
```

## 4.3 Cost model and the cold-start decision

With autoscaling, cost ≈ **node-hours × hourly machine price** + **storage** + network egress.
* If the service may **scale to zero** when idle, you pay mostly storage — but the first request after an idle period
  waits for a node to start (**cold start**, seconds to a minute).
* Keeping **min replicas = 1** removes the cold start but one machine bills **24 h/day**.

(The 2019 service allowed scaling to zero; on Vertex AI dedicated endpoints a minimum of one replica has been the
usual default — check whether scale-to-zero is offered for your endpoint type/region today.)

The cell below turns this into numbers with **assumed** prices (replace them with the current price list for your
region — prices change and vary by region).

In [ ]:
# ---- ASSUMPTIONS (replace with current regional prices) ----
PRICE_NODE_HOUR = 0.10      # USD per node-hour for a small CPU machine (assumption)
PRICE_GB_MONTH  = 0.02      # USD per GB-month of standard storage (assumption)
MODEL_GB        = sum(p.stat().st_size for p in MODEL_ROOT.rglob("*") if p.is_file()) / 1e9
HOURS_MONTH     = 730

scenarios = [
    # name, min replicas, avg extra replicas during busy hours, busy hours/month, cold starts acceptable?
    ("Hobby project (few requests/day)",      0, 1, 10,  "yes"),
    ("Startup, spiky daytime traffic",        1, 2, 200, "no for daytime"),
    ("Latency-critical product (p99 < 100ms)", 2, 4, 400, "never"),
]
rows = []
for name, min_rep, extra, busy_h, cold_ok in scenarios:
    node_hours = min_rep * HOURS_MONTH + extra * busy_h + (busy_h if min_rep == 0 else 0)
    cost = node_hours * PRICE_NODE_HOUR + MODEL_GB * PRICE_GB_MONTH
    rows.append({"scenario": name, "min replicas": min_rep, "node-hours/month": node_hours,
                 "est. USD/month": round(cost, 2), "cold starts OK?": cold_ok,
                 "choice": "scale to zero (accept cold start)" if min_rep == 0 else
                           f"min {min_rep} always-on + autoscale"})
cost_df = pd.DataFrame(rows)
print(f"Model size in storage: {MODEL_GB*1000:.2f} MB  (storage cost is negligible next to compute)")
print(cost_df.to_string(index=False))
RESULTS["cloud_cost"] = cost_df

**Decision table — cold start vs always-on**

| Situation | Choice | Why |
|---|---|---|
| Hobby / demo | scale to zero (if available) or delete the endpoint after use | almost no traffic; a few seconds of cold start is fine |
| Startup with spiky traffic | min 1 replica + autoscale up to N | users never see a cold start; extra nodes only during peaks |
| Latency-critical product | min ≥ 2 replicas (across zones) + headroom + warm-up | redundancy, no cold starts, room for traffic spikes |

## 4.4 Authentication: service accounts, not your own credentials
* **Your user credentials** can do *everything* in the project (delete buckets, read billing...). Embedding them in an
  application means anyone who gets hold of the app can do the same.
* A **service account** is an identity for the *application*. Give it **only** the role it needs — for calling an
  endpoint that is *Vertex AI User* (`roles/aiplatform.user`), nothing more (least privilege).
* If the client runs **outside** Google Cloud: create a JSON **private key** for the service account, store it
  **outside the repository** (secret manager, CI secret, or at least a path listed in `.gitignore`), and point
  `GOOGLE_APPLICATION_CREDENTIALS` at it. Rotate and delete keys when done; prefer **Workload Identity Federation**
  (no key file at all) where possible.
* If the client runs **on** Google Cloud (Cloud Run, GKE, Compute Engine): attach the service account to the
  workload — credentials come from the metadata server, **no key file** exists.

```bash
gcloud iam service-accounts create mnist-client --display-name="MNIST prediction client"
gcloud projects add-iam-policy-binding $PROJECT \
    --member="serviceAccount:mnist-client@$PROJECT.iam.gserviceaccount.com" --role="roles/aiplatform.user"
# only if the client runs outside GCP:
gcloud iam service-accounts keys create ~/secrets/mnist-client.json \
    --iam-account=mnist-client@$PROJECT.iam.gserviceaccount.com
echo "*.json" >> .gitignore          # never commit key files
```

## 4.5 Client libraries: generic vs service-specific
* **Generic API client** (`google-api-python-client`, built from discovery documents): one library for *every* Google
  API, OAuth + REST/JSON, you build resource paths yourself. This is what the 2019 code used against `ml v1`.
* **Service-specific client** (`google-cloud-aiplatform`): higher level, implements best practices (retries,
  auth, often gRPC under the hood) — **preferred whenever it exists**.

Both versions are below. They only run when `USE_CLOUD = True`.

In [ ]:
GCP_PROJECT  = "your-project-id"        # <- fill in to use
GCP_REGION   = "us-central1"
GCP_BUCKET   = "your-unique-prefix-task23"
ENDPOINT_ID  = "your-endpoint-id"
KEY_PATH     = os.path.expanduser("~/secrets/mnist-client.json")   # outside the repo

def predict_generic(X, project=GCP_PROJECT, model="my_mnist_model", version=None):
    """Legacy AI Platform style (generic discovery client) - kept to mirror the original material."""
    import googleapiclient.discovery
    name = f"projects/{project}/models/{model}"
    if version is not None:
        name += f"/versions/{version}"          # target one version: A/B test or canary group
    ml = googleapiclient.discovery.build("ml", "v1").projects()
    body = {"signature_name": "serving_default", "instances": np.asarray(X).tolist()}
    response = ml.predict(name=name, body=body).execute()
    if "error" in response:
        raise RuntimeError(response["error"])
    return np.array([pred[OUT_NAME] if isinstance(pred, dict) else pred for pred in response["predictions"]])

def predict_vertex(X, project=GCP_PROJECT, region=GCP_REGION, endpoint_id=ENDPOINT_ID):
    """Current Vertex AI SDK client."""
    from google.cloud import aiplatform
    aiplatform.init(project=project, location=region)
    endpoint = aiplatform.Endpoint(endpoint_id)
    response = endpoint.predict(instances=np.asarray(X).tolist())
    preds = [p[OUT_NAME] if isinstance(p, dict) else p for p in response.predictions]
    return np.array(preds), response.deployed_model_id    # which deployed version answered

if USE_CLOUD:
    os.environ["GOOGLE_APPLICATION_CREDENTIALS"] = KEY_PATH
    y_cloud, deployed_id = predict_vertex(X_new)
    print("Answered by deployed model", deployed_id)
    print("Matches local predictions:", np.allclose(y_cloud, model_v1.predict(X_new, verbose=0), atol=1e-4))
else:
    print("USE_CLOUD = False -> design-document mode; cloud client functions defined but not called.")

## 4.6 Targeting a specific version (canary)
* **Legacy API:** append `/versions/<id>` to the model path — `projects/P/models/my_mnist_model/versions/v2`.
* **Vertex AI:** deploy v2 to the **same endpoint** with a traffic split, e.g. `--traffic-split=OLD_ID=90,NEW_ID=10`;
  the response contains `deployed_model_id`, so the client (and logs) know which version answered. To send a
  *specific user group* to v2, deploy v2 to a second endpoint and route that group to it in your gateway.

## 4.7 Monitoring
Out of the box: request count, latency, errors, replica count, CPU/GPU utilisation in **Cloud Monitoring**, and
container/request logs in **Cloud Logging** (request-response logging can be sampled into BigQuery).
What I would add: alerting policies on **p99 latency**, **error rate > 1 %**, **replica count at max** (capacity),
a **budget alert** on spend, and model-quality monitoring (feature skew/drift jobs, Task 15).

## 4.8 Security checklist
- [ ] Budget alert created before any resource
- [ ] Bucket private, uniform access, no `allUsers`/`allAuthenticatedUsers`; nothing sensitive in its name
- [ ] Dedicated service account with only `roles/aiplatform.user` (client) / `roles/storage.objectViewer` (reader)
- [ ] No key files in the repo (`.gitignore`), keys stored in a secret manager, rotated, deleted after the project
- [ ] TLS only (the managed endpoint is HTTPS by default); auth tokens never logged
- [ ] Request-size limits and rate limiting at the gateway; quotas reviewed
- [ ] Audit logs enabled for the project

## 4.9 Teardown
Delete in reverse order of creation; the cell runs only when `USE_CLOUD = True`.

In [ ]:
teardown_cmds = [
    f"gcloud ai endpoints undeploy-model {ENDPOINT_ID} --region={GCP_REGION} --deployed-model-id=DEPLOYED_MODEL_ID",
    f"gcloud ai endpoints delete {ENDPOINT_ID} --region={GCP_REGION} --quiet",
    f"gcloud ai models delete MODEL_ID --region={GCP_REGION} --quiet",
    f"gcloud storage rm -r gs://{GCP_BUCKET}",
    f"gcloud iam service-accounts keys list --iam-account=mnist-client@{GCP_PROJECT}.iam.gserviceaccount.com",
    f"gcloud iam service-accounts delete mnist-client@{GCP_PROJECT}.iam.gserviceaccount.com --quiet",
]
if USE_CLOUD:
    for c in teardown_cmds:
        sh(c)
else:
    print("Teardown commands (run in order when USE_CLOUD = True):")
    print("\n".join("  " + c for c in teardown_cmds))
    print("\nFinal cost for this design-only run: USD 0.00 (no resources created)")

### Phase 4 — summary / deliverable
* ✅ Design document: current Vertex AI setup steps mapped from the original AI Platform steps, with `gcloud` commands
* ✅ Cost model with stated assumptions and a **cold-start vs always-on decision table**
* ✅ Service-account authentication explained, least-privilege role, key-handling rules
* ✅ Generic and Vertex SDK client code (`predict()` with error handling, version targeting for canaries)
* ✅ Monitoring plan, security checklist and teardown commands

---
# Phase 5 — Mobile, Embedded & Browser Deployment: TFLite, Quantization, TensorFlow.js

**Goal of this phase:** make the model small and cheap enough to run *on the device*, measure what each
optimisation really buys (size, accuracy, latency), and understand the arithmetic behind quantization.

## 5.1 The three objectives of TFLite (now branded LiteRT)
1. **Smaller model** → faster download, less storage and RAM on the phone.
2. **Less computation per prediction** → lower latency, less battery drain and heat.
3. **Fit the device** → use only ops the device runtime / accelerator (GPU delegate, NNAPI, Edge TPU) supports.

## 5.2 Convert to `.tflite` and what FlatBuffers are
The converter writes a **FlatBuffer**: a cross-platform binary format that can be **memory-mapped and used directly**
— no parsing step and no extra copy when the app loads the model (unlike protocol buffers, which must be decoded).

In [ ]:
TFLITE_DIR = WORK / "tflite"
TFLITE_DIR.mkdir(exist_ok=True)

def convert(saved_model_dir, name, configure=None):
    converter = tf.lite.TFLiteConverter.from_saved_model(str(saved_model_dir))
    if configure:
        configure(converter)
    tflite_bytes = converter.convert()
    path = TFLITE_DIR / f"{name}.tflite"
    path.write_bytes(tflite_bytes)
    return tflite_bytes, path

tfl_f32, tfl_f32_path = convert(MODEL_ROOT / "0001", "mnist_float32")
sm_size = sum(p.stat().st_size for p in (MODEL_ROOT / "0001").rglob("*") if p.is_file())
print(f"SavedModel folder: {human(sm_size)} | converted_model (float32 TFLite): {human(len(tfl_f32))}")

## 5.3 What the converter optimises
* **Pruning:** ops only needed for training (optimizer updates, gradients, dropout, metrics) are removed.
* **Constant folding / simplification:** expressions computable ahead of time are computed once,
  e.g. `3a + 4a + 5a` can become `12a` (one multiply instead of three multiplies and two adds).
* **Fusion:** several ops become one kernel, e.g. a Dense layer's MatMul + BiasAdd + ReLU becomes one
  `FULLY_CONNECTED` op with a fused activation, and **Batch Normalization** (a multiply and an add with fixed
  statistics at inference time) is **folded into the previous layer's weights and bias**.

We check this on two tiny graphs by listing the ops before and after conversion
(open both files in **Netron** — <https://netron.app> — to see the same thing visually).

In [ ]:
def tflite_ops(tflite_bytes):
    interp = tf.lite.Interpreter(model_content=tflite_bytes)
    try:
        return [op["op_name"] for op in interp._get_ops_details()]
    except Exception:
        return ["(op listing not available in this TF build)"]

def graph_ops(concrete_fn):
    skip = {"Placeholder", "NoOp", "Identity", "Const", "ReadVariableOp", "VarHandleOp"}
    return [op.type for op in concrete_fn.graph.get_operations() if op.type not in skip]

# (a) constant folding: 3a + 4a + 5a
@tf.function(input_signature=[tf.TensorSpec([None, 4], tf.float32)])
def three_terms(a):
    return 3.0 * a + 4.0 * a + 5.0 * a
cf_fn = three_terms.get_concrete_function()
cf_tfl = tf.lite.TFLiteConverter.from_concrete_functions([cf_fn], three_terms).convert()
print("3a+4a+5a  TF graph ops :", graph_ops(cf_fn))
print("3a+4a+5a  TFLite ops   :", tflite_ops(cf_tfl))

# (a') pure constant sub-expressions ARE folded: (2*3 + sqrt(16)) is computed once at conversion time
@tf.function(input_signature=[tf.TensorSpec([None, 4], tf.float32)])
def const_expr(a):
    return a * (tf.constant(2.0) * tf.constant(3.0) + tf.sqrt(tf.constant(16.0)))
ce_fn = const_expr.get_concrete_function()
ce_tfl = tf.lite.TFLiteConverter.from_concrete_functions([ce_fn], const_expr).convert()
print("a*(2*3+sqrt(16)) TF graph ops:", graph_ops(ce_fn))
print("a*(2*3+sqrt(16)) TFLite ops  :", tflite_ops(ce_tfl))

# (b) BatchNorm folding + activation fusion
tf.random.set_seed(SEED)
bn_model = keras.Sequential([keras.Input((16,)), keras.layers.Dense(8), keras.layers.BatchNormalization(),
                             keras.layers.ReLU(), keras.layers.Dense(3, activation="softmax")])
bn_model.compile(loss="sparse_categorical_crossentropy", optimizer="adam")
bn_model.fit(np.random.rand(256, 16).astype("float32"), np.random.randint(0, 3, 256), epochs=1, verbose=0)
bn_path = WORK / "bn_model" / "0001"
if bn_path.exists():
    shutil.rmtree(bn_path)
write_savedmodel(bn_model, bn_path, tf.TensorSpec([None, 16], tf.float32, name="images"))
bn_fn = tf.function(lambda x: bn_model(x, training=False)).get_concrete_function(
    tf.TensorSpec([None, 16], tf.float32))                      # inference graph with every op inlined
bn_tfl, _ = convert(bn_path, "bn_folding_demo")
print("\nDense+BN+ReLU TF graph ops:", sorted(set(graph_ops(bn_fn))))
print("Dense+BN+ReLU TFLite ops  :", tflite_ops(bn_tfl))
try:
    tf.lite.experimental.Analyzer.analyze(model_content=tfl_f32)
except Exception as e:
    print("Analyzer not available:", e)

**Interpretation:** the TF graph contains separate MatMul / BiasAdd / BatchNorm (or Mul/Add/Rsqrt) / Relu ops, while
the TFLite model only has `FULLY_CONNECTED` (with the ReLU fused in) and `SOFTMAX` — **no BatchNorm op survives**, its
scale and shift were folded into the weights. **Constant sub-expressions** (`2*3 + sqrt(16)`) collapse into a single
`MUL` by the pre-computed constant 10. The algebraic rewrite `3a + 4a + 5a → 12a` depends on the TF version's graph
optimiser — in the run above check whether the TFLite op list is shorter; if it still shows 3 MUL + 2 ADD, the converter
did not apply that particular rewrite (it is a possibility the source mentions, not a guarantee), so writing `12 * a`
yourself is still worth it.

## 5.4 Run the `.tflite` model with the interpreter
One helper handles every variant, including int8 inputs/outputs (it applies the stored scale/zero-point).

In [ ]:
def tflite_predict(tflite_bytes, X, batch=500):
    interp = tf.lite.Interpreter(model_content=tflite_bytes)
    inp = interp.get_input_details()[0]
    preds, current_shape = [], None
    for s in range(0, len(X), batch):
        xb = np.asarray(X[s:s + batch])
        if current_shape != xb.shape:
            interp.resize_tensor_input(inp["index"], xb.shape)
            interp.allocate_tensors()
            current_shape = xb.shape
        inp = interp.get_input_details()[0]
        out = interp.get_output_details()[0]
        if inp["dtype"] in (np.int8, np.uint8):
            scale, zp = inp["quantization"]
            info = np.iinfo(inp["dtype"])
            xb = np.clip(np.round(xb / scale + zp), info.min, info.max).astype(inp["dtype"])
        else:
            xb = xb.astype(inp["dtype"])
        interp.set_tensor(inp["index"], xb)
        interp.invoke()
        y = interp.get_tensor(out["index"])
        if out["dtype"] in (np.int8, np.uint8):
            scale, zp = out["quantization"]
            y = (y.astype(np.float32) - zp) * scale
        preds.append(y.astype(np.float32))
    return np.concatenate(preds)

def tflite_latency(tflite_bytes, x_one, repeats=7, inner=50):
    interp = tf.lite.Interpreter(model_content=tflite_bytes)
    inp = interp.get_input_details()[0]
    interp.resize_tensor_input(inp["index"], x_one.shape); interp.allocate_tensors()
    inp = interp.get_input_details()[0]
    if inp["dtype"] in (np.int8, np.uint8):
        scale, zp = inp["quantization"]
        x_one = np.clip(np.round(x_one / scale + zp), np.iinfo(inp["dtype"]).min,
                        np.iinfo(inp["dtype"]).max).astype(inp["dtype"])
    interp.set_tensor(inp["index"], x_one.astype(inp["dtype"]))
    return bench(interp.invoke, warmup=10, repeats=repeats, inner=inner)

y_tfl = tflite_predict(tfl_f32, X_new)
print("max |TFLite float32 - SavedModel| =", np.abs(y_tfl - model_v1.predict(X_new, verbose=0)).max())

## 5.5 Symmetric 8-bit quantization — by hand, then in NumPy

**Scheme:** find `m = max|w|`; map the float range `[−m, +m]` linearly onto the integers `[−127, +127]`
(−128 is left unused so the range is symmetric). `scale = m / 127`, `q = round(w / scale)`, `ŵ = q × scale`.
Zero always maps to exactly 0.

**Worked example** — weights between −1.5 and +0.8:
* `m = 1.5`, `scale = 1.5 / 127 ≈ 0.011811`
* byte −127 ↔ −1.5, byte 0 ↔ 0.0, byte +127 ↔ +1.5
* the largest weight 0.8 → `round(0.8 / 1.5 × 127) = round(67.73) = 68`
* so the bytes **+69 … +127 are never used** — the price of keeping the mapping symmetric.

In [ ]:
def quantize_symmetric(w, bits=8):
    qmax = 2 ** (bits - 1) - 1                  # 127
    m = float(np.max(np.abs(w)))
    scale = m / qmax if m > 0 else 1.0
    q = np.clip(np.round(w / scale), -qmax, qmax).astype(np.int8)
    return q, scale

def dequantize(q, scale):
    return q.astype(np.float32) * scale

w_example = np.array([-1.5, 0.0, 0.8], dtype=np.float32)
q, scale = quantize_symmetric(w_example)
print("weights       :", w_example)
print("scale (m/127) :", round(scale, 6))
print("quantized     :", q, "  <- 0.8 maps to", q[-1])
print("dequantized   :", dequantize(q, scale).round(5))
print("unused positive codes: +%d ... +127" % (q.max() + 1))
assert q.tolist() == [-127, 0, 68]

# round-trip error on a real weight matrix (first Dense layer of the v1 model)
W = [l for l in model_v1.layers if isinstance(l, keras.layers.Dense)][0].get_weights()[0]
Wq, s = quantize_symmetric(W)
err = np.abs(dequantize(Wq, s) - W)
print(f"\nReal kernel {W.shape}: scale={s:.5f}, max abs error={err.max():.5f} (≤ scale/2 = {s/2:.5f}), "
      f"mean abs error={err.mean():.6f}, size {human(W.nbytes)} -> {human(Wq.nbytes)}")

## 5.6 Half precision, weight-only quantization, full-integer quantization, QAT

| Variant | What is stored | Run-time behaviour |
|---|---|---|
| **float16** | weights in 16-bit floats (½ size) | CPU dequantizes to float32; GPU delegates can compute in fp16. Usually tiny accuracy loss. (Training in 16-bit is also faster and halves GPU memory — mixed precision.) |
| **Weight quantization** (`Optimize.DEFAULT`, the old `OPTIMIZE_FOR_SIZE`) | weights as int8 (≈ ¼ size) | the classic scheme converts weights **back to float32 at load time and caches them**, so there is **no RAM saving and no compute speed-up** — only a smaller download. (Newer runtimes may use "dynamic-range" int8 kernels for some ops, so measure.) |
| **Full-integer** | weights **and activations** int8 | integer-only arithmetic: fewer CPU cycles, less energy/heat, and runs on integer-only accelerators (Edge TPU, many MCUs). Needs **calibration**: a *representative dataset* (a few hundred real inputs) is run through the model to record each activation's min/max. |
| **QAT** | same as full-integer | the model was *trained* with fake-quantization noise, so it tolerates the int8 rounding better. |

In [ ]:
def cfg_float16(c):
    c.optimizations = [tf.lite.Optimize.DEFAULT]
    c.target_spec.supported_types = [tf.float16]

def cfg_weights(c):
    c.optimizations = [tf.lite.Optimize.DEFAULT]

def representative_dataset():
    idx = np.random.default_rng(SEED).choice(len(X_train), 300, replace=False)
    for i in idx:
        yield [X_train[i:i + 1]]

def cfg_full_int(c):
    c.optimizations = [tf.lite.Optimize.DEFAULT]
    c.representative_dataset = representative_dataset
    c.target_spec.supported_ops = [tf.lite.OpsSet.TFLITE_BUILTINS_INT8]
    c.inference_input_type = tf.int8
    c.inference_output_type = tf.int8

variants = {"TFLite float32": tfl_f32}
for name, cfg in [("TFLite float16", cfg_float16), ("TFLite weight-int8", cfg_weights),
                  ("TFLite full-int8", cfg_full_int)]:
    variants[name], _ = convert(MODEL_ROOT / "0001", name.replace(" ", "_").replace("-", "_"), cfg)
for k, v in variants.items():
    print(f"{k:20s} {human(len(v)):>10s}")

### Quantization-aware training (QAT)
The official QAT tool (`tensorflow-model-optimization`) targets `tf_keras` (Keras 2), so on Keras 3 we implement the
idea directly: a `QuantDense` layer that **fake-quantizes its weights and outputs to 8 bits in the forward pass**
(`tf.quantization.fake_quant_with_min_max_vars`) while gradients pass straight through (straight-through estimator).
The network learns weights that still work after rounding. We then copy the learned weights into a plain model and
apply full-integer conversion.

In [ ]:
class QuantDense(keras.layers.Layer):
    """Dense layer that simulates int8 weights and activations during training."""
    def __init__(self, units, activation=None, **kwargs):
        super().__init__(**kwargs)
        self.units, self.activation = units, keras.activations.get(activation)
    def build(self, input_shape):
        self.kernel = self.add_weight(name="kernel", shape=(int(input_shape[-1]), self.units),
                                      initializer="glorot_uniform")
        self.bias = self.add_weight(name="bias", shape=(self.units,), initializer="zeros")
    def call(self, x):
        m = tf.stop_gradient(tf.reduce_max(tf.abs(self.kernel))) + 1e-8
        kernel_q = tf.quantization.fake_quant_with_min_max_vars(self.kernel, -m, m, num_bits=8, narrow_range=True)
        y = self.activation(tf.matmul(x, kernel_q) + self.bias)
        a = tf.stop_gradient(tf.reduce_max(tf.abs(y))) + 1e-6
        return tf.quantization.fake_quant_with_min_max_vars(y, -a, a, num_bits=8)

qat_tflite, qat_acc_fake = None, None
try:
    tf.random.set_seed(SEED)
    qat_model = keras.Sequential([keras.Input((28, 28)), keras.layers.Flatten(),
                                  QuantDense(100, "relu"), QuantDense(10, "softmax")])
    qat_model.compile(loss="sparse_categorical_crossentropy", optimizer="nadam", metrics=["accuracy"])
    qat_model.fit(X_train, y_train, epochs=EPOCHS, validation_data=(X_valid, y_valid), verbose=2)
    qat_acc_fake = qat_model.evaluate(X_test, y_test, verbose=0)[1]

    plain = build_mnist_model()
    plain.layers[-2].set_weights(qat_model.layers[-2].get_weights())   # copy QuantDense -> Dense
    plain.layers[-1].set_weights(qat_model.layers[-1].get_weights())
    qat_dir = WORK / "mnist_qat" / "0001"
    if qat_dir.exists():
        shutil.rmtree(qat_dir)
    write_savedmodel(plain, qat_dir, MNIST_SPEC)
    qat_tflite, _ = convert(qat_dir, "mnist_qat_full_int8", cfg_full_int)
    variants["QAT -> full-int8"] = qat_tflite
    print(f"QAT model accuracy with fake-quant ops: {qat_acc_fake:.4f}")
except Exception as e:
    print("QAT step skipped:", type(e).__name__, str(e)[:300])

## 5.7 The comparison table (full 10,000-image test set; latency = 1 image, warm-up excluded, mean ± std)

In [ ]:
sm_fn = tf.saved_model.load(str(MODEL_ROOT / "0001")).signatures["serving_default"]
x1 = X_test[:1]
rows = []
probs_sm = np.concatenate([sm_fn(images=tf.constant(X_test[i:i + 2000]))["probs"].numpy()
                           for i in range(0, len(X_test), 2000)])
m, s = bench(lambda: sm_fn(images=tf.constant(x1)), warmup=10, repeats=7, inner=50)
rows.append({"variant": "SavedModel float32", "size": human(sm_size), "bytes": sm_size,
             "accuracy": (probs_sm.argmax(1) == y_test).mean(), "latency ms": f"{m:.3f} ± {s:.3f}",
             "notes": "server format; includes graph + variables"})
notes = {"TFLite float32": "same math, FlatBuffer, training ops pruned",
         "TFLite float16": "≈½ size; fp16 weights, computed in fp32 on CPU",
         "TFLite weight-int8": "≈¼ size; weights dequantized at load -> no CPU speed-up guaranteed",
         "TFLite full-int8": "int8 weights + activations, calibrated on 300 samples",
         "QAT -> full-int8": "trained with fake-quant noise, then full-int8"}
for name, b in variants.items():
    p = tflite_predict(b, X_test)
    m, s = tflite_latency(b, x1)
    rows.append({"variant": name, "size": human(len(b)), "bytes": len(b),
                 "accuracy": (p.argmax(1) == y_test).mean(), "latency ms": f"{m:.3f} ± {s:.3f}",
                 "notes": notes.get(name, "")})
tflite_df = pd.DataFrame(rows)
tflite_df["size vs float32 TFLite"] = (tflite_df["bytes"] / len(tfl_f32)).round(2)
tflite_df["accuracy"] = tflite_df["accuracy"].round(4)
print(tflite_df.drop(columns="bytes").to_string(index=False))
RESULTS["tflite_mnist"] = tflite_df

**Interpretation**
* float16 halves the file and weight-int8 cuts it ~4×, with essentially unchanged accuracy for this model.
* Weight-only int8 is **not** guaranteed to be faster: the classic scheme restores float32 weights at load time.
  It saves **download/storage**, not RAM or compute.
* Full-int8 keeps the size win *and* runs integer kernels — this is the variant for MCUs / Edge TPU; its accuracy
  drop is small here because a tiny dense MNIST model is very robust. QAT matters for larger, more sensitive models
  (e.g. MobileNets, models with depthwise convs or wide activation ranges), where post-training int8 can lose several
  points of accuracy.
* On a desktop CPU the absolute latencies are tiny; differences show up much more on phones and microcontrollers.

## 5.8 TensorFlow.js — running in the browser
Three reasons to run a model **in the browser**:
1. **intermittent / slow connectivity** (works offline once loaded — e.g. hiking app);
2. **very low latency** needs (no network round trip — e.g. real-time webcam effects, games);
3. **privacy** — the user's data (photos, typing) never leaves the device. Related idea: **federated learning**,
   where devices also *train* locally and only send model updates.

The converter turns a SavedModel into `model.json` (graph topology + weight manifest) plus **sharded binary weight
files** (`group1-shard1of1.bin`, … in ~4 MB chunks so browsers can cache them).

In [ ]:
TFJS_DIR = WORK / "tfjs_model"
conv = shutil.which("tensorflowjs_converter")
if conv:
    shutil.rmtree(TFJS_DIR, ignore_errors=True)
    sh(f'{conv} --input_format=tf_saved_model --signature_name=serving_default '
       f'--saved_model_tags=serve "{MODEL_ROOT / "0001"}" "{TFJS_DIR}"')
    if TFJS_DIR.exists():
        print("\n".join(dir_tree(TFJS_DIR)))
else:
    print("tensorflowjs_converter not installed. In a separate environment run:\n"
          "  pip install tensorflowjs\n"
          f"  tensorflowjs_converter --input_format=tf_saved_model --signature_name=serving_default "
          f"--saved_model_tags=serve {MODEL_ROOT/'0001'} {TFJS_DIR}\n"
          "Expected output:  tfjs_model/model.json  +  tfjs_model/group1-shard1of1.bin")

Loading and predicting in the browser:
```html
<script src="https://cdn.jsdelivr.net/npm/@tensorflow/tfjs"></script>
<script>
  async function run() {
    const model = await tf.loadGraphModel('tfjs_model/model.json');   // fetches json + shards (cached)
    const canvas = document.getElementById('digit');                    // a 28x28 drawing canvas
    const pixels = tf.browser.fromPixels(canvas, 1)                     // (28,28,1) int32 0..255
                     .squeeze().toFloat().div(255).expandDims(0);       // (1,28,28) float32 0..1
    const probs = model.predict(pixels);                                // {probs} or a tensor
    const out = probs.probs ? probs.probs : probs;
    console.log('digit =', (await out.argMax(-1).data())[0]);
    tf.dispose([pixels, out]);                                          // free GPU (WebGL) memory
  }
  run();
</script>
```

### Deployment-target comparison
| | Server (TF Serving / Vertex) | Mobile / embedded (TFLite) | Browser (TF.js) |
|---|---|---|---|
| **Latency** | network round trip + inference | on-device, no network | on-device, no network (WebGL/WASM/WebGPU backends) |
| **Privacy** | data leaves the device | data stays on device | data stays on device |
| **Offline** | no | yes | yes, after first load (cache) |
| **Model updates** | instant, server-side (versions) | app update or remote model download | redeploy files; browser cache control |
| **Size limits** | practically none (GPU RAM) | tight: app size, RAM, MCU flash (KB–MB) | tight: download time, browser memory |
| **Compute** | big GPUs/TPUs | phone CPU/GPU/NPU, MCU | user's device, varies a lot |
| **Typical use** | heavy models, shared by many clients | camera apps, keyboards, sensors | interactive web demos, privacy-sensitive web apps |

## 5.9 Going further & a question before shipping
* **TinyML** (Warden & Situnayake) — models for microcontrollers with kilobytes of RAM.
* **Practical Deep Learning for Cloud, Mobile, and Edge** (Koul, Ganju & Kasam) — end-to-end device deployment.

**Question I'd want answered before shipping to a real device:** *How do we update the model in the field — do we ship
it inside the app (update = app release) or download it at runtime, and what happens on devices that can't fit or
can't run the new version (fallback to the old model or to the server API)?* Close behind: which delegate
(GPU/NNAPI/Core ML) will actually run on our lowest-end supported phone, and what is the latency there?

### Phase 5 — summary / deliverable
* ✅ TFLite float32 / float16 / weight-int8 / full-int8 / QAT conversions with the **size–accuracy–latency table**
* ✅ Converter optimizations demonstrated (constant folding, BN folding, activation fusion) by listing ops
* ✅ NumPy symmetric quantizer with the −1.5…+0.8 example verified (0.8 → 68, codes 69–127 unused)
* ✅ Explanation of why weight-only quantization doesn't speed up inference
* ✅ TensorFlow.js conversion command/output, browser loading code, deployment-target table

---
# Phase 6 — Using GPUs: Memory, Placement & Parallel Execution

**Goal of this phase:** make sure TensorFlow really uses the GPU, control how much GPU memory each process takes,
decide where each op runs, and understand how TensorFlow executes a graph in parallel.

## 6.1 Why GPUs — and the "one machine first" rule
Neural-network training is mostly large matrix multiplications and convolutions — thousands of identical, independent
multiply-adds that a GPU runs in parallel. Days of CPU training become hours or minutes, which means **more
experiments per day** and the ability to **retrain often** on fresh data.

Before reaching for a cluster: adding GPUs to **one machine** is usually the better first step. GPUs inside one box
talk over PCIe/NVLink, which is far faster than a network, so 4 GPUs in one server can match 8 GPUs spread over
several servers. And **one powerful GPU often beats several weak ones** (less communication, simpler code).

## 6.2 Where to get a GPU
| Option | Notes |
|---|---|
| **Your own card** | Needs an NVIDIA card whose *compute capability* is supported by your TF version (check the TF install docs). Install the **NVIDIA driver** (talks to the hardware), **CUDA** (general-purpose GPU computing toolkit) and **cuDNN** (GPU-optimised deep-learning primitives: convolutions, RNNs, pooling, ...). On Linux, `pip install tensorflow[and-cuda]` bundles CUDA/cuDNN; alternatively run TF's GPU **Docker** image so only the host driver is needed. |
| **Cloud GPU VM** | Pay per hour; GPU quotas often **start at zero** (request an increase — can take hours/days); not every GPU type exists in every region; preconfigured *Deep Learning VM* images have drivers + TF installed. |
| **Free notebook runtime** (Colab, Kaggle) | Select a GPU accelerator. Limits: limited concurrency, idle disconnects, maximum session length (~12 h on Colab free), GPU type not guaranteed — **not for long jobs**, and save everything you care about to Drive/GitHub before the runtime is recycled. |

## 6.3 Verify the setup

In [ ]:
if shutil.which("nvidia-smi"):
    sh("nvidia-smi")
else:
    print("nvidia-smi not found -> no NVIDIA driver visible (CPU-only machine). The rest of the phase still runs;")
    print("GPU-only demonstrations are skipped and the logical-CPU devices stand in for GPUs.")
print("\nPhysical devices :", tf.config.list_physical_devices())
print("Logical devices  :", [d.name for d in tf.config.list_logical_devices()])
print("Built with CUDA  :", tf.test.is_built_with_cuda())
if HAS_GPU:
    print("GPU details      :", tf.config.experimental.get_device_details(physical_gpus[0]))

**How to read `nvidia-smi`:** the top table shows each GPU's *name*, *memory used / total*, *GPU-Util %* and
temperature/power; the bottom table lists the **processes** holding GPU memory (PID and MiB each).
In this notebook the first GPU was split into two virtual GPUs of `VGPU_MEM_MB` each, so this process holds roughly
`2 × VGPU_MEM_MB` once it has run something — not the whole card.

## 6.4 Default memory behaviour and the four fixes
By default TensorFlow **grabs (almost) all memory of every visible GPU** the first time it runs a computation, to
reduce fragmentation. Start a second TF process on the same GPU and it immediately runs **out of memory**.

Each fix must be applied **right after importing TensorFlow, before any computation**, so each one is demonstrated
in its own fresh process (this notebook's runtime is already initialised):

| Fix | How | When to use |
|---|---|---|
| 1. **Assign GPUs to processes** | `CUDA_DEVICE_ORDER=PCI_BUS_ID CUDA_VISIBLE_DEVICES=0,1 python3 program_1.py` and `CUDA_VISIBLE_DEVICES=3,2 python3 program_2.py` | Several jobs on a multi-GPU box (e.g. hyperparameter search). Program 2 sees *its* cards renamed `/gpu:0`, `/gpu:1` — TF numbers only the GPUs it can see, in the order listed. `PCI_BUS_ID` makes the numbering match `nvidia-smi` (default order is "fastest first"). |
| 2. **Fixed memory limit** | `tf.config.set_logical_device_configuration(gpu, [LogicalDeviceConfiguration(memory_limit=2048)])` | Several programs share every card, each with a guaranteed slice. Predictable — preferred in production. |
| 3. **Memory growth** | `tf.config.experimental.set_memory_growth(gpu, True)` or env `TF_FORCE_GPU_ALLOW_GROWTH=true` | Grab memory only as needed. But memory is **never released** until the process ends, and the footprint depends on the order of allocations → less deterministic. Colab enables it so notebooks can coexist. |
| 4. **Split one GPU into virtual GPUs** | several `LogicalDeviceConfiguration`s on one physical GPU | Test multi-GPU / distribution code on a single GPU (what this notebook does). |

In [ ]:
probe_script = WORK / "gpu_probe.py"
probe_script.write_text(r"""
import os, sys, time
os.environ["TF_CPP_MIN_LOG_LEVEL"] = "3"
mode = sys.argv[1]
import tensorflow as tf
gpus = tf.config.list_physical_devices("GPU")
if mode == "limit":
    for g in gpus:
        tf.config.set_logical_device_configuration(g, [tf.config.LogicalDeviceConfiguration(memory_limit=2048)])
elif mode == "growth":
    for g in gpus:
        tf.config.experimental.set_memory_growth(g, True)
elif mode == "split":
    tf.config.set_logical_device_configuration(gpus[0], [tf.config.LogicalDeviceConfiguration(memory_limit=2048)] * 2)
x = tf.random.normal([2048, 2048])          # first computation -> allocator initialised
y = tf.matmul(x, x)
print(f"[{mode}] pid={os.getpid()} visible={[g.name for g in gpus]} "
      f"logical={[d.name for d in tf.config.list_logical_devices('GPU')]}", flush=True)
try:
    info = tf.config.experimental.get_memory_info("GPU:0")
    print(f"[{mode}] TF allocator: current={info['current']/2**20:.0f} MiB peak={info['peak']/2**20:.0f} MiB", flush=True)
except Exception as e:
    print(f"[{mode}] memory info unavailable: {e}", flush=True)
time.sleep(float(sys.argv[2]) if len(sys.argv) > 2 else 8)
""")

def run_two(mode, env_extra=None):
    """Start two copies of the probe, snapshot nvidia-smi while both are alive."""
    env = dict(os.environ, **(env_extra or {}))
    procs = [subprocess.Popen([sys.executable, str(probe_script), mode, "10"], env=env,
                              stdout=subprocess.PIPE, stderr=subprocess.STDOUT, text=True) for _ in range(2)]
    time.sleep(7)
    smi = subprocess.run("nvidia-smi --query-compute-apps=pid,used_memory --format=csv,noheader",
                         shell=True, capture_output=True, text=True).stdout
    outs = [p.communicate(timeout=120)[0] for p in procs]
    print(f"===== mode: {mode} =====")
    for o in outs:
        lines = [l for l in o.splitlines() if l.startswith("[") or "Error" in l or "OOM" in l]
        print("\n".join(lines[-3:]) or o[-300:])
    print("nvidia-smi per-process memory:\n" + (smi.strip() or "(none)"))

if HAS_GPU:
    run_two("default")                                        # second process may fail with OOM
    run_two("limit")                                          # ~2 GiB each, both run
    run_two("growth")                                         # only what's needed
    run_two("default", {"TF_FORCE_GPU_ALLOW_GROWTH": "true"}) # growth via env variable
    run_two("split")                                          # 2 virtual GPUs per process
    if len(physical_gpus) >= 2:
        sh(f"CUDA_DEVICE_ORDER=PCI_BUS_ID CUDA_VISIBLE_DEVICES=1 {sys.executable} {probe_script} default 1")
else:
    print("No GPU: memory-management demos skipped. On a GPU runtime this cell launches two processes per option")
    print("and prints nvidia-smi's per-process memory (default: one process grabs everything; limit: ~2 GiB each;")
    print("growth: only what the matmul needs).")
    # CUDA_VISIBLE_DEVICES still demonstrates the renaming rule on CPU-only machines (no GPU visible at all):
    sh(f'CUDA_VISIBLE_DEVICES="" {sys.executable} -c "import os; os.environ[\'TF_CPP_MIN_LOG_LEVEL\']=\'3\'; '
       f'import tensorflow as tf; print(\'GPUs visible with CUDA_VISIBLE_DEVICES=empty:\', '
       f'tf.config.list_physical_devices(\'GPU\'))"')

## 6.5 Device placement
Every variable and op is placed on a device. A **kernel** is the implementation of one op for one dtype on one device
type; if no GPU kernel exists for an op/dtype, it must run on the CPU. By convention TF keeps **int32 variables
and most integer ops on the CPU** (integers are usually shapes/indices/counters, needed by the host anyway, and
GPU kernels for them are limited). The CPU is always **one device**, `/cpu:0`, no matter how many cores — multithreaded
kernels use the cores internally. (In this notebook the CPU was split into logical CPUs only if there is no GPU.)

In [ ]:
a = tf.Variable(42.0)                 # float32
b = tf.Variable(42)                   # int32
with tf.device("/cpu:0"):
    c = tf.Variable(42.0)             # forced onto the CPU
placement_df = pd.DataFrame([
    {"variable": "tf.Variable(42.0)  float32", "device": a.device},
    {"variable": "tf.Variable(42)    int32",   "device": b.device},
    {"variable": "with tf.device('/cpu:0')",   "device": c.device},
])
print(placement_df.to_string(index=False))

**Result:** on a GPU machine the float variable lands on `GPU:0` while the int32 variable stays on `CPU:0`
(no GPU placement for int32 resource variables by default); `tf.device` overrides the choice.
On a CPU-only machine all three are on the CPU.

## 6.6 Impossible placements, soft placement, placement logging

In [ ]:
print("Soft placement enabled by default:", tf.config.get_soft_device_placement())

tf.config.set_soft_device_placement(False)
try:
    with tf.device("/gpu:9"):          # a device that does not exist
        z = tf.Variable(1.0)
    print("No error?! placed on", z.device)
except Exception as e:
    print("Strict placement ->", type(e).__name__, ":", str(e).splitlines()[0][:150])

tf.config.set_soft_device_placement(True)
with tf.device("/gpu:9"):
    z = tf.Variable(1.0)
print("Soft placement   -> fell back to", z.device)

tf.debugging.set_log_device_placement(True)
r = tf.reduce_sum(tf.random.normal([4, 4]) @ tf.random.normal([4, 4]))   # placements printed to the log/stderr
tf.debugging.set_log_device_placement(False)
print("Result computed on:", r.device)

## 6.7 Placement rules of thumb
* **Preprocessing (tf.data, decoding, augmentation) on the CPU; neural-network math on the GPU(s).**
* **Minimise transfers** in and out of the GPU — PCIe bandwidth is limited; keep intermediate results on the GPU.
* **GPU RAM is scarce, fixed and expensive; CPU RAM is cheap** — anything not needed in the next few steps
  (the dataset, logs, old checkpoints) lives on the CPU side.
* TensorFlow once shipped a *dynamic placer* that tried to optimise placement automatically; simple human rules beat it,
  so it was dropped. Today `tf.keras` and `tf.data` place things sensibly by default — you intervene only for special
  cases (pinning a lookup table to the CPU, model parallelism, multiple models on multiple GPUs).

## 6.8 How TensorFlow executes a graph in parallel

**Algorithm**
1. Analyse the graph and give every op a **dependency counter** = number of inputs not yet computed.
2. Ops with counter **0** go into the **evaluation queue of their device**.
3. When an op finishes, decrement the counter of each op that consumes its output; any that reaches 0 is queued.
4. **CPU queue:** dispatched to the **inter-op thread pool** — several ops run at the same time. Ops with
   multithreaded kernels split their own work across the **intra-op thread pool**.
5. **GPU queue:** processed **one op at a time**, because each GPU op already uses most of the GPU's threads
   (via CUDA/cuDNN). Different GPUs have different queues and run in parallel.
6. **Stateful ops** (variable assignments, prints) keep the **program order** (automatic control dependencies in
   `tf.function`), so results are deterministic where it matters.

### My own six-op graph (paper walkthrough)
```
   A = x @ W1   (GPU)         B = cast(ids) (CPU)        C = x * 2 (CPU)
        │                         │      \                   │
        ▼                         ▼       \                  ▼
   D = relu(A) (GPU)          E = B + 1 (CPU)  \──────▶ F = E + C (CPU)
```
Dependencies: A,B,C → 0 ; D → 1 (A) ; E → 1 (B) ; F → 2 (E, C).

| t | GPU queue | CPU queue (inter-op pool) | Event |
|---|---|---|---|
| 0 | A | B, C (run in parallel) | three ops have 0 dependencies |
| 1 | D (counter→0 after A) | E (counter→0 after B) | C done: F counter 2→1 |
| 2 | — | F (counter→0 after E) | D done (GPU branch finished) |
| 3 | — | — | all done |

The simulator below executes exactly this algorithm and prints the same trace; then a real `tf.function` shows
that stateful ops respect program order.

In [ ]:
graph = {  # op: (device, inputs)
    "A": ("GPU", []), "B": ("CPU", []), "C": ("CPU", []),
    "D": ("GPU", ["A"]), "E": ("CPU", ["B"]), "F": ("CPU", ["E", "C"]),
}
consumers = {op: [o for o, (_, ins) in graph.items() if op in ins] for op in graph}
counter = {op: len(ins) for op, (_, ins) in graph.items()}
queues = {"GPU": [], "CPU": []}
for op, n in counter.items():
    if n == 0:
        queues[graph[op][0]].append(op)
step, trace = 0, []
while any(queues.values()):
    running = {"GPU": queues["GPU"][:1],           # GPU: one op at a time
               "CPU": list(queues["CPU"])}         # CPU: all ready ops in parallel (inter-op pool)
    queues = {"GPU": queues["GPU"][1:], "CPU": []}
    trace.append({"step": step, "GPU runs": running["GPU"], "CPU runs (parallel)": running["CPU"]})
    for op in running["GPU"] + running["CPU"]:
        for cons in consumers[op]:
            counter[cons] -= 1
            if counter[cons] == 0:
                queues[graph[cons][0]].append(cons)
    step += 1
print(pd.DataFrame(trace).to_string(index=False))

v = tf.Variable(0.0)
@tf.function
def ordered():
    v.assign(1.0); tf.print("  assign 1 ->", v)
    v.assign_add(10.0); tf.print("  add 10   ->", v)
    v.assign(v * 2.0); tf.print("  times 2  ->", v)
    return v.read_value()
print("Stateful ops inside tf.function run in program order:")
print("final value:", ordered().numpy(), "(1 -> 11 -> 22)")

## 6.9 Inter-op and intra-op thread pools
Thread-pool sizes are also fixed at runtime start, so each setting runs in a fresh process. The workload has
**independent** matmuls (benefits from inter-op parallelism) that are each **large** (benefits from intra-op threads).

In [ ]:
thread_script = WORK / "thread_probe.py"
thread_script.write_text(r"""
import os, sys, time
os.environ["TF_CPP_MIN_LOG_LEVEL"] = "3"; os.environ["CUDA_VISIBLE_DEVICES"] = ""
import tensorflow as tf
inter, intra = int(sys.argv[1]), int(sys.argv[2])
tf.config.threading.set_inter_op_parallelism_threads(inter)
tf.config.threading.set_intra_op_parallelism_threads(intra)
xs = [tf.random.normal([512, 512]) for _ in range(8)]
@tf.function
def work():
    return tf.add_n([tf.reduce_sum(x @ x) for x in xs])   # 8 independent matmuls
work()
t = []
for _ in range(5):
    t0 = time.perf_counter()
    for _ in range(10): work()
    t.append(time.perf_counter() - t0)
import statistics as st
print(f"{inter},{intra},{80/st.mean(t):.1f},{80/st.mean(t)*st.pstdev(t)/st.mean(t):.1f}")
""")
n_cores = os.cpu_count() or 1
settings = sorted({(1, 1), (1, 2), (2, 2), (n_cores, n_cores)})
rows = []
for inter, intra in settings:
    r = subprocess.run([sys.executable, str(thread_script), str(inter), str(intra)],
                       capture_output=True, text=True, timeout=600)
    try:
        i, j, thr, sd = r.stdout.strip().splitlines()[-1].split(",")
        rows.append({"inter-op threads": int(i), "intra-op threads": int(j),
                     "throughput (matmuls/s)": float(thr), "± std": float(sd)})
    except Exception:
        print("probe failed:", r.stderr[-300:])
threads_df = pd.DataFrame(rows)
print(f"CPU cores available: {n_cores}")
print(threads_df.to_string(index=False))
RESULTS["thread_pools"] = threads_df

**Interpretation:** throughput grows with more threads up to the number of physical cores; beyond that, threads only
compete. For a GPU-heavy training job these knobs rarely matter, but on CPU-only inference servers they are
important (TF Serving exposes them too).

## 6.10 Using several GPUs on one machine without distributing a single model
* **Train several models in parallel, one per GPU** — perfect for hyperparameter search: two models on two GPUs take
  about the same wall time as one model on one GPU.
* **Keep the GPU fed:** a `tf.data` pipeline with `.prefetch()` prepares the next batch on the CPU while the GPU
  trains; `tf.data.experimental.prefetch_to_device("/gpu:0")` (still available, check your version) copies it ahead of
  time so the GPU never waits for the transfer.
* **Split a two-input model**: run each branch (e.g. two CNNs) on a different GPU.
* **Fast ensembles:** one member model per GPU, average their predictions.

Experiment: train two independent models **sequentially** on one device vs **concurrently** on two devices
(`DEVICES[0]`, `DEVICES[1]`) using two threads.

In [ ]:
def train_one(device, steps=150, seed=0):
    with tf.device(device):
        tf.random.set_seed(seed)
        W1 = tf.Variable(tf.random.normal([784, 256], stddev=0.05))
        W2 = tf.Variable(tf.random.normal([256, 10], stddev=0.05))
        ds = (tf.data.Dataset.from_tensor_slices((X_train.reshape(-1, 784), y_train.astype("int64")))
              .shuffle(10000, seed=seed).batch(256).repeat().prefetch(tf.data.AUTOTUNE))
        @tf.function
        def step(x, y):
            with tf.GradientTape() as tape:
                logits = tf.nn.relu(x @ W1) @ W2
                loss = tf.reduce_mean(tf.nn.sparse_softmax_cross_entropy_with_logits(y, logits))
            grads = tape.gradient(loss, [W1, W2])
            for var, g in zip([W1, W2], grads):      # plain SGD update, lr = 0.1
                var.assign_sub(0.1 * g)
            return loss
        for x, y in ds.take(steps):
            loss = step(x, y)
        return float(loss)

train_one(DEVICES[0], steps=5)                      # warm-up (tracing, allocation)
t0 = time.perf_counter()
train_one(DEVICES[0], seed=1); train_one(DEVICES[0], seed=2)
sequential = time.perf_counter() - t0

t0 = time.perf_counter()
with cf.ThreadPoolExecutor(2) as ex:
    futs = [ex.submit(train_one, DEVICES[i % len(DEVICES)], 150, i + 1) for i in range(2)]
    [f.result() for f in futs]
parallel = time.perf_counter() - t0
par_df = pd.DataFrame([{"setup": "2 models, one after another, 1 device", "wall s": round(sequential, 2)},
                       {"setup": f"2 models in parallel on {DEVICES[:2]}", "wall s": round(parallel, 2)}])
print(par_df.to_string(index=False))
print(f"Speed-up: {sequential / parallel:.2f}x")
RESULTS["multi_device_parallel"] = par_df

**Interpretation:** with two **physical** GPUs the speed-up approaches 2×. With two *virtual* GPUs on one card (or two
logical CPUs) both models share the same hardware, so the gain is limited to filling idle gaps (often 1.0–1.4×) —
virtual devices are for **testing code paths**, not for speed. This is exactly why the timing numbers in Phase 7
must be interpreted with the hardware in mind.

### Phase 6 — summary / deliverable
* ✅ `nvidia-smi` + TensorFlow device listings
* ✅ The default "grab everything" behaviour and the four fixes, each in its own process (GPU runtime)
* ✅ Placement experiments: float→GPU, int32→CPU, forced `/cpu:0`, strict vs soft placement, placement logging
* ✅ Parallel-execution walkthrough on my own six-op graph + simulator + stateful-order check
* ✅ Thread-pool throughput table
* ✅ Multi-device parallel training experiment with measured speed-up

---
# Phase 7 — Training Across Multiple Devices: Model vs Data Parallelism & Distribution Strategies

**Goal of this phase:** understand the two ways to spread *one* training job over several devices, the trade-offs
inside data parallelism (mirrored vs centralized, sync vs async), why adding devices eventually stops helping, and
then use the Distribution Strategies API — including a custom distributed training loop.

## 7.1 Model parallelism — split the *model*
Each device holds a different part of the network.

```
 Fully connected net, split by layer          Split each layer vertically
 ┌─────────┐   ┌─────────┐   ┌─────────┐       GPU0: left half of every layer ─┐  every neuron needs ALL
 │ layer 1 │──▶│ layer 2 │──▶│ layer 3 │       GPU1: right half of every layer ─┘  outputs of the previous
 │  GPU 0  │   │  GPU 1  │   │  GPU 2  │       → cross-device traffic at every layer (slow)
 └─────────┘   └─────────┘   └─────────┘
  GPU1 waits for GPU0, GPU2 waits for GPU1 → only one device busy at a time
```
* **Fully connected nets:** little to gain — layer-wise splits make devices wait for each other; vertical splits need
  a full exchange of activations between devices at every layer.
* **Partially connected nets (CNNs):** a neuron only sees a local patch, so a vertical split needs only the border
  activations exchanged → model parallelism works better.
* **Deep RNNs split horizontally** (layer per device) behave like a pipeline: at time step 1 only device 0 works, at
  step 2 devices 0–1, ... once the pipeline fills, all devices are busy. In practice, though, a regular LSTM stack
  on **one** GPU (with cuDNN kernels) is usually much faster.
* Put the most communication-heavy parts on the **same machine** (or the same device).

## 7.2 Data parallelism — replicate the model, split the *data*
```
        mini-batch 1 ─▶ [replica on GPU0] ─▶ grads₁ ─┐
        mini-batch 2 ─▶ [replica on GPU1] ─▶ grads₂ ─┼─▶ average ─▶ same update applied everywhere
        mini-batch 3 ─▶ [replica on GPU2] ─▶ grads₃ ─┘
```
Every device has a full copy of the model and processes a **different mini-batch**; gradients are averaged and the
parameters updated. **Why it is the default choice:** it works the same for any architecture, needs no manual model
surgery, and every device is busy all the time; only gradients/parameters cross devices.

### Two ways to organise it
**(a) Mirrored strategy** — parameters are **mirrored on every device**; after each step all devices run an
**AllReduce**: a collaborative reduction (sum/mean/max) after which **every** device holds the same result. Because
every replica applies the same averaged gradients, replicas stay identical. Efficient AllReduce implementations
(NVIDIA NCCL, ring/hierarchical algorithms) exist off the shelf. Best fit: one machine with several GPUs.

**(b) Centralized parameters** — parameters live **outside** the workers: on the CPU, or on dedicated
**parameter servers** (CPU-only processes that only store and update variables). Workers pull parameters, compute
gradients, push them back. This layout allows both synchronous and asynchronous updates.

## 7.3 Synchronous vs asynchronous updates

| | Synchronous | Asynchronous |
|---|---|---|
| How | aggregator waits for **all** replicas' gradients, averages, updates; replicas wait for new params | each replica updates the params **as soon as** it finishes; nobody waits |
| Speed per step | limited by the **slowest replica** (stragglers) | as many steps/min as the fastest replicas can do |
| Bandwidth | all replicas copy parameters **at the same moment** → can saturate parameter servers | copies spread over time → less contention |
| Gradient quality | exact average of fresh gradients | **stale gradients**: by the time a replica pushes, others have updated ≈ N−1 times |
| Effect | stable, reproducible convergence | slower convergence, noise/wobble, possible divergence |

**Spare replicas (backup workers):** run 20 replicas but apply the update as soon as the **fastest 18** report; the
2 slowest (which change from step to step) are ignored. Step time is no longer set by stragglers, and training
survives a crashed worker.

**Stale-gradient mitigations:** (1) lower the learning rate; (2) drop or down-weight stale gradients;
(3) adjust the mini-batch size; (4) **warm-up**: run the first epochs with one replica (early gradients are large and
staleness hurts most there), then add replicas.

**Evidence:** Chen et al., Google Brain (2016), *Revisiting Distributed Synchronous SGD* — synchronous training with
a few spare replicas converged faster and reached a better model than asynchronous training. Synchronous data
parallelism (AllReduce) is the default today, but the area is still active (local SGD, gradient compression, ...).

## 7.4 Bandwidth saturation
Every step, gradients (and parameters) must cross the interconnect. Adding a device saves compute time but adds
communication; past some point **adding a GPU makes training slower**.
* **Large dense models** suffer most (every parameter's gradient is non-zero and must move);
* **small models** suffer less, but there is also little to gain;
* **large sparse models** (e.g. huge embeddings where each batch touches few rows) suffer least — most gradients are zero.

Speed-ups reported in the source material: dense models **≈25–40× on 50 GPUs**; a sparse model **≈300× on 500 GPUs**;
neural machine translation **6× on 8 GPUs**; Inception/ImageNet **32× on 50 GPUs**; saturation beyond a few dozen GPUs
(dense) or a few hundred (sparse).

Mitigations: a few powerful GPUs rather than many weak ones; group GPUs on few well-interconnected servers
(NVLink/InfiniBand); send gradients in 16-bit (**bfloat16**) to halve traffic; **shard** parameters across several
parameter servers.

## 7.5 Back-of-the-envelope model (my estimate, not a measurement)

**Assumptions**
* dense model, **P = 100 M parameters**, float32 gradients → **S = 400 MB** per step;
* compute time for one global batch on one GPU **T₁ = 1.0 s**, split perfectly: compute = T₁ / N;
* **10 Gbit/s** network = **1.25 GB/s** per machine, one GPU per machine, no compute/communication overlap;
* *Ring AllReduce:* each node sends/receives 2(N−1)/N × S → t_comm = 2(N−1)/N × S / B (+ 5 ms latency per hop);
* *Single parameter server:* it must receive N×S gradients and send N×S parameters → t_comm = 2NS / B.

In [ ]:
P, BYTES, T1, B = 100e6, 4, 1.0, 10e9 / 8      # params, bytes/param, seconds, bytes/s
S = P * BYTES
N = np.arange(1, 65)
ring = T1 / N + np.where(N > 1, 2 * (N - 1) / N * S / B + 0.005 * 2 * (N - 1), 0)
ps = T1 / N + np.where(N > 1, 2 * N * S / B, 0)
ring_bf16 = T1 / N + np.where(N > 1, 2 * (N - 1) / N * (S / 2) / B + 0.005 * 2 * (N - 1), 0)

plt.figure(figsize=(8, 4))
plt.plot(N, T1 / ring, label="ring AllReduce, fp32")
plt.plot(N, T1 / ring_bf16, label="ring AllReduce, bf16 gradients")
plt.plot(N, T1 / ps, label="single parameter server")
plt.plot(N, N, "k:", alpha=0.4, label="ideal linear")
plt.ylim(0, 8); plt.xlabel("number of GPUs (one per machine)"); plt.ylabel("speed-up vs 1 GPU")
plt.title("ESTIMATE: 100M-param dense model on 10 Gbit/s"); plt.legend(); plt.grid(alpha=0.3); plt.show()

print(f"Gradient size per step: {S/1e6:.0f} MB -> {S/B:.2f} s to send once over 10 Gbit/s")
print(f"Best ring speed-up: {(T1/ring).max():.2f}x at N={N[(T1/ring).argmax()]};  "
      f"PS best: {(T1/ps).max():.2f}x at N={N[(T1/ps).argmax()]}")

**Reading the estimate:** with one second of compute and 0.32 s just to move the gradients once, the network
dominates almost immediately — the ring flattens around **2–3×** no matter how many GPUs are added, and a single
parameter server peaks at 1–2 GPUs and then gets **worse**. bf16 gradients raise the ceiling noticeably. This is why
real large-scale training uses NVLink/InfiniBand (10–100× more bandwidth), overlaps communication with the backward
pass, and why the advice is "**more powerful GPUs in fewer machines first**".

## 7.6 Distribution Strategies API — `MirroredStrategy` on one machine
* `with strategy.scope():` — model and optimizer variables created inside are **mirrored variables** (one copy per
  device, kept in sync); Keras is distribution-aware.
* `fit()` splits every **global batch** across replicas → the batch size must be **divisible by the number of
  replicas** (here 128 on 2 replicas = 64 each). `predict()` splits batches across replicas too.
* Here `DEVICES` are the virtual GPUs (or logical CPUs) created in Setup, and `make_mirrored()` (Setup, Step B)
  picks a safe cross-device reduction on CPU-only machines.

In [ ]:
N_DIST = 20000 if QUICK else len(X_train)
Xd, yd = X_train[:N_DIST], y_train[:N_DIST]
GLOBAL_BATCH = 128

class EpochTimer(keras.callbacks.Callback):
    def on_epoch_begin(self, epoch, logs=None):
        self._t = time.perf_counter()
    def on_epoch_end(self, epoch, logs=None):
        self.times = getattr(self, "times", []) + [time.perf_counter() - self._t]

def mnist_net():
    return keras.Sequential([keras.Input((28, 28)), keras.layers.Flatten(),
                             keras.layers.Dense(300, activation="relu"),
                             keras.layers.Dense(100, activation="relu"),
                             keras.layers.Dense(10, activation="softmax")])

def train_under(strategy, epochs=2, batch=GLOBAL_BATCH):
    with strategy.scope():
        m = mnist_net()
        # Adam (not Nadam): Keras 3's Nadam keeps an extra non-slot variable that CentralStorageStrategy
        # cannot update from a replica context ("specify an aggregation method"), Adam works everywhere.
        m.compile(loss="sparse_categorical_crossentropy", optimizer="adam", metrics=["accuracy"])
    timer = EpochTimer()
    h = m.fit(Xd, yd, epochs=epochs, batch_size=batch, validation_data=(X_valid, y_valid),
              verbose=0, callbacks=[timer])
    return m, timer.times[-1], h.history["val_accuracy"][-1]   # last epoch: excludes tracing/warm-up

mirrored = make_mirrored(DEVICES)
print("Replicas in sync:", mirrored.num_replicas_in_sync, "| per-replica batch:",
      GLOBAL_BATCH // mirrored.num_replicas_in_sync)
mirrored_model, t_epoch, val_acc = train_under(mirrored)
print(f"MirroredStrategy: {t_epoch:.2f} s/epoch, val acc {val_acc:.4f}")
v0 = mirrored_model.trainable_variables[0]
inner = getattr(v0, "value", v0)
inner = v0 if callable(inner) else inner          # Keras 3 wraps the tf variable in .value
print("A variable created inside the scope is a:", type(inner).__name__)

### Saving and loading a distributed model
`save()` writes an ordinary, **non-mirrored** model — it loads on a single device. To run a loaded model on all
devices again, call `load_model()` **inside** a strategy scope. To restrict the devices:
`tf.distribute.MirroredStrategy(["/gpu:0", "/gpu:1"])`.

In [ ]:
dist_file = WORK / "mirrored_model.keras"
mirrored_model.save(dist_file)
single = keras.models.load_model(dist_file)
with mirrored.scope():
    redistributed = keras.models.load_model(dist_file)
print("Loaded normally    -> variable on:", single.trainable_variables[0].device if hasattr(
    single.trainable_variables[0], "device") else "default device")
print("Loaded in scope    -> replicas:", mirrored.num_replicas_in_sync,
      "| predictions equal:", np.allclose(single.predict(X_new, verbose=0),
                                          redistributed.predict(X_new, verbose=0), atol=1e-6))

## 7.7 AllReduce implementations
`MirroredStrategy(cross_device_ops=...)` chooses how gradients are combined:
* `tf.distribute.NcclAllReduce()` — **default on multiple GPUs**, NVIDIA's collective library (GPU only);
* `tf.distribute.HierarchicalCopyAllReduce()` — reduces in groups then combines (designed for 8-GPU boxes);
* `tf.distribute.ReductionToOneDevice()` — copies everything to one device, reduces there, broadcasts back (works on CPU).

In [ ]:
cdo_rows = []
for name, make in [("ReductionToOneDevice", tf.distribute.ReductionToOneDevice),
                   ("HierarchicalCopyAllReduce", tf.distribute.HierarchicalCopyAllReduce),
                   ("NcclAllReduce", tf.distribute.NcclAllReduce)]:
    try:
        strat = tf.distribute.MirroredStrategy(DEVICES, cross_device_ops=make())
        _, t, acc = train_under(strat)
        cdo_rows.append({"cross_device_ops": name, "s/epoch": round(t, 2), "val acc": round(acc, 4)})
    except Exception as e:
        cdo_rows.append({"cross_device_ops": name, "s/epoch": None,
                         "val acc": f"not supported here ({type(e).__name__})"})
cdo_df = pd.DataFrame(cdo_rows)
print(cdo_df.to_string(index=False))
RESULTS["allreduce_impls"] = cdo_df

## 7.8 Single device vs MirroredStrategy vs CentralStorageStrategy (3 repeats, mean ± std)
`CentralStorageStrategy(compute_devices=..., parameter_device=...)` keeps one copy of the variables on the
parameter device (default: the CPU, or the GPU if there is only one) and computes on all GPUs (default).

In [ ]:
def time_strategy(make_strategy, repeats=3):
    times, accs = [], []
    for _ in range(repeats):
        _, t, acc = train_under(make_strategy())
        times.append(t); accs.append(acc)
    return np.mean(times), np.std(times), np.mean(accs)

strategies = {
    "single device (OneDeviceStrategy)": lambda: tf.distribute.OneDeviceStrategy(DEVICES[0]),
    "MirroredStrategy": lambda: make_mirrored(DEVICES),
    "CentralStorageStrategy": lambda: tf.distribute.experimental.CentralStorageStrategy(
        compute_devices=DEVICES, parameter_device="/cpu:0"),
}
rows = []
for name, make in strategies.items():
    try:
        m, s, acc = time_strategy(make, repeats=3)
        rows.append({"strategy": name, "s/epoch (mean ± std)": f"{m:.2f} ± {s:.2f}", "mean": m,
                     "val acc": round(acc, 4)})
    except Exception as e:
        rows.append({"strategy": name, "s/epoch (mean ± std)": f"failed: {type(e).__name__}: {str(e)[:80]}",
                     "mean": np.nan, "val acc": None})
strat_df = pd.DataFrame(rows)
base = strat_df["mean"].iloc[0]
strat_df["speed-up vs single"] = (base / strat_df["mean"]).round(2)
print("Devices:", DEVICES, "| hardware:", "GPU" if HAS_GPU else "CPU (logical devices)")
print(strat_df.drop(columns="mean").to_string(index=False))
RESULTS["strategy_timing"] = strat_df.drop(columns="mean")

**Interpretation**
* On **two physical GPUs**, `MirroredStrategy` should approach 2× for a compute-heavy model; `CentralStorageStrategy`
  is usually slower because every step copies parameters from the CPU to each GPU and gradients back.
* On **virtual devices** (this notebook's default layout) both replicas share one physical device, so a speed-up is
  **not expected** — you mostly measure synchronisation overhead. For this small MNIST model the per-step work is so
  light that overhead dominates; that is the bandwidth-saturation lesson in miniature.
* Accuracy is the same within noise for all strategies: distribution changes *speed*, not the maths of
  synchronous SGD with the same global batch.

## 7.9 A custom training loop with a strategy
Pieces (from the current TF distributed-training guide):
* create the model and optimizer **inside** `strategy.scope()`;
* `strategy.experimental_distribute_dataset(ds)` splits each **global batch** into per-replica batches;
* in the replica step, compute the **per-example** loss and average it with
  `tf.nn.compute_average_loss(per_example_loss, global_batch_size=GLOBAL_BATCH)` — dividing by the *global* batch so
  that summing over replicas gives the correct mean;
* `strategy.run(step, args=...)` executes the step on every replica; `strategy.reduce(SUM, ...)` combines the losses.

*Adaptation notes:* with Keras 3 the optimizer's slot variables must be created **inside the scope before the
first step** (`optimizer.build(model.trainable_variables)`), otherwise they get created lazily inside the replicated
`tf.function` and the step fails; the per-example loss comes from `keras.losses.sparse_categorical_crossentropy`
(the function, not the class), and the last partial batch is dropped (`drop_remainder=True`) so every replica gets
an equal share.

In [ ]:
def custom_distributed_training(strategy, epochs=2, per_replica_batch=64, lr=1e-3, seed=SEED):
    global_batch = per_replica_batch * strategy.num_replicas_in_sync
    ds = (tf.data.Dataset.from_tensor_slices((Xd, yd)).shuffle(len(Xd), seed=seed)
          .batch(global_batch, drop_remainder=True).prefetch(tf.data.AUTOTUNE))
    dist_ds = strategy.experimental_distribute_dataset(ds)
    with strategy.scope():
        tf.random.set_seed(seed)
        model = mnist_net()
        optimizer = keras.optimizers.Adam(learning_rate=lr)
        if hasattr(optimizer, "build"):
            optimizer.build(model.trainable_variables)   # Keras 3: create slot variables up front, inside the scope

    def replica_step(x, y):
        with tf.GradientTape() as tape:
            probs = model(x, training=True)
            per_example = keras.losses.sparse_categorical_crossentropy(y, probs)
            loss = tf.nn.compute_average_loss(per_example, global_batch_size=global_batch)
        grads = tape.gradient(loss, model.trainable_variables)
        optimizer.apply_gradients(zip(grads, model.trainable_variables))
        return loss

    @tf.function
    def distributed_step(x, y):
        per_replica_losses = strategy.run(replica_step, args=(x, y))
        return strategy.reduce(tf.distribute.ReduceOp.SUM, per_replica_losses, axis=None)

    history = []
    for epoch in range(epochs):
        total, n = 0.0, 0
        for x, y in dist_ds:
            total += float(distributed_step(x, y)); n += 1
        val_acc = float(np.mean(np.argmax(model.predict(X_valid, verbose=0), 1) == y_valid))
        history.append((total / n, val_acc))
        print(f"  epoch {epoch+1}: loss {total/n:.4f}  val acc {val_acc:.4f}")
    return model, history

print("Custom loop under MirroredStrategy:")
_, custom_hist = custom_distributed_training(make_mirrored(DEVICES))

print("fit() under MirroredStrategy (same global batch):")
with make_mirrored(DEVICES).scope():
    tf.random.set_seed(SEED)
    fit_model = mnist_net()
    fit_model.compile(loss="sparse_categorical_crossentropy", optimizer=keras.optimizers.Adam(1e-3),
                      metrics=["accuracy"])
fit_hist = fit_model.fit(Xd, yd, epochs=2, batch_size=64 * len(DEVICES),
                         validation_data=(X_valid, y_valid), verbose=0).history
parity = pd.DataFrame([{"method": "custom loop + strategy.run", "final loss": custom_hist[-1][0],
                        "val acc": custom_hist[-1][1]},
                       {"method": "model.fit", "final loss": fit_hist["loss"][-1],
                        "val acc": fit_hist["val_accuracy"][-1]}]).round(4)
print(parity.to_string(index=False))
print("Parity (val acc within 1 point):", abs(parity["val acc"].diff().iloc[-1]) < 0.01)

## 7.10 Learning rate & batch size under data parallelism (hypothesis test)
**Hypothesis:** with N replicas and a fixed per-replica batch, the **global batch grows N×**, so each epoch has N×
fewer updates. Keeping the same learning rate should therefore make progress *per epoch* slower; scaling the
learning rate linearly with the global batch (plus a short warm-up so the first big steps don't blow up) should
recover it. We test (a) same LR vs (b) LR × N with a 50-step linear warm-up, using SGD with momentum
(where LR scaling matters most), for 1, 2 and (if available) 4 replicas.

In [ ]:
class LinearWarmup(keras.optimizers.schedules.LearningRateSchedule):
    def __init__(self, peak_lr, warmup_steps):
        self.peak_lr, self.warmup_steps = peak_lr, warmup_steps
    def __call__(self, step):
        ops = keras.ops if hasattr(keras, "ops") else tf
        step = ops.cast(step, "float32")
        return self.peak_lr * ops.minimum(1.0, (step + 1.0) / self.warmup_steps)
    def get_config(self):
        return {"peak_lr": self.peak_lr, "warmup_steps": self.warmup_steps}

class StepLoss(keras.callbacks.Callback):
    def on_train_begin(self, logs=None):
        self.losses = []
    def on_train_batch_end(self, batch, logs=None):
        self.losses.append(logs["loss"])

BASE_LR, PER_REPLICA, TARGET_LOSS = 0.01, 32, 0.5
replica_counts = [n for n in (1, 2, 4) if n <= len(DEVICES)]
rows = []
for n in replica_counts:
    for mode in ("same LR", "LR x N + warm-up"):
        strat = make_mirrored(DEVICES[:n])
        lr = BASE_LR if mode == "same LR" else LinearWarmup(BASE_LR * n, 50)
        with strat.scope():
            tf.random.set_seed(SEED)
            m = mnist_net()
            m.compile(loss="sparse_categorical_crossentropy",
                      optimizer=keras.optimizers.SGD(learning_rate=lr, momentum=0.9), metrics=["accuracy"])
        cb = StepLoss()
        h = m.fit(Xd, yd, epochs=2, batch_size=PER_REPLICA * n, validation_data=(X_valid, y_valid),
                  verbose=0, callbacks=[cb])
        hit = next((i + 1 for i, l in enumerate(cb.losses) if l < TARGET_LOSS), None)
        rows.append({"replicas": n, "global batch": PER_REPLICA * n, "learning rate": mode,
                     "steps (2 epochs)": len(cb.losses), f"steps to loss<{TARGET_LOSS}": hit,
                     "val acc": round(h.history["val_accuracy"][-1], 4)})
lr_df = pd.DataFrame(rows)
print(lr_df.to_string(index=False))
RESULTS["lr_batch_experiment"] = lr_df

**What I found** (fill in from the table — typical outcome): with the **same LR**, larger replica counts finish the
2 epochs with fewer updates and slightly lower validation accuracy; **scaling the LR with the global batch (with
warm-up)** recovers most of the gap and reaches the target loss in fewer steps. *Steps to target* is noisy because
Keras reports a running average of the loss within an epoch. Caveats: the linear-scaling rule breaks down at very
large batches (then LARS/LAMB-style optimizers are used), and with only 2 epochs these differences are small.

## 7.11 `TPUStrategy`
```python
resolver = tf.distribute.cluster_resolver.TPUClusterResolver()   # finds the TPU (Colab/Kaggle/Cloud TPU VM)
tf.config.experimental_connect_to_cluster(resolver)
tf.tpu.experimental.initialize_tpu_system(resolver)
strategy = tf.distribute.TPUStrategy(resolver)
with strategy.scope():
    model = mnist_net(); model.compile(...)        # then fit() exactly as before
```
It is used exactly like the other strategies. Constraints: TPUs are available on Google Cloud, Kaggle and (subject to
availability) Colab; input pipelines must be fast (`tf.data` with large batches, often from GCS buckets), shapes must
be static, and per-core batches should be multiples of 8/128 to fill the matrix units. The cell below runs only if a
TPU is attached.

In [ ]:
try:
    resolver = tf.distribute.cluster_resolver.TPUClusterResolver()
    tf.config.experimental_connect_to_cluster(resolver)
    tf.tpu.experimental.initialize_tpu_system(resolver)
    tpu_strategy = tf.distribute.TPUStrategy(resolver)
    print("TPU cores:", tpu_strategy.num_replicas_in_sync)
    _, t_tpu, acc_tpu = train_under(tpu_strategy, batch=128 * tpu_strategy.num_replicas_in_sync)
    print(f"TPU: {t_tpu:.2f} s/epoch, val acc {acc_tpu:.4f}")
except Exception as e:
    print("No TPU attached in this runtime -> TPUStrategy described above, not run.", type(e).__name__)

### Phase 7 — summary / deliverable
* ✅ Model vs data parallelism explained with diagrams (FC, CNN, RNN-pipeline cases)
* ✅ Mirrored vs centralized; **sync vs async table**, spare replicas, stale-gradient mitigations, 2016 finding
* ✅ Bandwidth saturation + reported speed-ups + **my own estimate plot** with stated assumptions
* ✅ `MirroredStrategy` training, save/load in and out of scope, AllReduce implementations compared
* ✅ **Single vs Mirrored vs CentralStorage timing table** (3 repeats, mean ± std) with interpretation
* ✅ Custom distributed training loop (`strategy.run` / `strategy.reduce`) with a parity check against `fit()`
* ✅ Batch-size / learning-rate experiment and `TPUStrategy` setup

---
# Phase 8 — Clusters, Cloud Training Jobs, Hyperparameter Tuning & Capstone

**Goal of this phase:** go beyond one machine (cluster spec, `TF_CONFIG`, multi-worker training), see how cloud
training and tuning jobs plug into the same code, run a local tuning analog, and finish with an end-to-end
production capstone.

## 8.1 What a TensorFlow cluster is
A **cluster** is a group of TensorFlow processes ("**tasks**", each a small TF server with an IP address and a port),
usually on different machines, working on one job. Every task has a **type** (also called its *job*):

| Type | Role |
|---|---|
| `worker` | does the computation, usually on GPUs |
| `chief` | a worker with extra duties: writes TensorBoard logs and checkpoints, saves the final model. Exactly one; if none is declared, **worker 0** plays the chief |
| `ps` | parameter server: only stores and updates variables, typically CPU-only; used **only** with `ParameterServerStrategy` |
| `evaluator` | evaluates checkpoints on validation data alongside training |

## 8.2 Cluster spec
A dictionary with one key per job type; each value lists the `host:port` of that job's tasks:

In [ ]:
cluster_spec = {
    "worker": ["machine-a.example.com:2222",     # /job:worker/task:0  (acts as chief if none declared)
               "machine-b.example.com:2222"],    # /job:worker/task:1
    "ps":     ["machine-a.example.com:2221"],    # /job:ps/task:0 -> shares machine-a, different port
}
for job, addrs in cluster_spec.items():
    for i, a in enumerate(addrs):
        print(f"/job:{job}/task:{i:<2} -> {a}")

* A task is named `/job:<type>/task:<index>`, and its devices `/job:worker/task:1/device:GPU:0`.
* By default **every task talks to every other task**, so firewalls must allow all these ports between all machines
  (using the **same port everywhere** keeps the rules simple).
* Several tasks may share one machine (worker + ps above) — split GPU memory between them if both use the GPU (Phase 6).

## 8.3 `TF_CONFIG`
Each process learns the cluster **and its own role** from the `TF_CONFIG` environment variable:
`{"cluster": <cluster spec>, "task": {"type": "worker", "index": 1}}`.
* Set it **outside Python** (shell, Kubernetes manifest, cloud platform) — the training script stays **identical on
  every machine**; only the environment differs.
* Each task needs a **distinct (type, index)** pair: it is how a process knows which address in the spec is *itself*
  (it starts its server on that port) and whether it is the chief.

In [ ]:
for i in range(len(cluster_spec["worker"])):
    cfg = {"cluster": cluster_spec, "task": {"type": "worker", "index": i}}
    print(f"machine for worker {i}:  export TF_CONFIG='{json.dumps(cfg)}'")
print(f"machine for ps 0:      export TF_CONFIG='{json.dumps({'cluster': cluster_spec, 'task': {'type': 'ps', 'index': 0}})}'")

## 8.4 `MultiWorkerMirroredStrategy` — two workers on this machine
Same idea as `MirroredStrategy`; the strategy reads `TF_CONFIG`. Each worker holds a full replica; gradients are
combined with a **collective AllReduce across machines**. No `ps` key in the spec.

*Version note:* with Keras 3 (TF ≥ 2.16) `model.fit()` under `MultiWorkerMirroredStrategy` fails in current releases
(it tries to convert the per-replica batch into a single tensor), so the worker script uses the **custom distributed
training loop** from Phase 7.9 — `strategy.run` + `strategy.reduce` — which is exactly what `fit()` does internally.
With `tf_keras` (Keras 2) the plain `model.fit(...)` version from the source works unchanged.

We launch **two worker processes on localhost** with different ports, forced onto the CPU
(`CUDA_VISIBLE_DEVICES=""`) so they don't fight over one GPU. Worker 1 starts **8 seconds later** to show the
blocking behaviour: worker 0 cannot pass the first collective operation until worker 1 joins; afterwards both advance
in lock-step. Only the chief keeps the saved model; the other worker saves to a temporary path (all workers must take
part in saving).

In [ ]:
MW_DIR = WORK / "multiworker"
MW_DIR.mkdir(exist_ok=True)
worker_script = MW_DIR / "mw_worker.py"
worker_script.write_text(r"""
import os, sys, json, time
os.environ["TF_CPP_MIN_LOG_LEVEL"] = "3"
os.environ["CUDA_VISIBLE_DEVICES"] = ""
import numpy as np, tensorflow as tf
from tensorflow import keras
cfg = json.loads(os.environ["TF_CONFIG"]); idx = cfg["task"]["index"]
def log(msg): print(f"{time.time():.2f} | worker {idx} | {msg}", flush=True)

log("process started, creating MultiWorkerMirroredStrategy")
strategy = tf.distribute.MultiWorkerMirroredStrategy()
log(f"strategy created, replicas in sync = {strategy.num_replicas_in_sync}")
(X, y), _ = keras.datasets.mnist.load_data()
X, y = (X[:8000] / 255.0).astype("float32"), y[:8000].astype("int64")
global_batch = 64 * strategy.num_replicas_in_sync
ds = tf.data.Dataset.from_tensor_slices((X, y)).shuffle(8000, seed=42).batch(global_batch, drop_remainder=True)
opts = tf.data.Options()
opts.experimental_distribute.auto_shard_policy = tf.data.experimental.AutoShardPolicy.DATA
ds = ds.with_options(opts)                       # each worker reads a different shard of every batch
dist_ds = strategy.experimental_distribute_dataset(ds)
log("building model inside scope (variable creation needs all workers)")
with strategy.scope():
    model = keras.Sequential([keras.Input((28, 28)), keras.layers.Flatten(),
                              keras.layers.Dense(128, activation="relu"), keras.layers.Dense(10, activation="softmax")])
    optimizer = keras.optimizers.Adam(1e-3)
    optimizer.build(model.trainable_variables)
log("model built -> all workers have joined")

def replica_step(x, y):
    with tf.GradientTape() as tape:
        per_example = keras.losses.sparse_categorical_crossentropy(y, model(x, training=True))
        loss = tf.nn.compute_average_loss(per_example, global_batch_size=global_batch)
    grads = tape.gradient(loss, model.trainable_variables)
    optimizer.apply_gradients(zip(grads, model.trainable_variables))   # gradients are all-reduced across workers
    return loss

@tf.function
def train_step(x, y):
    return strategy.reduce(tf.distribute.ReduceOp.SUM, strategy.run(replica_step, args=(x, y)), axis=None)

for epoch in range(3):
    log(f"epoch {epoch} begins")
    total, n = 0.0, 0
    for x, yb in dist_ds:
        total += float(train_step(x, yb)); n += 1
    log(f"epoch {epoch} ends  loss={total / n:.4f}  ({n} steps)")
out = sys.argv[1]
path = os.path.join(out, "final_model.keras") if idx == 0 else os.path.join(out, f"tmp_worker{idx}.keras")
model.save(path)
log(f"saved to {path}" + ("  (chief keeps this)" if idx == 0 else "  (temporary, deleted below)"))
if idx != 0:
    os.remove(path)
""")

ports = [free_port(), free_port()]
mw_cluster = {"worker": [f"localhost:{p}" for p in ports]}
procs = []
for i in range(2):
    env = dict(os.environ, TF_CONFIG=json.dumps({"cluster": mw_cluster, "task": {"type": "worker", "index": i}}))
    procs.append(subprocess.Popen([sys.executable, str(worker_script), str(MW_DIR)], env=env,
                                  stdout=subprocess.PIPE, stderr=subprocess.STDOUT, text=True))
    if i == 0:
        time.sleep(8)                    # worker 1 joins late on purpose
mw_logs = []
for p in procs:
    try:
        out, _ = p.communicate(timeout=900)
    except subprocess.TimeoutExpired:
        p.kill(); out, _ = p.communicate()
    mw_logs += [l for l in out.splitlines() if " | worker " in l]
mw_logs.sort(key=lambda l: float(l.split(" | ")[0]))
t0 = float(mw_logs[0].split(" | ")[0]) if mw_logs else 0
print(f"Cluster: {mw_cluster}\n")
for l in mw_logs:
    ts, rest = l.split(" | ", 1)
    print(f"+{float(ts) - t0:6.2f}s | {rest}")

**What the log shows:** worker 0 starts immediately but cannot finish creating its strategy: it **blocks** until
worker 1 starts ~8 s later (the collective group needs every member). From then on both report each epoch with
**identical losses** — they are synchronous replicas whose gradients are all-reduced at every step, so their weights
never diverge (small timestamp differences are just log flushing on a shared CPU). The chief (worker 0) keeps
`final_model.keras`; worker 1's temporary save is deleted.

## 8.5 Multi-worker AllReduce implementations
`MultiWorkerMirroredStrategy` supports two collective implementations:
* **RING** — ring AllReduce over gRPC; works on CPUs and GPUs, any network;
* **NCCL** — NVIDIA's library, much faster between GPUs (NVLink/InfiniBand aware).

By default (`AUTO`) TF picks heuristically. To force one:
```python
options = tf.distribute.experimental.CommunicationOptions(
    implementation=tf.distribute.experimental.CommunicationImplementation.NCCL)   # or .RING
strategy = tf.distribute.MultiWorkerMirroredStrategy(communication_options=options)
```
(The 2019 API used `communication=tf.distribute.experimental.CollectiveCommunication.NCCL`.)

## 8.6 `ParameterServerStrategy`
Asynchronous data parallelism with parameter servers: **workers run asynchronously** with respect to each other (each
pushes its own gradients to the `ps` tasks), while replicas **within** a multi-GPU worker are synchronous. Variables are
sharded across the `ps` tasks.

**Then vs now:** in 2019 every task ran the same script with `ParameterServerStrategy()`. Current TF uses a
**coordinator** model: workers and parameter servers just start a `tf.distribute.Server` and wait; one **chief/
coordinator** process creates the strategy and *schedules* steps onto workers via a `ClusterCoordinator`:
```python
# on workers and ps tasks:
server = tf.distribute.Server(cluster_resolver.cluster_spec(), job_name=..., task_index=..., protocol="grpc")
server.join()
# on the coordinator (chief):
resolver = tf.distribute.cluster_resolver.TFConfigClusterResolver()
strategy = tf.distribute.ParameterServerStrategy(resolver)
with strategy.scope():
    model = ...; model.compile(...)
model.fit(dataset_fn_based_input, epochs=..., steps_per_epoch=...)   # Keras fit supports it
# (or: coordinator = tf.distribute.coordinator.ClusterCoordinator(strategy); coordinator.schedule(step_fn, ...))
```
Differences recorded: a dedicated coordinator process, datasets created per worker through a
`DatasetCreator`/`dataset_fn`, `steps_per_epoch` required, and the cluster spec needs a `chief` plus `worker` and `ps`.

## 8.7 Cloud training jobs
Workflow (mapped to the current platform):
1. Install the **Google Cloud SDK** (or use the in-browser **Cloud Shell**), `gcloud init`, select the project.
2. Write the training code exactly as for your own cluster — the platform **sets `TF_CONFIG`** on every machine.
3. Package the code (a Python package with `setup.py`/`pyproject.toml`); the CLI builds it, uploads it to a
   **staging bucket**, and installs it on each machine.
4. Read data directly from buckets: `tf.data.TFRecordDataset("gs://bucket/train-*.tfrecord")` works because TF's
   file I/O goes through `tf.io.gfile`, which understands `gs://` — the job's **service account** needs read access.
5. The **chief** writes checkpoints, TensorBoard logs and the final model to the **job directory** (`gs://…/job_dir`).

**Original (AI Platform) command, annotated:**
```bash
gcloud ai-platform jobs submit training my_job_20191120_001 \   # unique job name
    --region asia-southeast1 \                                   # where the machines run
    --scale-tier PREMIUM_1 \                                     # preset: 20 workers incl. chief + 11 parameter servers
    --runtime-version 2.0 --python-version 3.7 \                 # TF + Python images
    --package-path /my_project/src/trainer \                     # local code folder -> built into a pip package
    --module-name trainer.task \                                 # the module to run (python -m trainer.task)
    --staging-bucket gs://my-staging-bucket \                    # where the package is uploaded
    --job-dir gs://my-mnist-model-bucket/trained_model \         # passed to the code as --job-dir
    -- \                                                         # everything after this goes to YOUR script
    --my-extra-argument1 foo --my-extra-argument2 bar
```
**Current (Vertex AI custom job) equivalent:**
```bash
gcloud ai custom-jobs create --region=us-central1 --display-name=mnist-multiworker \
  --worker-pool-spec=machine-type=n1-standard-8,replica-count=1,accelerator-type=NVIDIA_TESLA_T4,accelerator-count=2,executor-image-uri=$PREBUILT_TF_GPU_TRAIN_IMAGE,local-package-path=./src,python-module=trainer.task \
  --worker-pool-spec=machine-type=n1-standard-8,replica-count=3,accelerator-type=NVIDIA_TESLA_T4,accelerator-count=2,executor-image-uri=$PREBUILT_TF_GPU_TRAIN_IMAGE,local-package-path=./src,python-module=trainer.task \
  --args=--job-dir=gs://$BUCKET/jobs/mnist
```
(first pool = chief, second pool = workers; `AIP_MODEL_DIR` / `--job-dir` tell the chief where to save; verify the
prebuilt image URI and flags in the current Vertex docs.)

**Batch prediction:** the same job machinery can run a trained model over a huge dataset — each worker reads a shard
from storage, predicts and writes results back (Vertex **batch prediction jobs** do this as a managed service).

## 8.8 Black-box hyperparameter tuning

**The original YAML, annotated**
```yaml
trainingInput:
  hyperparameters:
    goal: MAXIMIZE                       # we want the metric as high as possible
    hyperparameterMetricTag: accuracy    # name of the summary the service reads from TensorBoard event files
    maxTrials: 10                        # total training runs
    maxParallelTrials: 2                 # how many run at the same time (more parallel = less Bayesian guidance)
    params:
      - parameterName: n_layers          # passed to the script as --n_layers=<value>
        type: INTEGER
        minValue: 10
        maxValue: 100
        scaleType: UNIT_LINEAR_SCALE     # flat prior: every value equally plausible
      - parameterName: momentum          # passed as --momentum=<value>
        type: DOUBLE
        minValue: 0.1
        maxValue: 1.0
        scaleType: UNIT_LOG_SCALE        # searched in log space (see note below)
```
* **Scale types as priors.** `UNIT_LINEAR_SCALE`: no preference. `UNIT_LOG_SCALE` / `UNIT_REVERSE_LOG_SCALE`: the
  range is mapped through a log (or reversed log) before the optimizer works on it. The 2019 material reads log scale
  as "the optimum is probably closer to the max" and reverse-log as "closer to the min"; mechanically, log scale gives
  equal attention to each *order of magnitude* (0.1–0.3 gets as much as 0.3–1.0), which is the natural choice for
  learning rates and similar multiplicative knobs.
* **How values reach the code:** as command-line arguments (`--n_layers 37 --momentum 0.62`) — the script parses them.
* **How the metric reaches the service:** the service watches the job directory for **TensorBoard event files** and
  reads the summary whose tag matches `hyperparameterMetricTag`. So a `keras.callbacks.TensorBoard(log_dir=job_dir)`
  callback is all you need. (On Vertex AI the trial reports via the `cloudml-hypertune` helper or the same idea, and the
  config is a `studySpec` passed to `gcloud ai hp-tuning-jobs create`.)
* The service is **Bayesian / black-box** (Google Vizier, Golovin et al. 2017): it builds a model of
  "hyperparameters → metric" from finished trials and picks the next trial where improvement is most likely.

## 8.9 Local analog: random search with 10 trials, 2 at a time
The trial script takes `--n_layers`, `--momentum` and `--job-dir`, trains a small model, and writes TensorBoard logs.
Because a tuning service reads the metric **from the event files**, our "service" does the same: it reads the
validation accuracy back from the logs, then ranks the trials. `n_layers` is scaled down to 1–8 (instead of 10–100)
so 10 trials finish quickly; momentum keeps the 0.1–1.0 log-scale range (capped at 0.99 — momentum 1.0 never decays).

In [ ]:
TUNE_DIR = WORK / "tuning"
shutil.rmtree(TUNE_DIR, ignore_errors=True)
TUNE_DIR.mkdir(parents=True)
trial_script = TUNE_DIR / "trial.py"
trial_script.write_text(r"""
import argparse, os
os.environ["TF_CPP_MIN_LOG_LEVEL"] = "3"
import numpy as np, tensorflow as tf
from tensorflow import keras
p = argparse.ArgumentParser()
p.add_argument("--n_layers", type=int, required=True)
p.add_argument("--momentum", type=float, required=True)
p.add_argument("--job-dir", dest="job_dir", required=True)
p.add_argument("--epochs", type=int, default=3)
a = p.parse_args()
(X, y), _ = keras.datasets.mnist.load_data()
X = (X / 255.0).astype("float32")
Xtr, ytr, Xva, yva = X[:12000], y[:12000], X[-5000:], y[-5000:]
tf.random.set_seed(0)
layers = [keras.Input((28, 28)), keras.layers.Flatten()]
layers += [keras.layers.Dense(64, activation="relu") for _ in range(a.n_layers)]
layers += [keras.layers.Dense(10, activation="softmax")]
model = keras.Sequential(layers)
model.compile(loss="sparse_categorical_crossentropy",
              optimizer=keras.optimizers.SGD(learning_rate=0.01, momentum=a.momentum), metrics=["accuracy"])
model.fit(Xtr, ytr, epochs=a.epochs, batch_size=64, validation_data=(Xva, yva), verbose=0,
          callbacks=[keras.callbacks.TensorBoard(log_dir=a.job_dir)])   # <- the only "reporting" needed
print("done", a.n_layers, a.momentum)
""")

rng = np.random.default_rng(SEED)
trials = [{"trial": i, "n_layers": int(rng.integers(1, 9)),
           "momentum": float(np.clip(10 ** rng.uniform(np.log10(0.1), np.log10(1.0)), 0.1, 0.99))}
          for i in range(10)]

def launch(t, gpu_index=None):
    env = dict(os.environ)
    if HAS_GPU and len(physical_gpus) >= 2 and gpu_index is not None:
        env["CUDA_VISIBLE_DEVICES"] = str(gpu_index)       # one trial per physical GPU
    elif HAS_GPU:
        env["TF_FORCE_GPU_ALLOW_GROWTH"] = "true"          # two trials share one GPU safely
    jd = TUNE_DIR / f"trial_{t['trial']:02d}"
    t["job_dir"] = str(jd)
    return subprocess.Popen([sys.executable, str(trial_script), "--n_layers", str(t["n_layers"]),
                             "--momentum", f"{t['momentum']:.4f}", "--job-dir", str(jd),
                             "--epochs", "3" if QUICK else "8"],
                            env=env, stdout=subprocess.PIPE, stderr=subprocess.STDOUT, text=True)

t0 = time.perf_counter()
for k in range(0, len(trials), 2):                 # maxParallelTrials = 2
    batch = trials[k:k + 2]
    pair = [launch(t, gpu_index=j) for j, t in enumerate(batch)]
    for t, p in zip(batch, pair):
        out, _ = p.communicate(timeout=1800)
        if p.returncode != 0:
            print(f"trial {t['trial']} FAILED:\n" + "\n".join(out.splitlines()[-5:]))
print(f"10 trials finished in {time.perf_counter() - t0:.0f} s")

In [ ]:
from tensorflow.core.util import event_pb2

def read_scalar(logdir, tag):
    """Read (step, value) pairs for `tag` from TensorBoard event files — what the tuning service does."""
    values = []
    for f in sorted(pathlib.Path(logdir).rglob("events.out.tfevents.*")):
        for rec in tf.data.TFRecordDataset(str(f)):
            ev = event_pb2.Event.FromString(rec.numpy())
            for v in ev.summary.value:
                if v.tag == tag:
                    val = float(tf.make_ndarray(v.tensor)) if v.HasField("tensor") else v.simple_value
                    values.append((ev.step, val))
    return sorted(values)

for t in trials:
    vals = read_scalar(pathlib.Path(t["job_dir"]) / "validation", "epoch_accuracy")
    t["val_accuracy"] = vals[-1][1] if vals else np.nan
tune_df = pd.DataFrame(trials).drop(columns="job_dir").sort_values("val_accuracy", ascending=False)
tune_df["rank"] = range(1, len(tune_df) + 1)
print(tune_df[["rank", "trial", "n_layers", "momentum", "val_accuracy"]].round(4).to_string(index=False))
RESULTS["tuning"] = tune_df

fig, ax = plt.subplots(figsize=(6, 4))
sc = ax.scatter(tune_df["momentum"], tune_df["n_layers"], c=tune_df["val_accuracy"], s=120, cmap="viridis")
ax.set_xscale("log"); ax.set_xlabel("momentum (log scale)"); ax.set_ylabel("n_layers")
plt.colorbar(sc, label="val accuracy"); ax.set_title("Random search trials (metric read from event files)")
plt.show()

**Interpretation:** with plain SGD at lr = 0.01 and only 3 epochs, **higher momentum** speeds learning up a lot,
while **very deep** stacks of small Dense layers train more slowly (harder optimisation). A Bayesian service would
notice this after a few trials and concentrate the remaining ones in the high-momentum / few-layers corner, whereas
random search keeps sampling everywhere — that's the efficiency argument for the managed tuner. (With
`USE_CLOUD = True` you would submit the same `trial.py` with the YAML above, scaled down to ~4 trials, and compare.)

---
## 8.10 Capstone — an end-to-end production project (Flowers classifier)

| Requirement | Section |
|---|---|
| SavedModel **with preprocessing inside** + post-export self-test | C1–C2 |
| Serve with TF Serving (Docker), REST + gRPC clients, benchmark (Task 18 rules) | C3 |
| Deploy **v2**, canary, roll back, logs | C4–C5 |
| **Batching** before/after | C6 |
| **TFLite** float16 + full-integer: size / accuracy / latency | C7 |
| Train v2 with **MirroredStrategy**, speed-up vs one device | C4 |
| Cost/latency/scale plan, security checklist, teardown checklist | C8 |

### C1 — Data and model (transfer learning, preprocessing inside)
Clients send **raw RGB images, 224×224, values 0–255**. Inside the model: `Resizing` to 160×160 and `Rescaling` to
[−1, 1] (MobileNetV2's expected range), then a frozen MobileNetV2 base and a small classification head.

In [ ]:
FLOWER_URL = "https://storage.googleapis.com/download.tensorflow.org/example_images/flower_photos.tgz"
DATA_DIR = WORK / "data"
DATA_DIR.mkdir(exist_ok=True)
flowers_dir = DATA_DIR / "flower_photos"
if not flowers_dir.exists():
    tgz = keras.utils.get_file("flower_photos.tgz", origin=FLOWER_URL)
    with tarfile.open(tgz) as tar:
        tar.extractall(DATA_DIR)

RAW_SIZE, CAP_IMG, CAP_BATCH = 224, 160, 32
train_ds, val_ds = keras.utils.image_dataset_from_directory(
    flowers_dir, validation_split=0.2, subset="both", seed=SEED,
    image_size=(RAW_SIZE, RAW_SIZE), batch_size=CAP_BATCH)
CLASS_NAMES = train_ds.class_names
N_CLASSES = len(CLASS_NAMES)
train_ds = train_ds.prefetch(tf.data.AUTOTUNE)
val_ds = val_ds.cache().prefetch(tf.data.AUTOTUNE)
X_val_cap = np.concatenate([x.numpy() for x, _ in val_ds.take(8 if QUICK else 1000)])
y_val_cap = np.concatenate([y.numpy() for _, y in val_ds.take(8 if QUICK else 1000)])
print("Classes:", CLASS_NAMES, "| held-out images used for comparisons:", len(X_val_cap))

def build_flowers(dropout=0.2):
    try:
        base = keras.applications.MobileNetV2(input_shape=(CAP_IMG, CAP_IMG, 3), include_top=False,
                                              weights="imagenet")
    except Exception as e:   # offline machine: keep the pipeline runnable, accuracy will be poor
        print("ImageNet weights unavailable (", type(e).__name__, ") -> using random weights")
        base = keras.applications.MobileNetV2(input_shape=(CAP_IMG, CAP_IMG, 3), include_top=False, weights=None)
    base.trainable = False
    inputs = keras.Input((RAW_SIZE, RAW_SIZE, 3), name="raw_image")
    x = keras.layers.Resizing(CAP_IMG, CAP_IMG)(inputs)
    x = keras.layers.Rescaling(1 / 127.5, offset=-1.0)(x)     # 0..255 -> -1..1 inside the model
    x = base(x, training=False)
    x = keras.layers.GlobalAveragePooling2D()(x)
    x = keras.layers.Dropout(dropout)(x)
    outputs = keras.layers.Dense(N_CLASSES, activation="softmax")(x)
    model = keras.Model(inputs, outputs)
    model.compile(loss="sparse_categorical_crossentropy", optimizer="adam", metrics=["accuracy"])
    return model

CAP_EPOCHS = 2 if QUICK else 5
with tf.device(DEVICES[0]):
    tf.random.set_seed(SEED)
    flowers_v1 = build_flowers()
    timer_v1 = EpochTimer()
    flowers_v1.fit(train_ds, epochs=CAP_EPOCHS, validation_data=val_ds, verbose=2, callbacks=[timer_v1])
v1_cap_acc = flowers_v1.evaluate(val_ds, verbose=0)[1]
print(f"Flowers v1 val accuracy: {v1_cap_acc:.4f}  | single-device epoch time: {timer_v1.times[-1]:.1f} s")

### C2 — Export with the helper (self-test included)

In [ ]:
CAP_NAME = "flowers_model"
CAP_SPEC = tf.TensorSpec([None, RAW_SIZE, RAW_SIZE, 3], tf.float32, name="images")
CAP_ROOT = SERVING_ROOT / CAP_NAME
shutil.rmtree(CAP_ROOT, ignore_errors=True)
export_model(flowers_v1, CAP_NAME, 1, input_spec=CAP_SPEC, test_inputs=X_val_cap[:5], atol=1e-4)
cap_in, cap_out, _ = signature_io(CAP_ROOT / "0001")
print("Signature:", cap_in, "->", cap_out)
print("\n".join(dir_tree(CAP_ROOT)))

(Tolerance 1e-4 here: MobileNet's many convolutions accumulate float rounding differences between the eager and
graph execution paths; for the dense MNIST model 1e-5 holds.)

### C3 — Serve it and benchmark REST vs gRPC

In [ ]:
CAP_REST, CAP_GRPC = 8521, 8520
local_cap = LocalVersionedServer(CAP_ROOT)
if SERVING_UP:
    docker_run_serving("tfs_flowers", [(CAP_ROOT, f"/models/{CAP_NAME}")], rest_port=CAP_REST, grpc_port=CAP_GRPC,
                       env={"MODEL_NAME": CAP_NAME})
    print("Flowers server ready:", wait_ready(CAP_NAME, CAP_REST))
    sh("docker logs --tail 8 tfs_flowers")
    y_cap_rest = rest_predict(X_val_cap[:4], CAP_NAME, rest_port=CAP_REST)
    if GRPC_OK:
        y_cap_grpc = grpc_predict(X_val_cap[:4], CAP_NAME, grpc_port=CAP_GRPC, input_name=cap_in,
                                  output_name=cap_out)
        print("REST == gRPC:", np.allclose(y_cap_rest, y_cap_grpc, atol=1e-5))
else:
    y_cap_rest, _ = local_cap.predict(X_val_cap[:4])
print("Predicted:", [CLASS_NAMES[i] for i in y_cap_rest.argmax(1)], "| true:",
      [CLASS_NAMES[i] for i in y_val_cap[:4]])

cap_bench = protocol_benchmark(X_val_cap, [1, 8, 32], CAP_NAME, cap_in, cap_out,
                               rest_port=CAP_REST, grpc_port=CAP_GRPC, local_server=local_cap)
print(cap_bench.to_string(index=False))
RESULTS["capstone_protocol_bench"] = cap_bench

### C4 — Train v2 with `MirroredStrategy` and measure the speed-up
v2 changes the dropout and trains on all `DEVICES` (global batch 32 → 16 per replica on 2 replicas, same global batch
as v1 so the maths is comparable).

In [ ]:
cap_strategy = make_mirrored(DEVICES)
with cap_strategy.scope():
    tf.random.set_seed(SEED + 1)
    flowers_v2 = build_flowers(dropout=0.3)
timer_v2 = EpochTimer()
flowers_v2.fit(train_ds, epochs=CAP_EPOCHS, validation_data=val_ds, verbose=2, callbacks=[timer_v2])
v2_cap_acc = flowers_v2.evaluate(val_ds, verbose=0)[1]
speedup_df = pd.DataFrame([
    {"model": "v1 (1 device)", "replicas": 1, "s/epoch (last)": round(timer_v1.times[-1], 1), "val acc": round(v1_cap_acc, 4)},
    {"model": "v2 (MirroredStrategy)", "replicas": cap_strategy.num_replicas_in_sync,
     "s/epoch (last)": round(timer_v2.times[-1], 1), "val acc": round(v2_cap_acc, 4)}])
speedup_df["speed-up"] = (timer_v1.times[-1] / speedup_df["s/epoch (last)"]).round(2)
print(speedup_df.to_string(index=False))
print("(virtual devices share one physical device -> expect ~1x; two real GPUs -> up to ~2x)")
RESULTS["capstone_speedup"] = speedup_df

### C5 — Deploy v2, canary 10 %, roll back

In [ ]:
try:
    export_model(flowers_v2, CAP_NAME, 2, input_spec=CAP_SPEC, test_inputs=X_val_cap[:5], atol=1e-4)
except Exception as e:   # some TF builds can't export mirrored variables directly -> re-load on one device first
    print("Direct export of the distributed model failed:", type(e).__name__, "-> exporting a single-device copy")
    flowers_v2.save(WORK / "flowers_v2.keras")
    flowers_v2_single = keras.models.load_model(WORK / "flowers_v2.keras")
    export_model(flowers_v2_single, CAP_NAME, 2, input_spec=CAP_SPEC, test_inputs=X_val_cap[:5], atol=1e-4)
cap_config = CAP_ROOT / "models.config"
def write_cap_config(versions):
    vs = " ".join(f"versions: {v}" for v in versions)
    cap_config.write_text(f'model_config_list {{ config {{ name: "{CAP_NAME}" base_path: "/models/{CAP_NAME}" '
                          f'model_platform: "tensorflow" model_version_policy {{ specific {{ {vs} }} }} }} }}')
write_cap_config([1, 2])

def cap_predict(X, version):
    if SERVING_UP:
        return rest_predict(X, CAP_NAME, version=version, rest_port=CAP_REST)
    return local_cap.predict(X, version=version)[0]

if SERVING_UP:
    docker_run_serving("tfs_flowers", [(CAP_ROOT, f"/models/{CAP_NAME}")], rest_port=CAP_REST, grpc_port=CAP_GRPC,
                       args="--model_config_file=/models/" + CAP_NAME + "/models.config "
                            "--model_config_file_poll_wait_seconds=5")
    # note: CAP_ROOT is mounted at /models/flowers_model, so the config path is inside it
    wait_ready(CAP_NAME, CAP_REST, want_version=2)
    print("Serving versions:", served_versions(CAP_NAME, CAP_REST))

rows = []
rng = np.random.default_rng(SEED)
for i in range(len(X_val_cap)):
    user = int(rng.integers(0, 10_000))
    v = assign_version(user, canary_share=0.10)
    t0 = time.perf_counter()
    p = cap_predict(X_val_cap[i:i + 1], v)
    rows.append({"version": v, "ms": (time.perf_counter() - t0) * 1000, "correct": int(p.argmax() == y_val_cap[i])})
canary = pd.DataFrame(rows).groupby("version").agg(requests=("correct", "size"), accuracy=("correct", "mean"),
                                                  p50_ms=("ms", "median"), p95_ms=("ms", lambda s: s.quantile(.95)))
print(canary.round(3).to_string())

# ---- roll back: remove v2 from the config and delete its folder ----
write_cap_config([1])
shutil.rmtree(CAP_ROOT / "0002", ignore_errors=True)
if SERVING_UP:
    t0 = time.time()
    while time.time() - t0 < 60 and 2 in served_versions(CAP_NAME, CAP_REST):
        time.sleep(1)
    print("After rollback, serving:", served_versions(CAP_NAME, CAP_REST))
    sh("docker logs --tail 12 tfs_flowers")
else:
    print("After rollback, local server versions:", local_cap.versions())
RESULTS["capstone_canary"] = canary

### C6 — Batching before/after (50 concurrent clients)

In [ ]:
cap_batch_rows = []
RPC_CAP = 4 if QUICK else 20
X_pool = X_val_cap
def cap_load_test(pred):
    lat, lock = [], threading.Lock()
    def client(cid):
        r_ = np.random.default_rng(cid)
        for _ in range(RPC_CAP):
            i = int(r_.integers(0, len(X_pool)))
            t0 = time.perf_counter(); pred(X_pool[i:i + 1])
            with lock:
                lat.append((time.perf_counter() - t0) * 1000)
    t0 = time.perf_counter()
    with cf.ThreadPoolExecutor(50) as ex:
        list(ex.map(client, range(50)))
    wall = time.perf_counter() - t0
    lat = np.array(lat)
    return {"throughput_rps": len(lat) / wall, "p50_ms": np.percentile(lat, 50),
            "p95_ms": np.percentile(lat, 95), "p99_ms": np.percentile(lat, 99)}

if SERVING_UP:
    for label, timeout_us in [("off", None), ("on, 5 ms", 5000)]:
        args = ""
        if timeout_us:
            (CAP_ROOT / "batching.config").write_text(
                f"max_batch_size {{ value: 32 }}\nbatch_timeout_micros {{ value: {timeout_us} }}\n"
                f"max_enqueued_batches {{ value: 1000 }}\nnum_batch_threads {{ value: 2 }}\n")
            args = f"--enable_batching=true --batching_parameters_file=/models/{CAP_NAME}/batching.config"
        docker_run_serving("tfs_flowers", [(CAP_ROOT, f"/models/{CAP_NAME}")], rest_port=CAP_REST,
                           grpc_port=CAP_GRPC, env={"MODEL_NAME": CAP_NAME}, args=args)
        wait_ready(CAP_NAME, CAP_REST)
        rest_predict(X_pool[:1], CAP_NAME, rest_port=CAP_REST)          # warm-up
        cap_batch_rows.append({"batching": label,
                               **cap_load_test(lambda x: rest_predict(x, CAP_NAME, rest_port=CAP_REST))})
else:
    fn_cap, _, _, _ = local_cap._get(1)
    call = lambda X: fn_cap(images=tf.constant(X))["probs"].numpy()
    call(X_pool[:1])
    cap_batch_rows.append({"batching": "off", **cap_load_test(call)})
    mb = MicroBatcher(call, max_batch=32, timeout_s=0.005)
    cap_batch_rows.append({"batching": "on, 5 ms", **cap_load_test(mb), "avg batch": float(np.mean(mb.batch_sizes))})
    mb.close()
cap_batch_df = pd.DataFrame(cap_batch_rows).round(1)
print("Backend:", "TF Serving" if SERVING_UP else "local MicroBatcher")
print(cap_batch_df.to_string(index=False))
RESULTS["capstone_batching"] = cap_batch_df

For a convolutional model the batched version is much more efficient per image: one MobileNet call on 32 images costs
far less than 32 separate calls, so throughput rises sharply while the median latency grows by about the batching
window.

### C7 — TFLite float16 and full-integer

In [ ]:
def cap_rep_data():
    for x, _ in train_ds.unbatch().batch(1).take(200):
        yield [tf.cast(x, tf.float32)]

def cfg_cap_int(c):
    c.optimizations = [tf.lite.Optimize.DEFAULT]
    c.representative_dataset = cap_rep_data
    c.target_spec.supported_ops = [tf.lite.OpsSet.TFLITE_BUILTINS_INT8]
    c.inference_input_type = tf.uint8          # raw pixels 0..255 fit uint8 exactly
    c.inference_output_type = tf.uint8

cap_variants = {}
for name, cfg in [("TFLite float32", None), ("TFLite float16", cfg_float16), ("TFLite full-int8", cfg_cap_int)]:
    try:
        cap_variants[name], _ = convert(CAP_ROOT / "0001", "flowers_" + name.split()[-1].replace("-", "_"), cfg)
    except Exception as e:
        print(name, "conversion failed:", type(e).__name__, str(e)[:200])

cap_sm = tf.saved_model.load(str(CAP_ROOT / "0001")).signatures["serving_default"]
cap_sm_size = sum(p.stat().st_size for p in (CAP_ROOT / "0001").rglob("*") if p.is_file())
x1 = X_val_cap[:1]
p_sm = np.concatenate([cap_sm(images=tf.constant(X_val_cap[i:i + 32]))["probs"].numpy()
                       for i in range(0, len(X_val_cap), 32)])
m, s = bench(lambda: cap_sm(images=tf.constant(x1)), warmup=3, repeats=5, inner=5)
rows = [{"variant": "SavedModel float32", "size": human(cap_sm_size),
         "accuracy": (p_sm.argmax(1) == y_val_cap).mean(), "latency ms (1 img)": f"{m:.1f} ± {s:.1f}"}]
for name, b in cap_variants.items():
    p = tflite_predict(b, X_val_cap, batch=32)
    m, s = tflite_latency(b, x1, repeats=5, inner=5)
    rows.append({"variant": name, "size": human(len(b)), "accuracy": (p.argmax(1) == y_val_cap).mean(),
                 "latency ms (1 img)": f"{m:.1f} ± {s:.1f}"})
cap_tfl_df = pd.DataFrame(rows)
cap_tfl_df["accuracy"] = cap_tfl_df["accuracy"].round(4)
print(f"Evaluated on {len(X_val_cap)} held-out images")
print(cap_tfl_df.to_string(index=False))
RESULTS["capstone_tflite"] = cap_tfl_df

### C8 — Results summary and production plans

In [ ]:
summary_rows = [
    {"item": "v1 val accuracy (frozen MobileNetV2 + head)", "value": f"{v1_cap_acc:.3f}"},
    {"item": "v2 val accuracy (MirroredStrategy)", "value": f"{v2_cap_acc:.3f}"},
    {"item": "serving backend used", "value": "TF Serving (Docker)" if SERVING_UP else "local fallback"},
    {"item": "SavedModel size", "value": human(cap_sm_size)},
]
for name, b in cap_variants.items():
    summary_rows.append({"item": f"{name} size", "value": human(len(b))})
print(pd.DataFrame(summary_rows).to_string(index=False))

#### Cost / latency / scale plan (one page)
*Assumptions — replace with real product numbers:* 2 M predictions/day, peak = 5× average, p95 latency budget
**150 ms** end-to-end, images sent as 224×224 JPEG bytes (not JSON floats).

| Decision | Plan | Reasoning |
|---|---|---|
| **Serving stack** | TF Serving in containers behind a load balancer (or Vertex AI endpoint if the team is small) | proven server, version management, batching |
| **Input format** | accept **JPEG bytes** (`{"b64": ...}`) and decode inside the model; gRPC for internal callers | 224×224×3 floats in JSON ≈ 3 MB/image; JPEG ≈ 20–40 KB |
| **Load** | avg 2 M/86,400 ≈ **23 QPS**, peak ≈ **115 QPS** | |
| **CPU vs GPU** | **CPU** replicas (MobileNetV2 @160 px ≈ tens of ms/image on a modern core, batching on) — move to one small GPU if QPS > ~500 or the model grows | GPUs only pay off at high, steady QPS |
| **Replicas** | min **2** (two zones, no cold start) → autoscale to ~6 at 60 % CPU | redundancy + headroom for 5× peaks |
| **Batching** | on, `batch_timeout ≈ 5 ms`, `max_batch_size 32` | throughput gain, well inside the 150 ms budget |
| **Edge option** | TFLite **int8** model in the mobile app for offline use; server stays the source of truth | ~4× smaller, privacy, offline |
| **Monitoring** | p50/p95/p99 latency, error rate, QPS, CPU, batch size, prediction-class distribution, input brightness/size drift | see reflection below |
| **Rollback criteria** | p99 > 300 ms for 10 min, error rate > 1 %, or class-distribution PSI > 0.2 vs previous version → automatic rollback to N−1 | |
| **Cost** | ~2–6 small CPU nodes ≈ predictable monthly bill; review after first month | |

#### Security checklist
- [ ] Service-account keys never in Git (`.gitignore`), stored in a secret manager, rotated; prefer keyless identity
- [ ] Least privilege: serving reads only the model bucket; CI deploys with a separate deploy account
- [ ] TLS everywhere (gateway → servers via mTLS or private network); no plain-HTTP model ports exposed
- [ ] Model and data **buckets private**, versioned, with access logs
- [ ] **Rate limiting** and **request-size limits** at the gateway (protects against abuse and huge JSON payloads)
- [ ] Input validation (shape, dtype, image size) before the model; reject and log malformed requests
- [ ] Dependency/container image scanning; pin the `tensorflow/serving` image version

#### Teardown checklist
- [ ] `docker rm -f tfs_mnist tfs_flowers tfs_big` (local containers)
- [ ] Cloud: undeploy & delete endpoints, delete models, delete buckets (or lifecycle rules), delete service accounts/keys
- [ ] Stop/delete GPU VMs and notebooks; check the billing report the next day
- [ ] Remove local secrets and temporary data (`task23_work/` if no longer needed)

In [ ]:
# Local teardown of containers started by this notebook
if DOCKER:
    for c in ("tfs_mnist", "tfs_flowers", "tfs_big"):
        sh(f"docker rm -f {c}", quiet=True)
    print("Containers removed.")

## 8.11 "Which tool when" guide

| Tool | Use it when | Latency | Throughput | Privacy | Cost | Ops burden |
|---|---|---|---|---|---|---|
| **local `model.predict()`** | notebooks, offline batch scoring, prototypes | lowest (in-process) | limited to one process | data stays local | free | none |
| **Flask/FastAPI service** | a demo, a model that *needs* Python (`py_function`, custom libs) | good | low–medium (GIL, no batching) | you control | low | you build versioning, batching, monitoring yourself |
| **TF Serving (self-hosted)** | production TF models, many versions/models, high QPS | low | high (C++, batching, gRPC) | you control | infra cost | medium–high (K8s, TLS, monitoring) |
| **Managed prediction service** (Vertex AI) | small team, need autoscaling/auth/logging fast | low (+ cold start if scaled to zero) | high, autoscaled | data goes to the cloud provider | pay per node-hour | low |
| **TFLite / LiteRT** | phones, embedded, microcontrollers, offline | very low (no network) | per device | best (on device) | free at run time | model update logistics |
| **TensorFlow.js** | interactive web apps, privacy-sensitive browser features | very low | per browser | best (in browser) | free at run time | download size, browser variety |
| **Single GPU** | most training jobs; start here | — | — | — | low | low |
| **Multi-GPU, one machine** (`MirroredStrategy`) | training too slow on one GPU | — | near-linear up to 4–8 GPUs | — | medium | low (few lines of code) |
| **Multi-machine cluster** (`MultiWorkerMirrored`, PS) | one machine is not enough (huge models/data) | — | limited by network bandwidth | — | high | high (TF_CONFIG, failures, networking) |
| **TPU** (`TPUStrategy`) | very large models/batches, static shapes, TF/JAX on Google Cloud | — | very high | — | per-hour, efficient at scale | medium (input pipeline tuning) |
| **Cloud tuning service** | many expensive trials, want Bayesian search and parallel trials | — | parallel trials | — | pay per trial | low |

**The source's advice, which I agree with:** before building a cluster, try **one machine with more or stronger GPUs**
— communication inside one box is an order of magnitude faster than across a network, and the code stays simple.

## 8.12 Reflection — the first month in production
I would watch three layers. **Service health:** p50/p95/p99 latency, error rate by type (4xx = client/input problems,
5xx = server problems), throughput, CPU/GPU utilisation, batch size and queueing time, and model-load events after each
deployment. Any SLO breach right after a deployment (e.g. p99 or error rate doubling for 10 minutes) triggers an
**immediate rollback** to the previous version folder — no debugging in production first.
**Inputs (data drift):** distributions of what clients send — for images: size, brightness, colour histograms, the
share of grey-scale or corrupt images; for any model: a PSI/KS test against the training distribution. Drift alone
does not mean the model is wrong, but it is the earliest warning, and a sudden shift often means an upstream bug (a
client that stopped resizing, or started sending BGR instead of RGB — exactly the skew that preprocessing inside the
model prevents). **Outputs and quality:** the predicted-class distribution and mean confidence per day, compared with the
canary baseline; then, as delayed labels arrive (user corrections, manual review of a random sample), real accuracy
per class. A sustained drop in labelled accuracy, or concept drift (new kinds of flowers or photos the model never
saw), triggers **retraining** on fresh data, a new version folder, and the same canary → promote/rollback cycle from
the runbook. After the first month I would turn these observations into fixed alert thresholds (Task 15).

### Phase 8 — summary / deliverable
* ✅ Cluster spec + `TF_CONFIG` design; **two-worker `MultiWorkerMirroredStrategy` run** with timestamped logs
* ✅ AllReduce options, `ParameterServerStrategy` (old vs coordinator API), annotated cloud-job commands (old + Vertex)
* ✅ Annotated tuning YAML; **local random-search analog** reading metrics from TensorBoard event files, ranked
* ✅ **Capstone**: preprocessing-inside SavedModel + self-test, served & benchmarked, v2 with `MirroredStrategy`,
  canary + rollback, batching before/after, TFLite float16/int8 comparison, plan + security + teardown checklists
* ✅ "Which tool when" guide and first-month monitoring reflection

---
# Exercises (Must Do)

**1. What does a SavedModel contain? How do you inspect it?**
A versioned folder with `saved_model.pb` (one or more *metagraphs*: computation graphs identified by tag sets such as
`serve`, each with named *signatures* describing input/output tensors), a `variables/` folder (weights, possibly
sharded, plus an index), and `assets/` (vocabularies and other files the graph reads). Inspect it with
`saved_model_cli show --dir … --all` (tags, signatures, names, dtypes, shapes), test it with `saved_model_cli run`,
or programmatically with `tf.saved_model.load(path).signatures` (Phase 1.6–1.7).

**2. When should you use TF Serving? Main features? Deployment tools?**
When a TensorFlow model must be served to other systems reliably and fast: high QPS, many models/versions, zero-downtime
updates. Features: C++ server with REST + gRPC, multi-model and multi-version serving, automatic loading/unloading from
the file system or a config file, version pinning, request batching, warm-up, monitoring endpoint. Deploy it with
**Docker**, orchestrate with **Kubernetes** (Deployments, Services, autoscaling), or let a **managed service**
(Vertex AI, or KServe on your own cluster) run it.

**3. How do you deploy a model across multiple TF Serving instances?**
Run several identical TF Serving containers that read the model from shared storage (a bucket), put a **load balancer**
in front, and let an orchestrator (Kubernetes) manage replicas, readiness probes (`/v1/models/<name>`), rolling updates
and autoscaling. Deploying a new version = publishing a new numbered folder that every instance picks up (Phase 3.7).

**4. When should you use gRPC rather than REST?**
For large inputs/outputs (images, audio, big batches), high request rates and latency-sensitive service-to-service
calls: gRPC sends compact binary tensors over HTTP/2, while REST turns every float into ~15–20 characters of JSON that
must be built and parsed. Use REST for small, occasional requests and clients where gRPC is awkward (browsers, quick
scripts, debugging with `curl`). Measured in Phase 2.6/2.9 and C3.

**5. How does TFLite reduce a model's size for mobile/embedded devices?**
(1) The converter **prunes** training-only ops, **folds constants**, and **fuses** ops (e.g. BatchNorm into the previous
layer, activations into convolutions/dense ops); (2) the model is stored as a compact **FlatBuffer** that can be mapped
straight into memory; (3) **quantization**: float16 weights (½ size), int8 weight quantization (¼ size), full-integer
quantization of weights *and* activations (¼ size + integer-only compute), optionally with quantization-aware training.

**6. What is quantization-aware training and why would you need it?**
Training with **fake-quantization ops** in the forward pass so the network experiences the rounding noise of int8
weights/activations and learns parameters that are robust to it (gradients pass through with a straight-through
estimator). You need it when **post-training quantization loses too much accuracy** — typical for compact models such
as MobileNets or models with wide activation ranges. It also records activation ranges during training, so separate
calibration becomes automatic (Phase 5.6).

**7. Model parallelism vs data parallelism — why is data parallelism generally recommended?**
Model parallelism splits **one model** across devices; data parallelism **replicates** the model on every device and
feeds each a different mini-batch, then combines gradients. Data parallelism is preferred because it works for any
architecture without manual splitting, keeps all devices busy, and only exchanges gradients/parameters, whereas model
parallelism makes devices wait for each other and often needs heavy cross-device activation traffic (Phase 7.1–7.2).

**8. Distribution strategies across multiple servers — which and how to choose?**
* `MultiWorkerMirroredStrategy` — synchronous, all-reduce, each worker holds a full replica. **Default choice** when the
  model fits on each device and the network is good.
* `ParameterServerStrategy` — asynchronous updates via parameter servers; useful for **very large or sparse models**
  (embeddings sharded across PS) or clusters with unreliable/preemptible workers where waiting for stragglers hurts.
* `TPUStrategy` for TPU pods. *Choose:* start with mirrored on one machine; go multi-worker mirrored when you need more;
  use parameter servers only if synchronous training is bottlenecked by stragglers or the parameters don't fit.

**9. Deploy, query, update, roll back** — done in Phases 2–3 (MNIST: v1 → v2 → rollback with the client-side plot)
and in the capstone C3–C5 (Flowers: REST + gRPC clients, v2 canary, rollback).

**10. MirroredStrategy vs CentralStorageStrategy timing** — done in Phase 7.8 (single device vs Mirrored vs
CentralStorage, 3 repeats each, mean ± std, with interpretation).

**11. Hyperparameter tuning** — done as the local random-search analog in Phase 8.9 (metric read from TensorBoard event
files); the managed version is documented (`USE_CLOUD=False` → design-document mode).

**12. Symmetric 8-bit quantization of `[-1.5, -0.4, 0.0, 0.3, 0.8]` by hand**

* `m = max|w| = 1.5`, `scale = 1.5 / 127 = 0.0118110`
* `q = round(w / scale)`:
  −1.5 → −127 ; −0.4 → round(−33.87) = **−34** ; 0.0 → **0** ; 0.3 → round(25.4) = **25** ; 0.8 → round(67.73) = **68**
* dequantized `q × scale`: −1.5000, −0.4016, 0.0000, 0.2953, 0.8031
* errors: 0, 0.0016, 0, 0.0047, 0.0031 → **maximum absolute error ≈ 0.0047** (always ≤ scale/2 ≈ 0.0059)

Verified in code:

In [ ]:
w = np.array([-1.5, -0.4, 0.0, 0.3, 0.8], dtype=np.float32)
q, scale = quantize_symmetric(w)
deq = dequantize(q, scale)
ex12 = pd.DataFrame({"w": w, "w/scale": (w / scale).round(2), "q (int8)": q, "dequantized": deq.round(4),
                     "abs error": np.abs(deq - w).round(4)})
print(f"scale = {scale:.7f}")
print(ex12.to_string(index=False))
print("max abs error =", np.abs(deq - w).max().round(4), "| bound scale/2 =", round(scale / 2, 4))
assert q.tolist() == [-127, -34, 0, 25, 68]

**13. JSON vs binary for 64 images of 224×224×3**

Assumption: **~17 characters per float** in JSON (a value like `0.47843137383461` plus a comma and brackets) vs 4 bytes in
binary → ≈ **4.3×** larger. 64 × 224 × 224 × 3 = 9,633,792 floats → binary ≈ **38.5 MB**, JSON ≈ **160+ MB**.
For raw 0–255 pixel values sent as floats (`123.0`), JSON is shorter (≈ 5–6 chars) — so we measure both:

In [ ]:
n_floats = 64 * 224 * 224 * 3
binary = n_floats * 4
est_json = n_floats * 17
sample = np.random.rand(8, 224, 224, 3).astype("float32")              # normalised images (0..1)
pix = np.random.randint(0, 256, (8, 224, 224, 3)).astype("float32")      # raw pixels as floats
meas_norm = len(json.dumps({"instances": sample.tolist()})) / sample.nbytes
meas_pix = len(json.dumps({"instances": pix.tolist()})) / pix.nbytes
ex13 = pd.DataFrame([
    {"case": "estimate (17 chars/float)", "JSON / binary": est_json / binary, "64-image JSON size": human(est_json)},
    {"case": "measured: normalised floats 0..1", "JSON / binary": meas_norm, "64-image JSON size": human(binary * meas_norm)},
    {"case": "measured: raw pixel floats 0..255", "JSON / binary": meas_pix, "64-image JSON size": human(binary * meas_pix)},
])
ex13["JSON / binary"] = ex13["JSON / binary"].round(2)
print(f"Binary float32 payload for 64 images: {human(binary)}")
print(ex13.to_string(index=False))

**14. Why do integer variables land on the CPU, and what does `set_soft_device_placement` change?**
A kernel is an implementation of an op for a given dtype and device; TensorFlow has **no GPU kernel for int32 variables**
(and few for integer ops), because integer tensors are mostly shapes, indices and counters that the host needs anyway.
So the placer puts them on `/cpu:0`. If you *force* such an op onto a GPU (or onto a device that doesn't exist), strict
placement raises an error. `tf.config.set_soft_device_placement(True)` tells TF to **fall back to a device that has a
kernel** (usually the CPU) instead of failing — convenient, but it can silently add CPU↔GPU copies, so log placements
(`tf.debugging.set_log_device_placement(True)`) when performance matters. Demonstrated in Phase 6.5–6.6.

**15. 20 synchronous replicas, ignoring the 2 slowest:**
* **Step time** is set by the **18th** fastest replica instead of the slowest one, so random stragglers (a busy machine,
  a slow network hop, GC pause) no longer delay every step — the tail latency of each step drops sharply.
* **Bandwidth:** each step still sends parameters to all 20 replicas, but only 18 gradient sets are aggregated, and
  the late gradients can be dropped as soon as they arrive (≈10 % less aggregation traffic at the critical moment).
  The compute of 2 replicas per step is "wasted" — that's the price.
* **Fault tolerance:** if up to 2 workers crash or hang, training continues without waiting for them; with plain
  synchronous training a single dead worker stalls everything.
* Gradient quality is barely affected: the averaged gradient comes from 18 instead of 20 mini-batches (slightly smaller
  effective batch), and no stale gradients are introduced.

---
# Deliverables Checklist (final status)

| Deliverable | Where in this notebook | Status |
|---|---|---|
| **Phase 1:** versioned SavedModel, CLI inspection & run, bundled preprocessing, `py_function` failure, `export_model` + self-test | 1.3–1.10 | ✅ |
| **Phase 2:** TF Serving container, REST + gRPC clients, payload & latency benchmarks, failure-drill table | 2.3–2.11 | ✅ (Docker or local fallback) |
| **Phase 3:** v2 rollout + rollback evidence, A/B (canary) test, batching charts, warm-up, runbook | 3.1–3.8 | ✅ |
| **Phase 4:** managed service design doc, cost estimate, cold-start table, security checklist, client & teardown | 4.1–4.9 | ✅ design-document mode |
| **Phase 5:** TFLite size/accuracy/latency table, NumPy quantization (−1.5…+0.8), QAT, TF.js, deployment table | 5.1–5.9 | ✅ |
| **Phase 6:** GPU verification, memory options, placement, execution walkthrough, thread pools, multi-device | 6.1–6.10 | ✅ (GPU-only demos run on a GPU runtime) |
| **Phase 7:** parallelism write-ups & diagrams, sync/async table, saturation estimate, strategy timing, custom loop, LR/batch | 7.1–7.11 | ✅ |
| **Phase 8:** cluster spec & `TF_CONFIG`, two-worker run, cloud-job/tuning YAML, tuning analog, capstone, guides | 8.1–8.12 | ✅ |
| **Exercises 1–15** | this section | ✅ (cloud exercises 9–11 done locally / as design) |
| **Code:** runs top to bottom, no secrets committed | whole notebook | ✅ (`USE_CLOUD=False`, no keys) |

In [ ]:
print("Key result tables collected during the run:")
for k, v in RESULTS.items():
    print(f"\n=== {k} ===")
    print(v.to_string() if hasattr(v, "to_string") else v)

## Glossary (quick reference)
| Term | Meaning |
|---|---|
| SavedModel | language-neutral model package: graph(s) + variables + assets |
| Metagraph / tag / signature | a graph with its run-time info / label selecting it (`serve`) / named entry point (`serving_default`) |
| Quiescing | letting an old version finish in-flight requests before unloading it |
| Canary release | sending a new version to a small share of traffic first |
| Automatic batching | merging requests that arrive within a short window into one model call |
| Warm-up | replaying sample requests at load time so users never hit the slow first call |
| FlatBuffer | TFLite's zero-copy, memory-mappable serialization format |
| Symmetric quantization | `[−m, +m] → [−127, +127]`, zero maps to zero |
| Calibration | running representative inputs to record activation ranges for int8 |
| Soft placement | fall back to a device that has a kernel instead of failing |
| AllReduce | collective reduction after which every device holds the result |
| Stale gradient | gradient computed on parameters that changed meanwhile (async training) |
| Spare replicas | aggregate only the fastest k of N replicas per step |
| `TF_CONFIG` | env variable holding the cluster spec and this task's type/index |